<h1>Fine tuning</h1>

In [1]:
#!/usr/bin/env python3
"""
Fine-tune RoBERTa for hierarchical multi-label semantic feature tagging of
NOUN/PROPN tokens.

Compatible with Python 3.9.21.

Key design changes
------------------
1. The model is trained only on independent semantic features. For each token,
   any feature that can be inherited from a more specific present feature is
   removed from the neural target. This is a per-token reduction rather than a
   global "terminal leaf only" rule, so features such as Human, Time, Shape,
   Communication, etc. can still be predicted directly when they are the most
   specific available feature for that token.

2. All inherited features are reconstructed deterministically from
   FEATURE_PARENTS for validation, testing, reporting, and later inference.

3. A separate sigmoid threshold is optimised for every neural output using the
   validation split only. The selected thresholds are frozen before the test
   split is evaluated and are saved to prediction_thresholds.json.

4. Train/validation splitting is performed at SENTENCE level with iterative
   multilabel stratification over sentence-level independent-feature presence.
   The split is fixed across repeated fine-tuning runs.

5. Fine-tuning is repeated over a predeclared list of random seeds. The same
   split is used for every run, and results are summarised with mean and SD.

6. The official test corpus may contain a direct independent target that never
   occurred independently in training. Such a label is never added from test
   data. Complete test truth is read directly from source annotations.

7. Held-out test uncertainty is estimated with a SENTENCE-CLUSTER bootstrap.
   All supervised noun/proper-noun targets from a sampled sentence move together.

Tagged data:
    Training/validation:
        ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl
    Test:
        ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl

Sentence/token sources:
    Training/validation:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-train.conllu
    Test:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-test.conllu

The validation split is created only from the training corpus. The official
test corpus remains untouched until final evaluation.

Loss:
    BCEWithLogitsLoss over INDEPENDENT feature labels only.

Headline metrics:
    Computed after deterministic hierarchy expansion, so they describe the
    complete semantic feature sets rather than only the neural outputs.

Requirements:
    python -m pip install torch transformers datasets accelerate conllu scikit-learn iterative-stratification

CUDA is required.
"""

from __future__ import annotations

import gc
import json
import shutil
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import torch
import torch.nn as nn

from conllu import parse_incr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import Dataset

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)


# =============================================================================
# Configuration
# =============================================================================

TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_train_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TEST_JSONL_PATH = Path(
    "../data/ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_test_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

TEST_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-test.conllu"
)

# MODEL_NAME = "FacebookAI/roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"

OUTPUT_DIR = Path("../models/roberta_semantic_features")

# Validation is sampled only from eligible TRAINING sentences.
# Stratification is performed at SENTENCE level using the multilabel feature
# presence vector for each sentence. The separate official test file is never
# split and is used only for final evaluation.
VALIDATION_FRACTION = 0.20

# Fixed seed used ONLY to create the sentence-level train/validation split.
# Keeping this fixed across model runs means repeated runs measure stochastic
# fine-tuning variation rather than variation in the validation partition.
SPLIT_SEED = 42

# Repeated fine-tuning seeds. Use the same list when comparing model variants.
# Thirty runs follow the experiment-design recommendation; reduce this list if
# RoBERTa-large makes 30 runs computationally impractical, and report the actual
# number used.
RUN_SEEDS = [
    11, 23, 37, 53, 71, 89, 107, 127, 149, 173,
    197, 223, 251, 281, 313, 347, 383, 421, 461, 503,
    547, 593, 641, 691, 743, 797, 853, 911, 971, 1031,
]

# RUN_SEEDS = [
#     11
# ]

# The multilabel stratifier is retried with deterministic derived seeds until
# every independent feature still has at least this many positive token targets
# in the TRAIN subset. This prevents a rare direct target from disappearing.
MIN_TRAIN_POSITIVES_PER_FEATURE = 1
MAX_STRATIFICATION_ATTEMPTS = 200

# Sentence-cluster bootstrap for uncertainty in the held-out test sample.
# The same sampled test sentences are used for every run within each bootstrap
# replicate, and the statistic is the mean metric across run seeds.
TEST_BOOTSTRAP_SAMPLES = 1700
TEST_BOOTSTRAP_CONFIDENCE = 0.95
TEST_BOOTSTRAP_SEED = 20260926

# Saving 30 RoBERTa-large models would use a great deal of disk space. The first
# run is therefore saved as a neutral REFERENCE model; it is not selected for
# having the best test result. Metrics and thresholds are saved for every run.
SAVE_REFERENCE_MODEL = True

MAX_LENGTH = 256

LEARNING_RATE = 4.0e-05
LEARNING_RATE_SCHEDULER = "linear"
WEIGHT_DECAY = 0.01
# WARMUP_STEPS=166
WARMUP_STEPS=278
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 4

NUM_EPOCHS = 5

# Threshold used during epoch-by-epoch Trainer evaluation. Final thresholds are
# optimised per independent feature on the validation set after training.
DEFAULT_PREDICTION_THRESHOLD = 0.5

# Validation-only threshold search grid.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.01

# Only these POS categories receive feature supervision.
TARGET_POS = {"NOUN", "PROPN"}

# A NOUN/PROPN with no features is considered a failed source annotation rather
# than a valid all-negative target.
EXCLUDE_EMPTY_FEATURE_RECORDS = True

# Optional per-feature switches. Any feature not listed here is enabled.
# Set a feature to False to remove it from:
#   - sentence-level stratification,
#   - neural training targets,
#   - threshold calibration,
#   - hierarchy-expanded gold/predicted feature sets,
#   - per-feature and aggregate evaluation, and
#   - the sentence-cluster bootstrap.
#
# Disabled intermediate hierarchy nodes are skipped but their enabled ancestors
# are still inherited. For example, if Human were disabled, Feminine would still
# inherit Animate/Natural/Concrete/Internal_Control via the hierarchy.
FEATURE_SWITCHES: Dict[str, bool] = {
    "State": False,
    "State_of_Matter": False,
    "Shape": False,
    "Rigidity": False,
}

# Aggregate per-feature test tables are always printed after all repeated runs.
# Set this to True only if you also want the two full per-feature tables printed
# after EVERY run (very verbose for 30 runs).
PRINT_PER_RUN_FEATURE_TABLES = False

EVAL_STRATEGY = "epoch"

SPLIT_PATH = OUTPUT_DIR / "sentence_split.json"
LABELS_PATH = OUTPUT_DIR / "semantic_features.json"
RUNS_DIR = OUTPUT_DIR / "runs"
REFERENCE_MODEL_DIR = OUTPUT_DIR / "reference_model"
REPEATED_RESULTS_PATH = OUTPUT_DIR / "repeated_run_results.json"
BOOTSTRAP_RESULTS_PATH = OUTPUT_DIR / "test_bootstrap_confidence_intervals.json"


# =============================================================================
# Feature hierarchy
# =============================================================================

def is_feature_enabled(feature: str) -> bool:
    """Return whether *feature* participates in this experiment."""

    return bool(FEATURE_SWITCHES.get(feature, True))


def validate_feature_switches(
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    test_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> None:
    """Fail early on misspelled feature-switch names and report disabled labels."""

    observed: Set[str] = set()

    for source in (train_tagged_records, test_tagged_records):
        for sentence_records in source.values():
            for record in sentence_records.values():
                raw = record.get("features", [])
                if isinstance(raw, list):
                    observed.update(
                        str(feature).strip()
                        for feature in raw
                        if str(feature).strip()
                    )

    known = set(observed) | set(FEATURE_PARENTS)
    for parents in FEATURE_PARENTS.values():
        known.update(parents)

    unknown = sorted(set(FEATURE_SWITCHES) - known)
    if unknown:
        raise ValueError(
            "FEATURE_SWITCHES contains unknown feature name(s): "
            + ", ".join(unknown)
        )

    disabled = sorted(
        feature for feature, enabled in FEATURE_SWITCHES.items() if not enabled
    )

    print("=" * 80)
    print("FEATURE SWITCHES")
    print("=" * 80)
    if disabled:
        print("Disabled features: " + ", ".join(disabled))
    else:
        print("Disabled features: none")
    print("All unlisted features are enabled.")
    print()


# child -> direct parent(s)
FEATURE_PARENTS: Dict[str, Set[str]] = {
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
        "Natural",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },
    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },
    "Inanimate": {
        "Concrete",
    },
    "Garment": {
        "Artefact",
    },
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },
    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },
    "Time": {
        "Abstract",
    },
    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


def validate_feature_hierarchy() -> None:
    """Fail early if FEATURE_PARENTS contains a cycle or invalid name."""

    all_features: Set[str] = set(FEATURE_PARENTS)

    for parents in FEATURE_PARENTS.values():
        all_features.update(parents)

    for feature in all_features:
        if not isinstance(feature, str) or not feature.strip():
            raise ValueError(
                "FEATURE_PARENTS contains an empty or non-string feature name."
            )

    visiting: Set[str] = set()
    visited: Set[str] = set()

    def visit(feature: str) -> None:
        if feature in visited:
            return

        if feature in visiting:
            raise ValueError(
                f"Cycle detected in FEATURE_PARENTS involving {feature!r}."
            )

        visiting.add(feature)

        for parent in FEATURE_PARENTS.get(feature, set()):
            visit(parent)

        visiting.remove(feature)
        visited.add(feature)

    for feature in sorted(all_features):
        visit(feature)


@lru_cache(maxsize=None)
def get_feature_ancestors(feature: str) -> frozenset:
    """Return every direct and indirect parent of *feature*."""

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(feature, set()):
        if is_feature_enabled(parent):
            ancestors.add(parent)
        # Traverse even through a disabled intermediate node so enabled
        # grandparents remain inherited.
        ancestors.update(get_feature_ancestors(parent))

    return frozenset(ancestors)


def expand_feature_hierarchy(features: Iterable[str]) -> Set[str]:
    """Add every deterministic ancestor required by FEATURE_PARENTS."""

    expanded: Set[str] = {
        feature for feature in features if is_feature_enabled(feature)
    }

    for feature in list(expanded):
        expanded.update(get_feature_ancestors(feature))

    return expanded


def reduce_to_independent_features(features: Iterable[str]) -> Set[str]:
    """
    Reduce a complete feature set to the features that must be predicted.

    A feature is redundant when it is an ancestor of another feature present
    for the same token. Such an ancestor can be reconstructed deterministically.

    Examples:
        {Human, Animate, Natural, Concrete, Internal_Control}
            -> {Human}

        {Feminine, Human, Animate, Natural, Concrete, Internal_Control}
            -> {Feminine}

        {Statement, Communication, Abstract, Time}
            -> {Statement, Time}

    This is deliberately PER TOKEN. It is not the same as training only labels
    that are global terminal leaves in the ontology. A generic Human can still
    be a direct neural target if no more specific Human child is present.
    """

    feature_set = set(features)

    inherited_from_other_features: Set[str] = set()

    for feature in feature_set:
        inherited_from_other_features.update(
            get_feature_ancestors(feature)
        )

    return feature_set - inherited_from_other_features


def missing_hierarchy_features(features: Iterable[str]) -> Set[str]:
    """Return ancestors required by the hierarchy but absent from *features*."""

    feature_set = set(features)
    return expand_feature_hierarchy(feature_set) - feature_set


# =============================================================================
# CUDA
# =============================================================================

def check_cuda() -> None:
    """Require CUDA rather than silently falling back to the CPU."""

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. This script is configured to require "
            "a CUDA-capable GPU."
        )

    print("=" * 80)
    print("CUDA")
    print("=" * 80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM: "
        f"{props.total_memory / (1024 ** 3):.2f} GiB"
    )
    print()


# =============================================================================
# Load CoNLL-U
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load all CoNLL-U sentences."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence in parse_incr(handle):
            sentences.append(sentence)

    return sentences


def get_sentence_words(sentence: Any) -> Tuple[List[str], List[int]]:
    """
    Return ordinary integer-ID tokens and their original CoNLL-U token IDs.

    Multiword-token placeholder rows and empty nodes are excluded.
    """

    words: List[str] = []
    token_ids: List[int] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        form = token.get("form")

        if form is None:
            continue

        words.append(str(form))
        token_ids.append(token_id)

    return words, token_ids


# =============================================================================
# Load tagged JSONL
# =============================================================================

def load_tagged_records(
    path: Path,
) -> Dict[int, Dict[int, Dict[str, Any]]]:
    """
    Return:

        sentence_index
            -> token_id
                -> JSONL record

    Only NOUN and PROPN records are retained.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find tagged JSONL:\n{path.resolve()}"
        )

    records: Dict[int, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    "Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if not isinstance(token_id, int):
                continue

            records[sentence_index][token_id] = record

    return dict(records)


# =============================================================================
# Labels
# =============================================================================

def clean_features(record: Dict[str, Any]) -> List[str]:
    """Return unique, non-empty COMPLETE feature names from a JSONL record."""

    raw = record.get("features", [])

    if not isinstance(raw, list):
        return []

    return sorted(
        {
            str(feature).strip()
            for feature in raw
            if (
                str(feature).strip()
                and is_feature_enabled(str(feature).strip())
            )
        }
    )


def get_independent_record_features(record: Dict[str, Any]) -> List[str]:
    """Return the per-token minimal feature set that the model must predict."""

    return sorted(
        reduce_to_independent_features(
            clean_features(record)
        )
    )


def build_full_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """
    Build the COMPLETE evaluation vocabulary from training data only.

    Deterministic ancestors are included even if a training record omitted one,
    although hierarchy closure is separately verified and should normally make
    this addition unnecessary.
    """

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            features = clean_features(record)
            labels.update(features)
            labels.update(expand_feature_hierarchy(features))

    if not labels:
        raise RuntimeError(
            "No semantic features were found in the training split."
        )

    return sorted(labels)


def build_prediction_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """Build the neural-output vocabulary from independent training targets."""

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            labels.update(
                get_independent_record_features(record)
            )

    if not labels:
        raise RuntimeError(
            "No independent semantic features were found in the training split."
        )

    return sorted(labels)


# =============================================================================
# Sentence selection and splitting
# =============================================================================

def get_eligible_sentence_indices(
    sentences: Sequence[Any],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> List[int]:
    """Eligible sentences contain at least one non-empty NOUN/PROPN target."""

    eligible: List[int] = []

    for sentence_index in range(len(sentences)):
        token_records = tagged_records.get(sentence_index, {})

        valid_records = [
            record
            for record in token_records.values()
            if clean_features(record)
        ]

        if valid_records:
            eligible.append(sentence_index)

    return eligible


def get_sentence_independent_feature_counts(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_index: int,
) -> Counter:
    """Count independent positive token targets in one source sentence."""

    counts: Counter = Counter()

    for record in tagged_records.get(sentence_index, {}).values():
        if not clean_features(record):
            continue

        for feature in get_independent_record_features(record):
            counts[feature] += 1

    return counts


def build_sentence_multilabel_matrix(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Tuple[np.ndarray, List[str], Dict[int, Counter]]:
    """
    Build one multilabel stratification row per SENTENCE.

    y_sentence[sentence, feature] is 1 when at least one eligible NOUN/PROPN
    token in that sentence has the independent feature. Token counts are kept
    separately for the post-split training-coverage constraint.
    """

    sentence_feature_counts: Dict[int, Counter] = {
        sentence_index: get_sentence_independent_feature_counts(
            tagged_records,
            sentence_index,
        )
        for sentence_index in eligible_indices
    }

    stratification_labels = sorted(
        {
            feature
            for counts in sentence_feature_counts.values()
            for feature, count in counts.items()
            if count > 0
        }
    )

    if not stratification_labels:
        raise RuntimeError(
            "No independent semantic features were found in eligible "
            "training sentences."
        )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    matrix = np.zeros(
        (len(eligible_indices), len(stratification_labels)),
        dtype=np.int8,
    )

    for row, sentence_index in enumerate(eligible_indices):
        for feature in sentence_feature_counts[sentence_index]:
            matrix[row, label2column[feature]] = 1

    return matrix, stratification_labels, sentence_feature_counts


def _sentence_prevalence_error(
    y_all: np.ndarray,
    validation_rows: np.ndarray,
    validation_fraction: float,
) -> float:
    """
    Score how closely validation sentence-level label prevalences match target.

    Lower is better. The score is used only to choose among deterministic
    iterative-stratification attempts that satisfy the training-coverage rule.
    """

    total_support = y_all.sum(axis=0).astype(np.float64)
    validation_support = y_all[validation_rows].sum(axis=0).astype(np.float64)

    target_support = total_support * float(validation_fraction)

    # Scale by at least one expected example so rare and common labels both
    # contribute without allowing extremely common labels to dominate.
    scale = np.maximum(target_support, 1.0)

    return float(
        np.mean(
            np.abs(validation_support - target_support) / scale
        )
    )


def make_train_validation_split(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Dict[str, Any]:
    """
    Create a SENTENCE-LEVEL multilabel-stratified split from TRAINING only.

    Each sentence receives a binary vector indicating which independent feature
    labels occur anywhere among its eligible NOUN/PROPN tokens. Iterative
    multilabel stratification then assigns whole sentences to train/validation.

    The split is retried deterministically when necessary so that every
    independent feature retains at least MIN_TRAIN_POSITIVES_PER_FEATURE
    positive TOKEN targets in the training subset.

    The official test corpus is deliberately not accepted by this function, so
    test labels cannot influence the split.
    """

    if not 0.0 < VALIDATION_FRACTION < 1.0:
        raise ValueError(
            "VALIDATION_FRACTION must be strictly between 0 and 1."
        )

    if MIN_TRAIN_POSITIVES_PER_FEATURE < 1:
        raise ValueError(
            "MIN_TRAIN_POSITIVES_PER_FEATURE must be at least 1."
        )

    if MAX_STRATIFICATION_ATTEMPTS < 1:
        raise ValueError(
            "MAX_STRATIFICATION_ATTEMPTS must be at least 1."
        )

    eligible_indices = list(eligible_indices)

    if len(eligible_indices) < 2:
        raise RuntimeError(
            "Need at least two eligible training sentences to create a "
            "train/validation split."
        )

    (
        sentence_matrix,
        stratification_labels,
        sentence_feature_counts,
    ) = build_sentence_multilabel_matrix(
        eligible_indices,
        tagged_records,
    )

    all_token_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_token_counts.update(counts)

    x_dummy = np.zeros((len(eligible_indices), 1), dtype=np.int8)

    best_candidate: Optional[Dict[str, Any]] = None
    rejected_for_coverage = 0

    for attempt in range(MAX_STRATIFICATION_ATTEMPTS):
        attempt_seed = SPLIT_SEED + attempt

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=VALIDATION_FRACTION,
            random_state=attempt_seed,
        )

        train_rows, validation_rows = next(
            splitter.split(x_dummy, sentence_matrix)
        )

        train_indices = [
            eligible_indices[int(row)]
            for row in train_rows
        ]
        validation_indices = [
            eligible_indices[int(row)]
            for row in validation_rows
        ]

        train_token_counts: Counter = Counter()

        for sentence_index in train_indices:
            train_token_counts.update(
                sentence_feature_counts[sentence_index]
            )

        missing_or_undercovered = sorted(
            feature
            for feature, total_count in all_token_counts.items()
            if (
                total_count > 0
                and train_token_counts[feature]
                < MIN_TRAIN_POSITIVES_PER_FEATURE
            )
        )

        if missing_or_undercovered:
            rejected_for_coverage += 1
            continue

        prevalence_error = _sentence_prevalence_error(
            sentence_matrix,
            validation_rows,
            VALIDATION_FRACTION,
        )

        candidate = {
            "train": train_indices,
            "validation": validation_indices,
            "split_seed_used": attempt_seed,
            "attempt": attempt + 1,
            "prevalence_error": prevalence_error,
            "stratification_labels": stratification_labels,
        }

        if (
            best_candidate is None
            or prevalence_error < best_candidate["prevalence_error"]
        ):
            best_candidate = candidate

        # Iterative stratification is already designed to balance multilabel
        # prevalence. Once a valid candidate is found, further attempts are not
        # needed unless coverage forced earlier candidates to be rejected.
        if rejected_for_coverage == 0:
            break

    if best_candidate is None:
        raise RuntimeError(
            "Could not create a sentence-level multilabel-stratified "
            "train/validation split that preserved the required training "
            f"support after {MAX_STRATIFICATION_ATTEMPTS} attempts."
        )

    train_indices = best_candidate["train"]
    validation_indices = best_candidate["validation"]

    train_set = set(train_indices)
    validation_set = set(validation_indices)

    if train_set & validation_set:
        raise RuntimeError(
            "Internal split error: at least one sentence occurs in both "
            "training and validation."
        )

    if train_set | validation_set != set(eligible_indices):
        raise RuntimeError(
            "Internal split error: sentence-level split does not exactly cover "
            "the eligible training sentences."
        )

    print()
    print("Sentence-level multilabel-stratified train/validation split:")
    print(
        f"  Eligible training sentences : {len(eligible_indices):,}"
    )
    print(
        f"  Target validation sentences : "
        f"{round(len(eligible_indices) * VALIDATION_FRACTION):,}"
    )
    print(
        f"  Actual validation sentences : {len(validation_indices):,}"
    )
    print(
        f"  Training sentences retained : {len(train_indices):,}"
    )
    print(
        f"  Split seed used             : "
        f"{best_candidate['split_seed_used']}"
    )
    print(
        f"  Stratification labels       : "
        f"{len(stratification_labels):,}"
    )
    print(
        f"  Coverage-rejected attempts  : {rejected_for_coverage:,}"
    )
    print(
        f"  Mean relative prevalence error: "
        f"{best_candidate['prevalence_error']:.6f}"
    )

    # Show sentence-level prevalence agreement, which is the quantity actually
    # controlled by the stratifier.
    validation_fraction_actual = (
        len(validation_indices) / len(eligible_indices)
    )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    row_by_sentence = {
        sentence_index: row
        for row, sentence_index in enumerate(eligible_indices)
    }

    train_rows_lookup = [
        row_by_sentence[index]
        for index in train_indices
    ]
    validation_rows_lookup = [
        row_by_sentence[index]
        for index in validation_indices
    ]

    print()
    print(
        f"{'Feature':35s} "
        f"{'All sent.':>10s} "
        f"{'Train':>10s} "
        f"{'Valid':>10s} "
        f"{'Valid %':>10s}"
    )
    print("-" * 80)

    for feature in stratification_labels:
        column = label2column[feature]
        all_support = int(sentence_matrix[:, column].sum())
        train_support = int(
            sentence_matrix[train_rows_lookup, column].sum()
        )
        validation_support = int(
            sentence_matrix[validation_rows_lookup, column].sum()
        )

        print(
            f"{feature:35s} "
            f"{all_support:10d} "
            f"{train_support:10d} "
            f"{validation_support:10d} "
            f"{validation_support / max(all_support, 1):10.3f}"
        )

    print(
        f"\nActual validation sentence fraction: "
        f"{validation_fraction_actual:.4f}"
    )
    print()

    return best_candidate


# =============================================================================
# Dataset
# =============================================================================

class SemanticFeatureDataset(Dataset):
    """
    RoBERTa multi-label token-classification dataset.

    IMPORTANT:
        labels contain only INDEPENDENT neural targets.

    Labels have shape:
        [sequence_length, number_of_prediction_features]

    For supervised tokens:
        0 = independent feature absent
        1 = independent feature present

    For ignored tokens:
        -100 across all labels

    Only the first RoBERTa subword corresponding to each source token receives
    supervision.
    """

    def __init__(
        self,
        sentences: Sequence[Any],
        sentence_indices: Sequence[int],
        tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
        tokenizer: Any,
        label2id: Dict[str, int],
        max_length: int,
    ) -> None:
        self.sentences = sentences
        self.sentence_indices = list(sentence_indices)
        self.tagged_records = tagged_records
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.num_labels = len(label2id)
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.sentence_indices)

    def __getitem__(
        self,
        dataset_index: int,
    ) -> Dict[str, Any]:
        sentence_index = self.sentence_indices[dataset_index]
        sentence = self.sentences[sentence_index]

        words, token_ids = get_sentence_words(sentence)

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        labels: List[List[float]] = []
        previous_word_id: Optional[int] = None
        sentence_records = self.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                labels.append([-100.0] * self.num_labels)
                previous_word_id = word_id
                continue

            # Ignore all but the first subword.
            if word_id == previous_word_id:
                labels.append([-100.0] * self.num_labels)
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                labels.append([-100.0] * self.num_labels)
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                labels.append([-100.0] * self.num_labels)
                continue

            full_features = clean_features(record)

            if not full_features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                labels.append([-100.0] * self.num_labels)
                continue

            independent_features = reduce_to_independent_features(
                full_features
            )

            target = [0.0] * self.num_labels

            for feature in independent_features:
                label_id = self.label2id.get(feature)

                if label_id is not None:
                    target[label_id] = 1.0

            labels.append(target)

        encoding["labels"] = labels
        encoding["sentence_index"] = sentence_index

        return encoding


# =============================================================================
# Data collator
# =============================================================================

class MultiLabelTokenCollator:
    """Dynamically pad inputs and 3-D multi-label token targets."""

    def __init__(
        self,
        tokenizer: Any,
        num_labels: int,
    ) -> None:
        self.tokenizer = tokenizer
        self.num_labels = num_labels

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        clean_features_list: List[Dict[str, Any]] = []
        labels_list: List[List[List[float]]] = []

        for item in features:
            item = dict(item)
            item.pop("sentence_index", None)

            labels = item.pop("labels")
            labels_list.append(labels)
            clean_features_list.append(item)

        batch = self.tokenizer.pad(
            clean_features_list,
            padding=True,
            return_tensors="pt",
        )

        batch_size = len(labels_list)
        max_seq_length = batch["input_ids"].shape[1]

        padded_labels = torch.full(
            (
                batch_size,
                max_seq_length,
                self.num_labels,
            ),
            -100.0,
            dtype=torch.float32,
        )

        for batch_index, labels in enumerate(labels_list):
            label_tensor = torch.tensor(
                labels,
                dtype=torch.float32,
            )

            sequence_length = label_tensor.shape[0]

            padded_labels[
                batch_index,
                :sequence_length,
                :,
            ] = label_tensor

        batch["labels"] = padded_labels

        return batch


# =============================================================================
# Model
# =============================================================================

class RobertaMultiLabelModel(nn.Module):
    """RoBERTa token classifier with BCE loss for independent feature labels."""

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        id2label: Dict[int, str],
        label2id: Dict[str, int],
    ) -> None:
        super().__init__()

        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )

        self.config = self.model.config

        self.loss_function = nn.BCEWithLogitsLoss(
            reduction="mean"
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        logits = outputs.logits
        loss = None

        if labels is not None:
            valid_mask = labels[:, :, 0] != -100.0

            valid_logits = logits[valid_mask]
            valid_labels = labels[valid_mask]

            if valid_logits.numel() == 0:
                loss = logits.sum() * 0.0
            else:
                loss = self.loss_function(
                    valid_logits,
                    valid_labels,
                )

        return {
            "loss": loss,
            "logits": logits,
        }

    def save_pretrained(
        self,
        save_directory,
        **kwargs,
    ):
        return self.model.save_pretrained(
            save_directory,
            **kwargs,
        )


# =============================================================================
# Hierarchy matrices and array helpers
# =============================================================================

def build_hierarchy_expansion_matrix(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
) -> np.ndarray:
    """
    Build a boolean matrix mapping neural outputs to complete feature sets.

    Shape:
        [number_of_prediction_features, number_of_full_features]
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    matrix = np.zeros(
        (len(prediction_labels), len(full_labels)),
        dtype=np.int8,
    )

    for prediction_id, feature in enumerate(prediction_labels):
        expanded = expand_feature_hierarchy({feature})

        for expanded_feature in expanded:
            full_id = full_label2id.get(expanded_feature)

            if full_id is None:
                raise RuntimeError(
                    "Hierarchy expansion produced a feature absent from the "
                    f"full vocabulary: {expanded_feature!r}."
                )

            matrix[prediction_id, full_id] = 1

    return matrix


def expand_binary_feature_matrix(
    independent_matrix: np.ndarray,
    hierarchy_matrix: np.ndarray,
) -> np.ndarray:
    """Expand [tokens, independent] binary values to [tokens, complete]."""

    expanded = independent_matrix.astype(np.int32) @ hierarchy_matrix.astype(
        np.int32
    )

    return (expanded > 0).astype(np.int32)


def extract_valid_token_arrays(
    logits: np.ndarray,
    labels: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return probabilities and true independent labels for supervised tokens."""

    valid_token_mask = labels[:, :, 0] != -100.0

    probabilities = sigmoid_numpy(logits)[
        valid_token_mask
    ]

    true_labels = labels[
        valid_token_mask
    ].astype(np.int32)

    return probabilities, true_labels


def build_full_truth_matrix_for_dataset(
    dataset: "SemanticFeatureDataset",
    full_labels: Sequence[str],
) -> np.ndarray:
    """Build complete source truth for every supervised token in dataset order.

    This is intentionally independent of the neural-output vocabulary.  It is
    therefore safe and necessary for a held-out test corpus that may contain a
    feature such as ``Human`` as a direct independent target even when that
    feature never occurred independently in the training corpus.

    The model is still trained only on training-derived independent output
    heads.  This function merely preserves the complete gold annotations for
    fair hierarchy-expanded evaluation.
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    rows: List[np.ndarray] = []

    for sentence_index in dataset.sentence_indices:
        sentence = dataset.sentences[sentence_index]
        words, token_ids = get_sentence_words(sentence)

        encoding = dataset.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=dataset.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        previous_word_id: Optional[int] = None
        sentence_records = dataset.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                previous_word_id = word_id
                continue

            # Match SemanticFeatureDataset.__getitem__: only the first subword
            # for each source token receives supervision.
            if word_id == previous_word_id:
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                continue

            features = clean_features(record)

            if not features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                continue

            expanded_features = expand_feature_hierarchy(features)
            row = np.zeros(len(full_labels), dtype=np.int32)

            for feature in expanded_features:
                label_id = full_label2id.get(feature)

                if label_id is None:
                    raise RuntimeError(
                        "Complete source truth contains a feature absent from "
                        f"the training-derived full vocabulary: {feature!r}."
                    )

                row[label_id] = 1

            rows.append(row)

    if not rows:
        return np.zeros((0, len(full_labels)), dtype=np.int32)

    return np.stack(rows, axis=0)


# =============================================================================
# Metrics and thresholds
# =============================================================================

def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""

    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


def make_threshold_vector(
    prediction_labels: Sequence[str],
    thresholds: Dict[str, float],
) -> np.ndarray:
    """Return thresholds in neural-output column order."""

    return np.asarray(
        [
            float(
                thresholds.get(
                    label,
                    DEFAULT_PREDICTION_THRESHOLD,
                )
            )
            for label in prediction_labels
        ],
        dtype=np.float32,
    )


def threshold_probabilities(
    probabilities: np.ndarray,
    threshold_vector: np.ndarray,
) -> np.ndarray:
    """Apply one threshold per independent feature."""

    return (
        probabilities >= threshold_vector.reshape(1, -1)
    ).astype(np.int32)


def calculate_average_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    prefix: str = "",
) -> Dict[str, float]:
    """Calculate micro and macro precision/recall/F1."""

    if y_true.shape[0] == 0:
        return {
            f"{prefix}precision_micro": 0.0,
            f"{prefix}recall_micro": 0.0,
            f"{prefix}f1_micro": 0.0,
            f"{prefix}precision_macro": 0.0,
            f"{prefix}recall_macro": 0.0,
            f"{prefix}f1_macro": 0.0,
        }

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        f"{prefix}precision_micro": float(precision_micro),
        f"{prefix}recall_micro": float(recall_micro),
        f"{prefix}f1_micro": float(f1_micro),
        f"{prefix}precision_macro": float(precision_macro),
        f"{prefix}recall_macro": float(recall_macro),
        f"{prefix}f1_macro": float(f1_macro),
    }


def calculate_hierarchical_metrics_from_probabilities(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    threshold_vector: np.ndarray,
    hierarchy_matrix: np.ndarray,
    y_true_full: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Report both direct neural-output metrics and complete hierarchy metrics.

    Unprefixed metrics are the COMPLETE hierarchy metrics.  During ordinary
    training/validation, complete truth can be reconstructed from the
    independent targets.  For a genuinely held-out test corpus, ``y_true_full``
    may instead be supplied directly from the source annotations so an unseen
    direct independent label is not silently erased from the gold truth.
    """

    y_pred_independent = threshold_probabilities(
        probabilities,
        threshold_vector,
    )

    if y_true_full is None:
        y_true_full = expand_binary_feature_matrix(
            y_true_independent,
            hierarchy_matrix,
        )

    if y_true_full.shape[0] != y_true_independent.shape[0]:
        raise RuntimeError(
            "Complete and independent truth matrices contain different "
            f"numbers of supervised tokens: {y_true_full.shape[0]} vs "
            f"{y_true_independent.shape[0]}."
        )

    y_pred_full = expand_binary_feature_matrix(
        y_pred_independent,
        hierarchy_matrix,
    )

    metrics = calculate_average_metrics(
        y_true_full,
        y_pred_full,
        prefix="",
    )

    metrics.update(
        calculate_average_metrics(
            y_true_independent,
            y_pred_independent,
            prefix="direct_",
        )
    )

    return metrics


def make_compute_metrics(
    prediction_labels: Sequence[str],
    hierarchy_matrix: np.ndarray,
):
    """Create Trainer metrics using the fixed default 0.5 thresholds."""

    default_thresholds = np.full(
        len(prediction_labels),
        DEFAULT_PREDICTION_THRESHOLD,
        dtype=np.float32,
    )

    def compute_metrics(eval_prediction) -> Dict[str, float]:
        logits, labels = eval_prediction

        probabilities, y_true_independent = extract_valid_token_arrays(
            logits,
            labels,
        )

        return calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=y_true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )

    return compute_metrics


def build_threshold_grid() -> np.ndarray:
    """Build an inclusive, stable validation threshold search grid."""

    if THRESHOLD_STEP <= 0:
        raise ValueError("THRESHOLD_STEP must be > 0.")

    if not 0.0 < THRESHOLD_MIN < 1.0:
        raise ValueError("THRESHOLD_MIN must be between 0 and 1.")

    if not 0.0 < THRESHOLD_MAX < 1.0:
        raise ValueError("THRESHOLD_MAX must be between 0 and 1.")

    if THRESHOLD_MIN > THRESHOLD_MAX:
        raise ValueError("THRESHOLD_MIN cannot exceed THRESHOLD_MAX.")

    count = int(
        round(
            (THRESHOLD_MAX - THRESHOLD_MIN)
            / THRESHOLD_STEP
        )
    ) + 1

    return np.linspace(
        THRESHOLD_MIN,
        THRESHOLD_MAX,
        count,
        dtype=np.float64,
    )


def binary_metrics_for_column(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Tuple[float, float, float, int]:
    """Return binary precision, recall, F1 and positive support."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    support = int(y_true.sum())

    return (
        float(precision),
        float(recall),
        float(f1),
        support,
    )


def optimise_feature_thresholds(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    prediction_labels: Sequence[str],
) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """
    Optimise one sigmoid threshold per independent feature on VALIDATION ONLY.

    Objective:
        maximise that feature's binary F1.

    Stability rules:
    - If validation has no positives or no negatives for a feature, keep 0.5.
    - If several thresholds tie for best F1, choose the one closest to 0.5.
      This avoids unnecessarily extreme thresholds for flat validation curves.
    """

    grid = build_threshold_grid()

    thresholds: Dict[str, float] = {}
    diagnostics: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(prediction_labels):
        y_true = y_true_independent[:, label_id]
        probs = probabilities[:, label_id]

        positive_support = int(y_true.sum())
        negative_support = int((1 - y_true).sum())

        default_pred = (
            probs >= DEFAULT_PREDICTION_THRESHOLD
        ).astype(np.int32)

        (
            default_precision,
            default_recall,
            default_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            default_pred,
        )

        if positive_support == 0 or negative_support == 0:
            best_threshold = DEFAULT_PREDICTION_THRESHOLD
        else:
            best_f1 = -1.0
            best_candidates: List[float] = []

            for threshold in grid:
                y_pred = (probs >= threshold).astype(np.int32)

                _, _, f1, _ = binary_metrics_for_column(
                    y_true,
                    y_pred,
                )

                if f1 > best_f1 + 1e-12:
                    best_f1 = f1
                    best_candidates = [float(threshold)]
                elif abs(f1 - best_f1) <= 1e-12:
                    best_candidates.append(float(threshold))

            best_threshold = min(
                best_candidates,
                key=lambda value: (
                    abs(value - DEFAULT_PREDICTION_THRESHOLD),
                    value,
                ),
            )

        best_pred = (
            probs >= best_threshold
        ).astype(np.int32)

        (
            best_precision,
            best_recall,
            best_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            best_pred,
        )

        thresholds[feature] = float(best_threshold)

        diagnostics[feature] = {
            "threshold": float(best_threshold),
            "support": positive_support,
            "negative_support": negative_support,
            "precision_default": default_precision,
            "recall_default": default_recall,
            "f1_default": default_f1,
            "precision_optimised": best_precision,
            "recall_optimised": best_recall,
            "f1_optimised": best_f1,
            "f1_change": best_f1 - default_f1,
        }

    return thresholds, diagnostics


def calculate_per_feature_metrics_from_binary(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    """Calculate binary precision/recall/F1 independently for each column."""

    results: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(labels):
        precision, recall, f1, support = binary_metrics_for_column(
            y_true[:, label_id],
            y_pred[:, label_id],
        )

        results[feature] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }

    return results


# =============================================================================
# Dataset diagnostics and validation
# =============================================================================

def count_feature_occurrences(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
    independent_only: bool = False,
) -> Counter:
    """Count complete or independent positive labels in a split."""

    counts: Counter = Counter()

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if independent_only:
                features = get_independent_record_features(record)
            else:
                features = clean_features(record)

            for feature in features:
                counts[feature] += 1

    return counts


def count_empty_records(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
) -> int:
    """Count NOUN/PROPN records having no complete feature tags."""

    count = 0

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if not clean_features(record):
                count += 1

    return count


def verify_full_label_coverage(
    full_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """Ensure validation/test contain no complete label unseen in training."""

    training_labels = set(full_labels)

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed: Set[str] = set()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                observed.update(clean_features(record))

        unknown = observed - training_labels

        if unknown:
            raise RuntimeError(
                f"{split_name} contains full labels unseen in training: "
                f"{sorted(unknown)}"
            )


def verify_prediction_label_coverage(
    prediction_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
    training_file_prediction_labels: Optional[Sequence[str]] = None,
) -> Dict[str, Dict[str, int]]:
    """Check independent-target coverage without leaking test labels.

    Validation remains strict: every validation independent target must have a
    training-derived neural output head.

    Test is different.  A genuine held-out corpus may contain a direct
    independent target that never occurred independently in training.  Such a
    label must NOT be added to the model from the test set.  Instead, report it
    and let complete-feature evaluation use the source gold matrix directly.
    """

    training_labels = set(prediction_labels)
    training_file_labels = (
        set(training_file_prediction_labels)
        if training_file_prediction_labels is not None
        else set(training_labels)
    )

    unseen_test_support: Dict[str, Dict[str, int]] = {}

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed_counts: Counter = Counter()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                for feature in get_independent_record_features(record):
                    observed_counts[feature] += 1

        observed = set(observed_counts)
        unknown = observed - training_labels

        if not unknown:
            continue

        lost_due_to_split = unknown & training_file_labels
        absent_from_training_file = unknown - training_file_labels

        if split_name == "validation":
            details: List[str] = []

            if lost_due_to_split:
                details.append(
                    "present somewhere in the training file but absent from "
                    f"the training subset: {sorted(lost_due_to_split)}"
                )

            if absent_from_training_file:
                details.append(
                    "never occur as independent targets anywhere in the "
                    f"training file: {sorted(absent_from_training_file)}"
                )

            raise RuntimeError(
                "validation contains independent labels with no learnable "
                "training output. "
                + "; ".join(details)
            )

        # For the official test set, do not modify the model vocabulary.
        unseen_test_support = {
            feature: {
                "support": int(observed_counts[feature]),
                "present_in_training_file_as_independent": int(
                    feature in training_file_labels
                ),
            }
            for feature in sorted(unknown)
        }

        print()
        print("=" * 80)
        print("HELD-OUT TEST LABEL COVERAGE WARNING")
        print("=" * 80)
        print(
            "The test corpus contains direct independent targets that have no "
            "training-derived neural output head. They will NOT be added to "
            "the model vocabulary. Complete test truth will be read directly "
            "from the source annotations instead."
        )

        for feature in sorted(unknown):
            print(
                f"  {feature:35s} support={observed_counts[feature]:,}"
            )

        print()

    return unseen_test_support


def verify_hierarchy_closure(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """
    Ensure source feature sets already contain all ancestors required by the
    hierarchy. This prevents evaluation against inconsistent source targets.
    """

    problems: List[str] = []
    max_examples = 20

    for split_name in ("train", "validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        problem_count = 0

        for sentence_index in sentence_indices:
            for token_id, record in tagged_records.get(
                sentence_index,
                {},
            ).items():
                features = clean_features(record)

                if not features:
                    continue

                missing = missing_hierarchy_features(features)

                if not missing:
                    continue

                problem_count += 1

                if len(problems) < max_examples:
                    problems.append(
                        f"{split_name}: sentence={sentence_index}, "
                        f"token={token_id}, "
                        f"form={record.get('form')!r}, "
                        f"missing={sorted(missing)}"
                    )

        if problem_count:
            print(
                f"Hierarchy closure problems in {split_name}: "
                f"{problem_count:,} token(s)"
            )

    if problems:
        example_text = "\n".join(
            f"  - {problem}"
            for problem in problems
        )

        raise RuntimeError(
            "The input dataset is not hierarchy-closed. Fix the source tags "
            "before training so deterministic reduction/expansion is exact.\n"
            f"Examples:\n{example_text}"
        )


def print_split_statistics(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    print("=" * 80)
    print("DATASET SPLIT")
    print("=" * 80)

    for split_name in ("train", "validation", "test"):
        tagged_records, indices = split_sources[split_name]

        full_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=False,
        )

        independent_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=True,
        )

        empty_count = count_empty_records(
            tagged_records,
            indices,
        )

        noun_count = sum(
            len(tagged_records.get(index, {}))
            for index in indices
        )

        source_name = (
            "training file"
            if split_name in {"train", "validation"}
            else "test file"
        )

        print(
            f"{split_name:12s}: "
            f"{len(indices):,} sentences | "
            f"{noun_count:,} NOUN/PROPN tokens | "
            f"{sum(full_counts.values()):,} complete positives -> "
            f"{sum(independent_counts.values()):,} independent positives | "
            f"{empty_count:,} empty-feature tokens excluded | "
            f"source={source_name}"
        )

    print()


def print_prediction_label_statistics(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    train_sentence_indices: Sequence[int],
) -> None:
    """Show which labels are neural outputs versus deterministic-only parents."""

    train_independent_counts = count_feature_occurrences(
        train_tagged_records,
        train_sentence_indices,
        independent_only=True,
    )

    prediction_set = set(prediction_labels)
    deterministic_only = [
        feature
        for feature in full_labels
        if feature not in prediction_set
    ]

    print("=" * 80)
    print("NEURAL OUTPUT FEATURES")
    print("=" * 80)
    print(
        f"Complete evaluation features: {len(full_labels)}"
    )
    print(
        f"Independent neural outputs: {len(prediction_labels)}"
    )
    print(
        f"Deterministic-only features in this training split: "
        f"{len(deterministic_only)}"
    )
    print()

    print(
        f"{'Feature':35s} "
        f"{'Independent train support':>26s}"
    )
    print("-" * 64)

    for feature in prediction_labels:
        print(
            f"{feature:35s} "
            f"{train_independent_counts[feature]:26d}"
        )

    if deterministic_only:
        print()
        print("Deterministic-only in this split:")

        for feature in deterministic_only:
            print(f"  - {feature}")

    print()


# =============================================================================
# Repeated-run and bootstrap helpers
# =============================================================================

def get_supervised_token_count_for_sentence(
    dataset: "SemanticFeatureDataset",
    sentence_index: int,
) -> int:
    """
    Count supervised first-subword targets for one source sentence.

    This mirrors SemanticFeatureDataset.__getitem__ exactly and is used to map
    flat prediction rows back to sentence clusters for test bootstrapping.
    """

    sentence = dataset.sentences[sentence_index]
    words, token_ids = get_sentence_words(sentence)

    encoding = dataset.tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=dataset.max_length,
        add_special_tokens=True,
    )

    word_ids = encoding.word_ids()
    previous_word_id: Optional[int] = None
    sentence_records = dataset.tagged_records.get(sentence_index, {})

    count = 0

    for word_id in word_ids:
        if word_id is None:
            previous_word_id = word_id
            continue

        if word_id == previous_word_id:
            continue

        previous_word_id = word_id

        if not 0 <= word_id < len(token_ids):
            continue

        conllu_token_id = token_ids[word_id]
        record = sentence_records.get(conllu_token_id)

        if record is None:
            continue

        features = clean_features(record)

        if not features and EXCLUDE_EMPTY_FEATURE_RECORDS:
            continue

        count += 1

    return count


def build_sentence_row_slices(
    dataset: "SemanticFeatureDataset",
) -> List[slice]:
    """
    Map every dataset sentence to its contiguous flat supervised-token rows.
    """

    slices: List[slice] = []
    start = 0

    for sentence_index in dataset.sentence_indices:
        count = get_supervised_token_count_for_sentence(
            dataset,
            sentence_index,
        )
        stop = start + count
        slices.append(slice(start, stop))
        start = stop

    return slices


def summarise_repeated_metrics(
    metric_dicts: Sequence[Dict[str, float]],
) -> Dict[str, Dict[str, float]]:
    """Return mean/SD/min/max across repeated seeded runs."""

    if not metric_dicts:
        return {}

    common_keys = sorted(
        set.intersection(
            *[
                {
                    key
                    for key, value in metrics.items()
                    if isinstance(
                        value,
                        (float, int, np.floating, np.integer),
                    )
                }
                for metrics in metric_dicts
            ]
        )
    )

    summary: Dict[str, Dict[str, float]] = {}

    for key in common_keys:
        values = np.asarray(
            [float(metrics[key]) for metrics in metric_dicts],
            dtype=np.float64,
        )

        summary[key] = {
            "mean": float(values.mean()),
            "sd": float(values.std(ddof=1))
            if len(values) > 1
            else 0.0,
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "n_runs": int(len(values)),
        }

    return summary


def _metrics_from_confusion_counts(
    tp_by_label: np.ndarray,
    fp_by_label: np.ndarray,
    fn_by_label: np.ndarray,
) -> Dict[str, float]:
    """Calculate six headline metrics directly from labelwise counts."""

    tp_by_label = tp_by_label.astype(np.float64, copy=False)
    fp_by_label = fp_by_label.astype(np.float64, copy=False)
    fn_by_label = fn_by_label.astype(np.float64, copy=False)

    tp = float(tp_by_label.sum())
    fp = float(fp_by_label.sum())
    fn = float(fn_by_label.sum())

    precision_micro = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_micro = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision_micro + recall_micro > 0:
        f1_micro = (
            2.0
            * precision_micro
            * recall_micro
            / (precision_micro + recall_micro)
        )
    else:
        f1_micro = 0.0

    precision_den = tp_by_label + fp_by_label
    recall_den = tp_by_label + fn_by_label

    precision_by_label = np.divide(
        tp_by_label,
        precision_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=precision_den > 0,
    )
    recall_by_label = np.divide(
        tp_by_label,
        recall_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=recall_den > 0,
    )

    f1_den = precision_by_label + recall_by_label
    f1_by_label = np.divide(
        2.0 * precision_by_label * recall_by_label,
        f1_den,
        out=np.zeros_like(precision_by_label, dtype=np.float64),
        where=f1_den > 0,
    )

    return {
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_micro": float(f1_micro),
        "precision_macro": float(precision_by_label.mean()),
        "recall_macro": float(recall_by_label.mean()),
        "f1_macro": float(f1_by_label.mean()),
    }


def build_sentence_confusion_arrays(
    y_true_full: np.ndarray,
    run_predictions_full: Sequence[np.ndarray],
    sentence_slices: Sequence[slice],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Precompute TP/FP/FN by [run, sentence, label] for fast cluster bootstrap.
    """

    n_runs = len(run_predictions_full)
    n_sentences = len(sentence_slices)
    n_labels = y_true_full.shape[1]

    tp = np.zeros(
        (n_runs, n_sentences, n_labels),
        dtype=np.int32,
    )
    fp = np.zeros_like(tp)
    fn = np.zeros_like(tp)

    for sentence_pos, row_slice in enumerate(sentence_slices):
        y_true_sentence = y_true_full[row_slice]

        for run_pos, y_pred_full in enumerate(run_predictions_full):
            y_pred_sentence = y_pred_full[row_slice]

            tp[run_pos, sentence_pos] = (
                (y_true_sentence == 1)
                & (y_pred_sentence == 1)
            ).sum(axis=0)

            fp[run_pos, sentence_pos] = (
                (y_true_sentence == 0)
                & (y_pred_sentence == 1)
            ).sum(axis=0)

            fn[run_pos, sentence_pos] = (
                (y_true_sentence == 1)
                & (y_pred_sentence == 0)
            ).sum(axis=0)

    return tp, fp, fn


def bootstrap_mean_test_metrics(
    y_true_full: np.ndarray,
    run_predictions_full: Sequence[np.ndarray],
    sentence_slices: Sequence[slice],
) -> Dict[str, Any]:
    """
    Sentence-cluster bootstrap CIs for the MEAN performance across run seeds.

    Each bootstrap replicate:
      1. samples test sentences with replacement;
      2. applies the SAME sampled sentence multiplicities to every trained run;
      3. computes each run's six hierarchy-expanded metrics;
      4. averages each metric across runs.

    Thus the interval quantifies held-out TEST-SAMPLE uncertainty around the
    mean performance of the repeated-run procedure. Training-seed variability is
    reported separately via mean and SD across runs.
    """

    if TEST_BOOTSTRAP_SAMPLES < 1:
        raise ValueError("TEST_BOOTSTRAP_SAMPLES must be at least 1.")

    if not 0.0 < TEST_BOOTSTRAP_CONFIDENCE < 1.0:
        raise ValueError(
            "TEST_BOOTSTRAP_CONFIDENCE must be strictly between 0 and 1."
        )

    if not run_predictions_full:
        raise ValueError(
            "At least one run prediction matrix is required for bootstrap."
        )

    n_sentences = len(sentence_slices)

    if n_sentences == 0:
        raise ValueError("No test sentences are available for bootstrap.")

    expected_rows = sum(
        row_slice.stop - row_slice.start
        for row_slice in sentence_slices
    )

    if expected_rows != y_true_full.shape[0]:
        raise RuntimeError(
            "Sentence slices do not cover the complete test truth matrix: "
            f"{expected_rows:,} rows vs {y_true_full.shape[0]:,}."
        )

    for run_pos, y_pred_full in enumerate(run_predictions_full):
        if y_pred_full.shape != y_true_full.shape:
            raise RuntimeError(
                f"Run {run_pos} prediction shape {y_pred_full.shape} does "
                f"not match test truth shape {y_true_full.shape}."
            )

    tp, fp, fn = build_sentence_confusion_arrays(
        y_true_full,
        run_predictions_full,
        sentence_slices,
    )

    rng = np.random.default_rng(TEST_BOOTSTRAP_SEED)

    metric_names = [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]

    bootstrap_values = {
        metric: np.empty(TEST_BOOTSTRAP_SAMPLES, dtype=np.float64)
        for metric in metric_names
    }

    for bootstrap_index in range(TEST_BOOTSTRAP_SAMPLES):
        sampled_sentences = rng.integers(
            0,
            n_sentences,
            size=n_sentences,
        )

        multiplicities = np.bincount(
            sampled_sentences,
            minlength=n_sentences,
        ).astype(np.int32)

        run_metric_values = {
            metric: []
            for metric in metric_names
        }

        for run_pos in range(len(run_predictions_full)):
            tp_by_label = np.tensordot(
                multiplicities,
                tp[run_pos],
                axes=(0, 0),
            )
            fp_by_label = np.tensordot(
                multiplicities,
                fp[run_pos],
                axes=(0, 0),
            )
            fn_by_label = np.tensordot(
                multiplicities,
                fn[run_pos],
                axes=(0, 0),
            )

            metrics = _metrics_from_confusion_counts(
                tp_by_label,
                fp_by_label,
                fn_by_label,
            )

            for metric in metric_names:
                run_metric_values[metric].append(metrics[metric])

        for metric in metric_names:
            bootstrap_values[metric][bootstrap_index] = float(
                np.mean(run_metric_values[metric])
            )

    alpha = 1.0 - TEST_BOOTSTRAP_CONFIDENCE
    lower_q = alpha / 2.0
    upper_q = 1.0 - lower_q

    intervals: Dict[str, Dict[str, float]] = {}

    for metric in metric_names:
        values = bootstrap_values[metric]

        intervals[metric] = {
            "bootstrap_mean": float(values.mean()),
            "bootstrap_sd": float(values.std(ddof=1))
            if len(values) > 1
            else 0.0,
            "ci_lower": float(np.quantile(values, lower_q)),
            "ci_upper": float(np.quantile(values, upper_q)),
            "confidence": float(TEST_BOOTSTRAP_CONFIDENCE),
            "bootstrap_samples": int(TEST_BOOTSTRAP_SAMPLES),
        }

    return {
        "bootstrap_unit": "sentence",
        "statistic": "mean_metric_across_repeated_training_runs",
        "seed": TEST_BOOTSTRAP_SEED,
        "n_test_sentences": n_sentences,
        "n_runs": len(run_predictions_full),
        "intervals": intervals,
    }


def make_json_serialisable(value: Any) -> Any:
    """Recursively convert NumPy scalars/arrays to ordinary JSON values."""

    if isinstance(value, dict):
        return {
            str(key): make_json_serialisable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_serialisable(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (np.floating, np.integer)):
        return value.item()

    return value


# =============================================================================
# Reporting helpers
# =============================================================================

def print_metric_dictionary(
    metrics: Dict[str, float],
) -> None:
    for key, value in sorted(metrics.items()):
        print(f"{key}: {value}")


def print_threshold_table(
    prediction_labels: Sequence[str],
    diagnostics: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 100)
    print("VALIDATION-OPTIMISED THRESHOLDS")
    print("=" * 100)

    heading = (
        f"{'Feature':35s} "
        f"{'Threshold':>10s} "
        f"{'Support':>9s} "
        f"{'F1@0.5':>10s} "
        f"{'F1 opt':>10s} "
        f"{'Change':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in prediction_labels:
        result = diagnostics[feature]

        print(
            f"{feature:35s} "
            f"{result['threshold']:10.2f} "
            f"{int(result['support']):9d} "
            f"{result['f1_default']:10.4f} "
            f"{result['f1_optimised']:10.4f} "
            f"{result['f1_change']:+10.4f}"
        )


def print_per_feature_table(
    title: str,
    labels: Sequence[str],
    results: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)

    heading = (
        f"{'Feature':35s} "
        f"{'Precision':>10s} "
        f"{'Recall':>10s} "
        f"{'F1':>10s} "
        f"{'Support':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = results[feature]

        print(
            f"{feature:35s} "
            f"{result['precision']:10.4f} "
            f"{result['recall']:10.4f} "
            f"{result['f1']:10.4f} "
            f"{result['support']:10d}"
        )


def summarise_repeated_per_feature_metrics(
    run_documents: Sequence[Dict[str, Any]],
    result_key: str,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    """Summarise per-feature test precision/recall/F1 across repeated runs."""

    summary: Dict[str, Dict[str, float]] = {}

    for feature in labels:
        precision_values = np.asarray(
            [
                run_document[result_key][feature]["precision"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        recall_values = np.asarray(
            [
                run_document[result_key][feature]["recall"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        f1_values = np.asarray(
            [
                run_document[result_key][feature]["f1"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        supports = {
            int(run_document[result_key][feature]["support"])
            for run_document in run_documents
        }

        if len(supports) != 1:
            raise RuntimeError(
                f"Test support changed across runs for feature {feature!r}: "
                f"{sorted(supports)}"
            )

        summary[feature] = {
            "precision_mean": float(precision_values.mean()),
            "precision_sd": float(precision_values.std(ddof=1))
            if len(precision_values) > 1
            else 0.0,
            "recall_mean": float(recall_values.mean()),
            "recall_sd": float(recall_values.std(ddof=1))
            if len(recall_values) > 1
            else 0.0,
            "f1_mean": float(f1_values.mean()),
            "f1_sd": float(f1_values.std(ddof=1))
            if len(f1_values) > 1
            else 0.0,
            "f1_min": float(f1_values.min()),
            "f1_max": float(f1_values.max()),
            "support": int(next(iter(supports))),
        }

    return summary


def print_repeated_per_feature_table(
    title: str,
    labels: Sequence[str],
    summary: Dict[str, Dict[str, float]],
) -> None:
    """Print repeated-run mean/SD test performance for each feature."""

    print()
    print("=" * 118)
    print(title)
    print("=" * 118)

    heading = (
        f"{'Feature':35s} "
        f"{'P mean':>8s} "
        f"{'P SD':>8s} "
        f"{'R mean':>8s} "
        f"{'R SD':>8s} "
        f"{'F1 mean':>8s} "
        f"{'F1 SD':>8s} "
        f"{'F1 min':>8s} "
        f"{'F1 max':>8s} "
        f"{'Support':>8s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = summary[feature]
        print(
            f"{feature:35s} "
            f"{result['precision_mean']:8.4f} "
            f"{result['precision_sd']:8.4f} "
            f"{result['recall_mean']:8.4f} "
            f"{result['recall_sd']:8.4f} "
            f"{result['f1_mean']:8.4f} "
            f"{result['f1_sd']:8.4f} "
            f"{result['f1_min']:8.4f} "
            f"{result['f1_max']:8.4f} "
            f"{int(result['support']):8d}"
        )


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )
    RUNS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    validate_feature_hierarchy()
    check_cuda()

    if not RUN_SEEDS:
        raise ValueError("RUN_SEEDS must contain at least one integer seed.")

    if len(set(RUN_SEEDS)) != len(RUN_SEEDS):
        raise ValueError("RUN_SEEDS contains duplicate seeds.")

    # -------------------------------------------------------------------------
    # Load independent TRAIN and held-out TEST corpora
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("LOADING DATA")
    print("=" * 80)

    train_sentences = load_conllu(TRAIN_CONLLU_PATH)
    test_sentences = load_conllu(TEST_CONLLU_PATH)

    train_tagged_records = load_tagged_records(TRAIN_JSONL_PATH)
    test_tagged_records = load_tagged_records(TEST_JSONL_PATH)

    validate_feature_switches(
        train_tagged_records,
        test_tagged_records,
    )

    print(
        "Training CoNLL-U sentences: "
        f"{len(train_sentences):,}"
    )
    print(
        "Test CoNLL-U sentences: "
        f"{len(test_sentences):,}"
    )
    print(
        "Training sentences represented in tagged JSONL: "
        f"{len(train_tagged_records):,}"
    )
    print(
        "Test sentences represented in tagged JSONL: "
        f"{len(test_tagged_records):,}"
    )

    train_target_record_count = sum(
        len(records)
        for records in train_tagged_records.values()
    )
    test_target_record_count = sum(
        len(records)
        for records in test_tagged_records.values()
    )

    print(
        "Training tagged NOUN/PROPN records: "
        f"{train_target_record_count:,}"
    )
    print(
        "Test tagged NOUN/PROPN records: "
        f"{test_target_record_count:,}"
    )
    print()

    # -------------------------------------------------------------------------
    # Fixed sentence-level TRAIN/VALIDATION split; TEST remains untouched
    # -------------------------------------------------------------------------

    eligible_train_indices = get_eligible_sentence_indices(
        train_sentences,
        train_tagged_records,
    )
    eligible_test_indices = get_eligible_sentence_indices(
        test_sentences,
        test_tagged_records,
    )

    print(
        "Eligible training-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_train_indices):,}"
    )
    print(
        "Eligible test-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_test_indices):,}"
    )

    train_validation_split = make_train_validation_split(
        eligible_train_indices,
        train_tagged_records,
    )

    train_indices = train_validation_split["train"]
    validation_indices = train_validation_split["validation"]
    test_indices = list(eligible_test_indices)

    split_sources = {
        "train": (train_tagged_records, train_indices),
        "validation": (train_tagged_records, validation_indices),
        "test": (test_tagged_records, test_indices),
    }

    split_document = {
        "split_unit": "sentence",
        "stratification": "iterative_multilabel_sentence_presence",
        "split_seed_requested": SPLIT_SEED,
        "split_seed_used": train_validation_split["split_seed_used"],
        "split_attempt": train_validation_split["attempt"],
        "validation_fraction": VALIDATION_FRACTION,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "prevalence_error": train_validation_split["prevalence_error"],
        "stratification_labels": (
            train_validation_split["stratification_labels"]
        ),
        "train_jsonl": str(TRAIN_JSONL_PATH),
        "test_jsonl": str(TEST_JSONL_PATH),
        "train_conllu": str(TRAIN_CONLLU_PATH),
        "test_conllu": str(TEST_CONLLU_PATH),
        "index_spaces": {
            "train": "training corpus",
            "validation": "training corpus",
            "test": "test corpus",
        },
        "train": train_indices,
        "validation": validation_indices,
        "test": test_indices,
    }

    with SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            make_json_serialisable(split_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    # The reduction is exact only when source data are hierarchy-closed.
    verify_hierarchy_closure(split_sources)

    # -------------------------------------------------------------------------
    # Separate complete and neural-output vocabularies
    # -------------------------------------------------------------------------

    training_file_prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        eligible_train_indices,
    )

    # Vocabularies are derived from TRAIN only, never validation/test.
    full_labels = build_full_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    verify_full_label_coverage(
        full_labels,
        split_sources,
    )

    unseen_test_independent_labels = verify_prediction_label_coverage(
        prediction_labels,
        split_sources,
        training_file_prediction_labels=training_file_prediction_labels,
    )

    prediction_label2id = {
        label: index
        for index, label in enumerate(prediction_labels)
    }

    prediction_id2label = {
        index: label
        for label, index in prediction_label2id.items()
    }

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    hierarchy_matrix = build_hierarchy_expansion_matrix(
        prediction_labels,
        full_labels,
    )

    label_document = {
        "labels": prediction_labels,
        "label2id": prediction_label2id,
        "id2label": {
            str(key): value
            for key, value in prediction_id2label.items()
        },
        "prediction_labels": prediction_labels,
        "prediction_label2id": prediction_label2id,
        "full_labels": full_labels,
        "full_label2id": full_label2id,
        "feature_parents": {
            child: sorted(parents)
            for child, parents in sorted(FEATURE_PARENTS.items())
        },
        "target_representation": "per_token_independent_features",
        "split_unit": "sentence",
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature
            for feature, enabled in FEATURE_SWITCHES.items()
            if not enabled
        ),
    }

    with LABELS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            label_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print_split_statistics(split_sources)

    print_prediction_label_statistics(
        prediction_labels,
        full_labels,
        train_tagged_records,
        train_indices,
    )

    # -------------------------------------------------------------------------
    # Tokeniser and fixed datasets
    # -------------------------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
        add_prefix_space=True,
    )

    train_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=train_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    validation_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=validation_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    test_dataset = SemanticFeatureDataset(
        sentences=test_sentences,
        sentence_indices=test_indices,
        tagged_records=test_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    data_collator = MultiLabelTokenCollator(
        tokenizer=tokenizer,
        num_labels=len(prediction_labels),
    )

    # Complete held-out gold truth is fixed across all training runs.
    test_true_full = build_full_truth_matrix_for_dataset(
        test_dataset,
        full_labels,
    )

    test_sentence_slices = build_sentence_row_slices(
        test_dataset,
    )

    test_slice_rows = sum(
        row_slice.stop - row_slice.start
        for row_slice in test_sentence_slices
    )

    if test_slice_rows != test_true_full.shape[0]:
        raise RuntimeError(
            "Test sentence-cluster mapping does not align with source truth: "
            f"{test_slice_rows:,} mapped rows vs "
            f"{test_true_full.shape[0]:,} truth rows."
        )

    # -------------------------------------------------------------------------
    # Repeated fine-tuning runs
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("REPEATED FINE-TUNING")
    print("=" * 80)
    print(f"Runs: {len(RUN_SEEDS)}")
    print(f"Seeds: {RUN_SEEDS}")
    print(
        "Train/validation split is fixed across all runs; only the "
        "fine-tuning seed changes."
    )
    print()

    use_bf16 = torch.cuda.is_bf16_supported()

    all_run_documents: List[Dict[str, Any]] = []
    all_test_metric_dicts: List[Dict[str, float]] = []
    all_validation_metric_dicts: List[Dict[str, float]] = []
    all_test_predictions_full: List[np.ndarray] = []

    work_root = OUTPUT_DIR / "_trainer_work"
    work_root.mkdir(parents=True, exist_ok=True)

    for run_number, run_seed in enumerate(RUN_SEEDS, start=1):
        print()
        print("=" * 100)
        print(
            f"RUN {run_number}/{len(RUN_SEEDS)} | "
            f"SEED {run_seed}"
        )
        print("=" * 100)

        set_seed(run_seed)

        run_work_dir = work_root / f"seed_{run_seed}"
        run_result_path = RUNS_DIR / f"seed_{run_seed}.json"

        if run_work_dir.exists():
            shutil.rmtree(run_work_dir)

        model = RobertaMultiLabelModel(
            model_name=MODEL_NAME,
            num_labels=len(prediction_labels),
            id2label=prediction_id2label,
            label2id=prediction_label2id,
        )

        training_args = TrainingArguments(
            output_dir=str(run_work_dir),

            learning_rate=LEARNING_RATE,
            lr_scheduler_type=LEARNING_RATE_SCHEDULER,
            weight_decay=WEIGHT_DECAY,

            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,

            gradient_accumulation_steps=(
                GRADIENT_ACCUMULATION_STEPS
            ),

            num_train_epochs=NUM_EPOCHS,

            eval_strategy=EVAL_STRATEGY,
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=50,

            load_best_model_at_end=True,

            # Model selection is performed on VALIDATION only, using complete
            # hierarchy-expanded macro F1 at the fixed 0.5 threshold.
            metric_for_best_model="f1_macro",
            greater_is_better=True,

            save_total_limit=2,

            bf16=use_bf16,
            fp16=not use_bf16,

            dataloader_num_workers=4,
            dataloader_pin_memory=True,

            optim="adamw_torch",
            report_to="none",

            seed=run_seed,
            data_seed=run_seed,

            remove_unused_columns=False,
            eval_accumulation_steps=16,
            warmup_steps = WARMUP_STEPS,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=make_compute_metrics(
                prediction_labels,
                hierarchy_matrix,
            ),
        )

        trainer.train()

        # ---------------------------------------------------------------------
        # Validation: first report 0.5, then calibrate thresholds on validation
        # ---------------------------------------------------------------------

        validation_output = trainer.predict(
            validation_dataset,
            metric_key_prefix="validation_default",
        )

        (
            validation_probabilities,
            validation_true_independent,
        ) = extract_valid_token_arrays(
            validation_output.predictions,
            validation_output.label_ids,
        )

        default_threshold_vector = np.full(
            len(prediction_labels),
            DEFAULT_PREDICTION_THRESHOLD,
            dtype=np.float32,
        )

        validation_default_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=validation_probabilities,
                y_true_independent=validation_true_independent,
                threshold_vector=default_threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
            )
        )

        (
            optimised_thresholds,
            threshold_diagnostics,
        ) = optimise_feature_thresholds(
            probabilities=validation_probabilities,
            y_true_independent=validation_true_independent,
            prediction_labels=prediction_labels,
        )

        threshold_vector = make_threshold_vector(
            prediction_labels,
            optimised_thresholds,
        )

        validation_optimised_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=validation_probabilities,
                y_true_independent=validation_true_independent,
                threshold_vector=threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
            )
        )

        print_threshold_table(
            prediction_labels,
            threshold_diagnostics,
        )

        print()
        print("VALIDATION @ 0.5")
        print_metric_dictionary(validation_default_metrics)

        print()
        print("VALIDATION WITH VALIDATION-CALIBRATED THRESHOLDS")
        print_metric_dictionary(validation_optimised_metrics)

        # ---------------------------------------------------------------------
        # Held-out TEST: thresholds are frozen
        # ---------------------------------------------------------------------

        test_output = trainer.predict(
            test_dataset,
            metric_key_prefix="test_default",
        )

        test_probabilities, test_true_independent = (
            extract_valid_token_arrays(
                test_output.predictions,
                test_output.label_ids,
            )
        )

        if test_true_full.shape[0] != test_probabilities.shape[0]:
            raise RuntimeError(
                "Test source-truth alignment failed: predictions contain "
                f"{test_probabilities.shape[0]:,} supervised tokens but "
                f"source truth contains {test_true_full.shape[0]:,}."
            )

        # Explicit test-at-0.5 diagnostic.
        test_default_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=test_probabilities,
                y_true_independent=test_true_independent,
                threshold_vector=default_threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
                y_true_full=test_true_full,
            )
        )

        # Final per-run test result uses validation-calibrated thresholds.
        test_optimised_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=test_probabilities,
                y_true_independent=test_true_independent,
                threshold_vector=threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
                y_true_full=test_true_full,
            )
        )

        test_pred_independent = threshold_probabilities(
            test_probabilities,
            threshold_vector,
        )

        test_pred_full = expand_binary_feature_matrix(
            test_pred_independent,
            hierarchy_matrix,
        )

        all_test_predictions_full.append(
            test_pred_full.astype(np.int8, copy=True)
        )

        per_feature_full = calculate_per_feature_metrics_from_binary(
            y_true=test_true_full,
            y_pred=test_pred_full,
            labels=full_labels,
        )

        per_feature_independent = (
            calculate_per_feature_metrics_from_binary(
                y_true=test_true_independent,
                y_pred=test_pred_independent,
                labels=prediction_labels,
            )
        )

        print()
        print("TEST @ 0.5 (DIAGNOSTIC)")
        print_metric_dictionary(test_default_metrics)

        print()
        print("TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS")
        print_metric_dictionary(test_optimised_metrics)

        if PRINT_PER_RUN_FEATURE_TABLES:
            print_per_feature_table(
                title=(
                    f"RUN {run_number} / SEED {run_seed}: TEST PERFORMANCE "
                    "BY COMPLETE FEATURE (AFTER INHERITANCE)"
                ),
                labels=full_labels,
                results=per_feature_full,
            )
            print_per_feature_table(
                title=(
                    f"RUN {run_number} / SEED {run_seed}: TEST PERFORMANCE "
                    "BY INDEPENDENT NEURAL OUTPUT"
                ),
                labels=prediction_labels,
                results=per_feature_independent,
            )

        # ---------------------------------------------------------------------
        # Save a neutral reference model, not a cherry-picked best seed
        # ---------------------------------------------------------------------

        is_reference_run = (
            SAVE_REFERENCE_MODEL
            and run_number == 1
        )

        if is_reference_run:
            if REFERENCE_MODEL_DIR.exists():
                shutil.rmtree(REFERENCE_MODEL_DIR)

            trainer.save_model(str(REFERENCE_MODEL_DIR))
            tokenizer.save_pretrained(str(REFERENCE_MODEL_DIR))

            with (
                REFERENCE_MODEL_DIR / "semantic_features.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    label_document,
                    handle,
                    ensure_ascii=False,
                    indent=2,
                )

            reference_threshold_document = {
                "seed": run_seed,
                "selection_policy": (
                    "first_predeclared_seed_not_best_test_run"
                ),
                "optimised_on": "validation",
                "objective": "per_feature_binary_f1",
                "thresholds": optimised_thresholds,
                "diagnostics": threshold_diagnostics,
            }

            with (
                REFERENCE_MODEL_DIR / "prediction_thresholds.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    make_json_serialisable(
                        reference_threshold_document
                    ),
                    handle,
                    ensure_ascii=False,
                    indent=2,
                )

        run_document = {
            "run_number": run_number,
            "seed": run_seed,
            "is_reference_model": is_reference_run,
            "model": MODEL_NAME,
            "target_representation": (
                "per_token_independent_features"
            ),
            "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
            "disabled_features": sorted(
                feature
                for feature, enabled in FEATURE_SWITCHES.items()
                if not enabled
            ),
            "split": {
                "unit": "sentence",
                "split_seed": (
                    train_validation_split["split_seed_used"]
                ),
                "validation_fraction": VALIDATION_FRACTION,
                "train_sentences": len(train_indices),
                "validation_sentences": len(validation_indices),
                "test_sentences": len(test_indices),
            },
            "threshold_calibration": {
                "data": "validation_only",
                "objective": "per_feature_binary_f1",
                "thresholds": optimised_thresholds,
                "diagnostics": threshold_diagnostics,
            },
            "validation_metrics_at_0_5": (
                validation_default_metrics
            ),
            "validation_metrics_optimised": (
                validation_optimised_metrics
            ),
            "test_metrics_at_0_5": test_default_metrics,
            "test_metrics_optimised": test_optimised_metrics,
            "per_feature_complete": per_feature_full,
            "per_feature_independent": (
                per_feature_independent
            ),
            "unseen_test_independent_labels": (
                unseen_test_independent_labels
            ),
        }

        with run_result_path.open(
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                make_json_serialisable(run_document),
                handle,
                ensure_ascii=False,
                indent=2,
            )

        all_run_documents.append(run_document)
        all_test_metric_dicts.append(test_optimised_metrics)
        all_validation_metric_dicts.append(
            validation_optimised_metrics
        )

        # Release run-specific GPU state before initialising the next model.
        del test_output
        del validation_output
        del trainer
        del model

        gc.collect()
        torch.cuda.empty_cache()

        # Checkpoints were needed only to recover the best validation epoch.
        # Remove them after each run to avoid storing dozens of RoBERTa-large
        # copies. The neutral first-seed reference model was saved separately.
        if run_work_dir.exists():
            shutil.rmtree(run_work_dir)

    if work_root.exists() and not any(work_root.iterdir()):
        work_root.rmdir()

    # -------------------------------------------------------------------------
    # Repeated-run summary: stochastic training variability
    # -------------------------------------------------------------------------

    validation_summary = summarise_repeated_metrics(
        all_validation_metric_dicts
    )
    test_summary = summarise_repeated_metrics(
        all_test_metric_dicts
    )

    print()
    print("=" * 100)
    print("REPEATED-RUN TEST SUMMARY")
    print("=" * 100)

    headline_metrics = [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]

    print(
        f"{'Metric':25s} "
        f"{'Mean':>10s} "
        f"{'SD':>10s} "
        f"{'Min':>10s} "
        f"{'Max':>10s}"
    )
    print("-" * 70)

    for metric in headline_metrics:
        result = test_summary[metric]

        print(
            f"{metric:25s} "
            f"{result['mean']:10.4f} "
            f"{result['sd']:10.4f} "
            f"{result['minimum']:10.4f} "
            f"{result['maximum']:10.4f}"
        )

    # -------------------------------------------------------------------------
    # Repeated-run per-feature test summaries
    # -------------------------------------------------------------------------

    per_feature_complete_summary = summarise_repeated_per_feature_metrics(
        all_run_documents,
        result_key="per_feature_complete",
        labels=full_labels,
    )
    per_feature_independent_summary = summarise_repeated_per_feature_metrics(
        all_run_documents,
        result_key="per_feature_independent",
        labels=prediction_labels,
    )

    print_repeated_per_feature_table(
        title=(
            "REPEATED-RUN TEST PERFORMANCE BY COMPLETE FEATURE "
            "(AFTER INHERITANCE)"
        ),
        labels=full_labels,
        summary=per_feature_complete_summary,
    )

    print_repeated_per_feature_table(
        title=(
            "REPEATED-RUN TEST PERFORMANCE BY INDEPENDENT NEURAL OUTPUT"
        ),
        labels=prediction_labels,
        summary=per_feature_independent_summary,
    )

    # -------------------------------------------------------------------------
    # Sentence-cluster bootstrap: held-out test-sample uncertainty
    # -------------------------------------------------------------------------

    print()
    print("=" * 100)
    print("SENTENCE-CLUSTER BOOTSTRAP ON HELD-OUT TEST SET")
    print("=" * 100)

    bootstrap_results = bootstrap_mean_test_metrics(
        y_true_full=test_true_full,
        run_predictions_full=all_test_predictions_full,
        sentence_slices=test_sentence_slices,
    )

    print(
        f"Bootstrap samples: {TEST_BOOTSTRAP_SAMPLES:,}"
    )
    print(
        f"Confidence level : "
        f"{TEST_BOOTSTRAP_CONFIDENCE:.1%}"
    )
    print(
        "Bootstrap unit   : sentence (all eligible NOUN/PROPN targets "
        "from a sampled sentence move together)"
    )
    print()

    ci_percent = int(round(TEST_BOOTSTRAP_CONFIDENCE * 100))

    print(
        f"{'Metric':25s} "
        f"{'Run mean':>10s} "
        f"{'Run SD':>10s} "
        f"{str(ci_percent) + '% CI low':>12s} "
        f"{str(ci_percent) + '% CI high':>12s}"
    )
    print("-" * 78)

    for metric in headline_metrics:
        repeated = test_summary[metric]
        interval = bootstrap_results["intervals"][metric]

        print(
            f"{metric:25s} "
            f"{repeated['mean']:10.4f} "
            f"{repeated['sd']:10.4f} "
            f"{interval['ci_lower']:12.4f} "
            f"{interval['ci_upper']:12.4f}"
        )

    # -------------------------------------------------------------------------
    # Save experiment-level results
    # -------------------------------------------------------------------------

    bootstrap_document = {
        "model": MODEL_NAME,
        "run_seeds": RUN_SEEDS,
        "split_seed": train_validation_split["split_seed_used"],
        "test_bootstrap": bootstrap_results,
    }

    with BOOTSTRAP_RESULTS_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            make_json_serialisable(bootstrap_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    repeated_document = {
        "model": MODEL_NAME,
        "experimental_design": {
            "test_set": "fixed_external_held_out_file",
            "train_validation_split_unit": "sentence",
            "train_validation_stratification": (
                "iterative_multilabel_sentence_presence"
            ),
            "split_seed": (
                train_validation_split["split_seed_used"]
            ),
            "validation_fraction": VALIDATION_FRACTION,
            "number_of_runs": len(RUN_SEEDS),
            "run_seeds": RUN_SEEDS,
            "same_split_for_every_run": True,
            "threshold_calibration": (
                "per_feature_binary_f1_on_validation_only"
            ),
            "test_bootstrap_unit": "sentence",
            "test_bootstrap_samples": TEST_BOOTSTRAP_SAMPLES,
            "test_bootstrap_confidence": (
                TEST_BOOTSTRAP_CONFIDENCE
            ),
        },
        "data_sources": {
            "train_jsonl": str(TRAIN_JSONL_PATH),
            "test_jsonl": str(TEST_JSONL_PATH),
            "train_conllu": str(TRAIN_CONLLU_PATH),
            "test_conllu": str(TEST_CONLLU_PATH),
        },
        "split_sizes": {
            "train_sentences": len(train_indices),
            "validation_sentences": len(validation_indices),
            "test_sentences": len(test_indices),
        },
        "prediction_labels": prediction_labels,
        "full_labels": full_labels,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature
            for feature, enabled in FEATURE_SWITCHES.items()
            if not enabled
        ),
        "unseen_test_independent_labels": (
            unseen_test_independent_labels
        ),
        "validation_repeated_run_summary": validation_summary,
        "test_repeated_run_summary": test_summary,
        "test_per_feature_complete_repeated_run_summary": (
            per_feature_complete_summary
        ),
        "test_per_feature_independent_repeated_run_summary": (
            per_feature_independent_summary
        ),
        "test_bootstrap": bootstrap_results,
        "runs": [
            {
                "run_number": run_document["run_number"],
                "seed": run_document["seed"],
                "is_reference_model": (
                    run_document["is_reference_model"]
                ),
                "result_file": str(
                    RUNS_DIR
                    / f"seed_{run_document['seed']}.json"
                ),
                "validation_metrics_optimised": (
                    run_document[
                        "validation_metrics_optimised"
                    ]
                ),
                "test_metrics_at_0_5": (
                    run_document["test_metrics_at_0_5"]
                ),
                "test_metrics_optimised": (
                    run_document["test_metrics_optimised"]
                ),
            }
            for run_document in all_run_documents
        ],
    }

    with REPEATED_RESULTS_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            make_json_serialisable(repeated_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print()
    print("=" * 100)
    print("OUTPUTS")
    print("=" * 100)
    print(
        "Fixed sentence split:\n"
        f"  {SPLIT_PATH.resolve()}"
    )
    print(
        "Semantic feature metadata:\n"
        f"  {LABELS_PATH.resolve()}"
    )
    print(
        "Per-run results:\n"
        f"  {RUNS_DIR.resolve()}"
    )
    print(
        "Repeated-run summary:\n"
        f"  {REPEATED_RESULTS_PATH.resolve()}"
    )
    print(
        "Test bootstrap confidence intervals:\n"
        f"  {BOOTSTRAP_RESULTS_PATH.resolve()}"
    )

    if SAVE_REFERENCE_MODEL:
        print(
            "Neutral reference model (first predeclared seed; not "
            "best-test selected):\n"
            f"  {REFERENCE_MODEL_DIR.resolve()}"
        )


if __name__ == "__main__":
    main()


/root/.pyenv/versions/3.10.16/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA
PyTorch version: 2.13.0+cu130
CUDA available: True
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.92 GiB

LOADING DATA
FEATURE SWITCHES
Disabled features: Rigidity, Shape, State, State_of_Matter
All unlisted features are enabled.

Training CoNLL-U sentences: 12,544
Test CoNLL-U sentences: 2,077
Training sentences represented in tagged JSONL: 11,511
Test sentences represented in tagged JSONL: 1,815
Training tagged NOUN/PROPN records: 47,371
Test tagged NOUN/PROPN records: 6,198

Eligible training-file sentences with at least one non-empty feature target: 11,198
Eligible test-file sentences with at least one non-empty feature target: 1,759

Sentence-level multilabel-stratified train/validation split:
  Eligible training sentences : 11,198
  Target validation sentences : 2,240
  Actual validation sentences : 2,313
  Training sentences retained : 8,885
  Split seed used             : 42
  Stratification labels       : 37
  Coverage-rejected attempts  : 0
  Mean relative p

Loading weights: 100%|██████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 1004.37it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.207690,0.044180,0.927027,0.864808,0.894837,0.827124,0.714831,0.750216,0.867592,0.787253,0.825472,0.726392,0.594526,0.629807
2,0.135570,0.033519,0.935426,0.905097,0.920012,0.896073,0.843001,0.863226,0.883085,0.858242,0.870486,0.856569,0.768604,0.793954
3,0.096490,0.032318,0.940322,0.920989,0.930555,0.890351,0.881520,0.883684,0.891138,0.870769,0.880836,0.865939,0.816817,0.826237
4,0.057375,0.030974,0.943405,0.928503,0.935895,0.916148,0.887277,0.900817,0.892913,0.881465,0.887152,0.885976,0.826188,0.847765
5,0.044076,0.031142,0.944013,0.932180,0.938059,0.914585,0.895896,0.904800,0.895024,0.888205,0.891602,0.899176,0.844273,0.863155



VALIDATION-OPTIMISED THRESHOLDS
Feature                              Threshold   Support     F1@0.5     F1 opt     Change
-----------------------------------------------------------------------------------------
Abstract                                  0.62      1405     0.8932     0.8978    +0.0046
Animal                                    0.39       184     0.9503     0.9532    +0.0029
Animate                                   0.46        13     0.8696     0.9167    +0.0471
Artefact                                  0.61       503     0.8998     0.9063    +0.0065
Body_Part                                 0.31        74     0.9167     0.9262    +0.0095
Comestible                                0.27       157     0.9712     0.9745    +0.0033
Communication                             0.44       534     0.8116     0.8145    +0.0028
Currency                                  0.76        45     0.9890     1.0000    +0.0110
Duration                                  0.69       340     0.8571

...


TEST @ 0.5 (DIAGNOSTIC)
direct_f1_macro: 0.7755572812348935
direct_f1_micro: 0.8620650443484194
direct_precision_macro: 0.8061542609647487
direct_precision_micro: 0.8651146867511126
direct_recall_macro: 0.7700790773620725
direct_recall_micro: 0.8590368271954674
f1_macro: 0.8525567308304566
f1_micro: 0.9164325671153235
precision_macro: 0.8632584311235468
precision_micro: 0.9209092248023958
recall_macro: 0.8480894494879525
recall_micro: 0.911999222092571

TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS
direct_f1_macro: 0.7781089163668287
direct_f1_micro: 0.8624743209312943
direct_precision_macro: 0.7900550350676052
direct_precision_micro: 0.8687205425911024
direct_recall_macro: 0.777416608369548
direct_recall_micro: 0.8563172804532578
f1_macro: 0.8513057146023986
f1_micro: 0.9133169533169533
precision_macro: 0.867728460873218
precision_micro: 0.9232068348897278
recall_macro: 0.8431495573284743
recall_micro: 0.9036367172306495

RUN 30/30 | SEED 1031


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 952.43it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.203807,0.045841,0.920098,0.850930,0.884164,0.820364,0.731008,0.765675,0.874441,0.773480,0.820868,0.756479,0.632962,0.675447
2,0.135210,0.034737,0.930402,0.904489,0.917263,0.878879,0.842643,0.855806,0.882568,0.850110,0.866035,0.843238,0.765058,0.787653
3,0.099294,0.031976,0.941279,0.912867,0.926855,0.905731,0.865565,0.882847,0.893461,0.866886,0.879973,0.896407,0.800542,0.826786
4,0.060902,0.031914,0.944212,0.918399,0.931127,0.918437,0.876285,0.895897,0.892310,0.878974,0.885592,0.905672,0.820185,0.848252
5,0.042254,0.031607,0.942155,0.930677,0.936381,0.906928,0.891249,0.898776,0.892192,0.888205,0.890194,0.885625,0.838339,0.852601



VALIDATION-OPTIMISED THRESHOLDS
Feature                              Threshold   Support     F1@0.5     F1 opt     Change
-----------------------------------------------------------------------------------------
Abstract                                  0.73      1405     0.8961     0.8984    +0.0023
Animal                                    0.43       184     0.9503     0.9532    +0.0029
Animate                                   0.50        13     0.9167     0.9167    +0.0000
Artefact                                  0.50       503     0.9049     0.9049    +0.0000
Body_Part                                 0.27        74     0.9103     0.9189    +0.0086
Comestible                                0.35       157     0.9484     0.9590    +0.0106
Communication                             0.37       534     0.8019     0.8095    +0.0076
Currency                                  0.66        45     0.9890     1.0000    +0.0110
Duration                                  0.90       340     0.8375


TEST @ 0.5 (DIAGNOSTIC)
direct_f1_macro: 0.7737578458884958
direct_f1_micro: 0.8660121446001929
direct_precision_macro: 0.8179179291778617
direct_precision_micro: 0.8674397453387903
direct_recall_macro: 0.759511119834006
direct_recall_micro: 0.8645892351274788
f1_macro: 0.8568531117045212
f1_micro: 0.918007176507921
precision_macro: 0.876731119719527
precision_micro: 0.9218098926417961
recall_macro: 0.8434767218131436
recall_micro: 0.9142357059509918

TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS
direct_f1_macro: 0.7776689085695722
direct_f1_micro: 0.8629880093197704
direct_precision_macro: 0.7983369104632524
direct_precision_micro: 0.865595075239398
direct_recall_macro: 0.7686052502698012
direct_recall_micro: 0.8603966005665722
f1_macro: 0.8535155187904646
f1_micro: 0.9167216641001198
precision_macro: 0.876238436670763
precision_micro: 0.9217912795556211
recall_macro: 0.8386159591309453
recall_micro: 0.91170750680669

REPEATED-RUN TEST SUMMARY
Metric                          Mean

<h1>Dataset stats</h1>

In [13]:
#!/usr/bin/env python3
"""
Inspect the TRAIN / VALIDATION / TEST composition used by the RoBERTa
semantic-feature experiment.

Compatible with Python 3.9.21.

Prints:
    - source sentence counts
    - sentence-level exclusions due to having no usable feature-bearing targets
    - retained train / validation / test sentence counts
    - empty-feature NOUN/PROPN records inside each retained split
    - tagged NOUN count
    - tagged PROPN count
    - total tagged NOUN/PROPN count
    - complete feature-frequency breakdown
    - independent-feature frequency breakdown
    - side-by-side feature-frequency comparison

The train/validation split follows the same procedure as the training script:
    1. Exclude training sentences with no non-empty NOUN/PROPN feature target.
    2. Reduce complete feature sets to independent features.
    3. Represent each sentence by independent-feature presence.
    4. Use iterative multilabel stratified splitting at SENTENCE level.
    5. Require every independent feature to retain at least one positive
       token target in the training subset.
    6. Use the untouched eligible official test sentences as the test set.
"""

from __future__ import annotations

import json
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence, Set, Tuple

import numpy as np

from conllu import parse_incr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


# =============================================================================
# Configuration
# =============================================================================

TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
)

TEST_JSONL_PATH = Path(
    "../data/ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl"
)

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

TEST_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-test.conllu"
)


# Exactly as in the training script.
VALIDATION_FRACTION = 0.20
SPLIT_SEED = 42

MIN_TRAIN_POSITIVES_PER_FEATURE = 1
MAX_STRATIFICATION_ATTEMPTS = 200

TARGET_POS = {"NOUN", "PROPN"}


# Exactly as in the supplied training script.
FEATURE_SWITCHES: Dict[str, bool] = {
    "State": False,
    "State_of_Matter": False,
    "Shape": False,
    "Rigidity": False,
}


# =============================================================================
# Feature hierarchy
# =============================================================================

FEATURE_PARENTS: Dict[str, Set[str]] = {
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
        "Natural",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },
    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },
    "Inanimate": {
        "Concrete",
    },
    "Garment": {
        "Artefact",
    },
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },
    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },
    "Time": {
        "Abstract",
    },
    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


def is_feature_enabled(feature: str) -> bool:
    return bool(FEATURE_SWITCHES.get(feature, True))


@lru_cache(maxsize=None)
def get_feature_ancestors(feature: str) -> frozenset:
    """
    Return all enabled direct and indirect parents.

    Traversal continues through disabled intermediate nodes, matching
    the training script.
    """

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(feature, set()):
        if is_feature_enabled(parent):
            ancestors.add(parent)

        ancestors.update(
            get_feature_ancestors(parent)
        )

    return frozenset(ancestors)


def reduce_to_independent_features(
    features: Iterable[str],
) -> Set[str]:
    """
    Remove a feature if it can be deterministically inherited from
    another feature present on the same token.
    """

    feature_set = set(features)

    inherited_from_other_features: Set[str] = set()

    for feature in feature_set:
        inherited_from_other_features.update(
            get_feature_ancestors(feature)
        )

    return feature_set - inherited_from_other_features


# =============================================================================
# Data loading
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load all CoNLL-U sentences."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence in parse_incr(handle):
            sentences.append(sentence)

    return sentences


def load_tagged_records(
    path: Path,
) -> Dict[int, Dict[int, Dict[str, Any]]]:
    """
    Load NOUN and PROPN records.

    Returns:

        sentence_index
            -> token_id
                -> JSON record
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find tagged JSONL:\n{path.resolve()}"
        )

    records: Dict[
        int,
        Dict[int, Dict[str, Any]]
    ] = defaultdict(dict)

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as exc:
                print(
                    f"Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if not isinstance(token_id, int):
                continue

            records[sentence_index][token_id] = record

    return dict(records)


# =============================================================================
# Feature handling
# =============================================================================

def clean_features(
    record: Dict[str, Any],
) -> List[str]:
    """
    Return unique enabled complete features.

    This deliberately honours FEATURE_SWITCHES exactly as the
    training script does.
    """

    raw = record.get("features", [])

    if not isinstance(raw, list):
        return []

    return sorted(
        {
            str(feature).strip()
            for feature in raw
            if (
                str(feature).strip()
                and is_feature_enabled(
                    str(feature).strip()
                )
            )
        }
    )


def get_independent_record_features(
    record: Dict[str, Any],
) -> List[str]:
    """Return the features used as direct neural targets."""

    return sorted(
        reduce_to_independent_features(
            clean_features(record)
        )
    )


# =============================================================================
# Eligible sentences
# =============================================================================

def get_eligible_sentence_indices(
    sentences: Sequence[Any],
    tagged_records: Dict[
        int,
        Dict[int, Dict[str, Any]]
    ],
) -> List[int]:
    """
    A sentence is eligible when it contains at least one NOUN/PROPN
    target with a non-empty enabled feature set.
    """

    eligible: List[int] = []

    for sentence_index in range(len(sentences)):

        token_records = tagged_records.get(
            sentence_index,
            {},
        )

        valid_records = [
            record
            for record in token_records.values()
            if clean_features(record)
        ]

        if valid_records:
            eligible.append(sentence_index)

    return eligible


# =============================================================================
# Sentence-level multilabel split
# =============================================================================

def get_sentence_independent_feature_counts(
    tagged_records: Dict[
        int,
        Dict[int, Dict[str, Any]]
    ],
    sentence_index: int,
) -> Counter:
    """
    Count independent positive token targets in one sentence.
    """

    counts: Counter = Counter()

    for record in tagged_records.get(
        sentence_index,
        {},
    ).values():

        if not clean_features(record):
            continue

        for feature in get_independent_record_features(
            record
        ):
            counts[feature] += 1

    return counts


def build_sentence_multilabel_matrix(
    eligible_indices: Sequence[int],
    tagged_records: Dict[
        int,
        Dict[int, Dict[str, Any]]
    ],
) -> Tuple[
    np.ndarray,
    List[str],
    Dict[int, Counter],
]:
    """
    Build the sentence-level presence matrix used by the original split.
    """

    sentence_feature_counts: Dict[int, Counter] = {
        sentence_index:
            get_sentence_independent_feature_counts(
                tagged_records,
                sentence_index,
            )
        for sentence_index in eligible_indices
    }

    stratification_labels = sorted(
        {
            feature
            for counts in sentence_feature_counts.values()
            for feature, count in counts.items()
            if count > 0
        }
    )

    if not stratification_labels:
        raise RuntimeError(
            "No independent features found in eligible training sentences."
        )

    label2column = {
        label: column
        for column, label
        in enumerate(stratification_labels)
    }

    matrix = np.zeros(
        (
            len(eligible_indices),
            len(stratification_labels),
        ),
        dtype=np.int8,
    )

    for row, sentence_index in enumerate(
        eligible_indices
    ):
        for feature in sentence_feature_counts[
            sentence_index
        ]:
            matrix[
                row,
                label2column[feature],
            ] = 1

    return (
        matrix,
        stratification_labels,
        sentence_feature_counts,
    )


def sentence_prevalence_error(
    y_all: np.ndarray,
    validation_rows: np.ndarray,
    validation_fraction: float,
) -> float:

    total_support = (
        y_all.sum(axis=0)
        .astype(np.float64)
    )

    validation_support = (
        y_all[validation_rows]
        .sum(axis=0)
        .astype(np.float64)
    )

    target_support = (
        total_support * validation_fraction
    )

    scale = np.maximum(
        target_support,
        1.0,
    )

    return float(
        np.mean(
            np.abs(
                validation_support
                - target_support
            )
            / scale
        )
    )


def make_train_validation_split(
    eligible_indices: Sequence[int],
    tagged_records: Dict[
        int,
        Dict[int, Dict[str, Any]]
    ],
) -> Dict[str, Any]:
    """
    Reproduce the training script's train/validation split.
    """

    eligible_indices = list(
        eligible_indices
    )

    (
        sentence_matrix,
        stratification_labels,
        sentence_feature_counts,
    ) = build_sentence_multilabel_matrix(
        eligible_indices,
        tagged_records,
    )

    all_token_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_token_counts.update(counts)

    x_dummy = np.zeros(
        (len(eligible_indices), 1),
        dtype=np.int8,
    )

    best_candidate = None
    rejected_for_coverage = 0

    for attempt in range(
        MAX_STRATIFICATION_ATTEMPTS
    ):

        attempt_seed = SPLIT_SEED + attempt

        splitter = (
            MultilabelStratifiedShuffleSplit(
                n_splits=1,
                test_size=VALIDATION_FRACTION,
                random_state=attempt_seed,
            )
        )

        train_rows, validation_rows = next(
            splitter.split(
                x_dummy,
                sentence_matrix,
            )
        )

        train_indices = [
            eligible_indices[int(row)]
            for row in train_rows
        ]

        validation_indices = [
            eligible_indices[int(row)]
            for row in validation_rows
        ]

        train_token_counts: Counter = Counter()

        for sentence_index in train_indices:
            train_token_counts.update(
                sentence_feature_counts[
                    sentence_index
                ]
            )

        missing_or_undercovered = sorted(
            feature
            for feature, total_count
            in all_token_counts.items()
            if (
                total_count > 0
                and train_token_counts[feature]
                < MIN_TRAIN_POSITIVES_PER_FEATURE
            )
        )

        if missing_or_undercovered:
            rejected_for_coverage += 1
            continue

        prevalence_error = (
            sentence_prevalence_error(
                sentence_matrix,
                validation_rows,
                VALIDATION_FRACTION,
            )
        )

        candidate = {
            "train": train_indices,
            "validation": validation_indices,
            "split_seed_used": attempt_seed,
            "attempt": attempt + 1,
            "prevalence_error": prevalence_error,
            "stratification_labels":
                stratification_labels,
        }

        if (
            best_candidate is None
            or prevalence_error
            < best_candidate["prevalence_error"]
        ):
            best_candidate = candidate

        # Matches the supplied training script.
        if rejected_for_coverage == 0:
            break

    if best_candidate is None:
        raise RuntimeError(
            "Could not construct a valid "
            "train/validation split."
        )

    return best_candidate


# =============================================================================
# Split statistics
# =============================================================================

def get_split_statistics(
    tagged_records: Dict[
        int,
        Dict[int, Dict[str, Any]]
    ],
    sentence_indices: Sequence[int],
) -> Dict[str, Any]:

    noun_count = 0
    propn_count = 0

    empty_noun_count = 0
    empty_propn_count = 0

    complete_feature_counts: Counter = Counter()
    independent_feature_counts: Counter = Counter()

    sentences_with_empty_records: Set[int] = set()

    for sentence_index in sentence_indices:

        sentence_records = tagged_records.get(
            sentence_index,
            {},
        )

        for record in sentence_records.values():

            upos = record.get("upos")

            features = clean_features(record)

            if not features:

                sentences_with_empty_records.add(
                    sentence_index
                )

                if upos == "NOUN":
                    empty_noun_count += 1

                elif upos == "PROPN":
                    empty_propn_count += 1

                continue

            # Only count genuinely tagged/usable records here.
            if upos == "NOUN":
                noun_count += 1

            elif upos == "PROPN":
                propn_count += 1

            complete_feature_counts.update(
                features
            )

            independent_feature_counts.update(
                get_independent_record_features(
                    record
                )
            )

    return {
        "sentences": len(sentence_indices),

        "tagged_nouns": noun_count,
        "tagged_proper_nouns": propn_count,

        "tagged_total":
            noun_count + propn_count,

        "empty_feature_nouns":
            empty_noun_count,

        "empty_feature_proper_nouns":
            empty_propn_count,

        "empty_feature_records":
            empty_noun_count
            + empty_propn_count,

        "sentences_containing_empty_records":
            len(sentences_with_empty_records),

        "complete_feature_counts":
            complete_feature_counts,

        "independent_feature_counts":
            independent_feature_counts,
    }


def print_feature_breakdown(
    counts: Counter,
    tagged_token_count: int,
) -> None:

    if not counts:
        print("  No features.")
        return

    print(
        f"  {'Feature':35s}"
        f"{'Count':>10s}"
        f"{'% tokens':>12s}"
    )

    print("  " + "-" * 57)

    for feature in sorted(counts):

        count = counts[feature]

        percentage = (
            100.0 * count / tagged_token_count
            if tagged_token_count
            else 0.0
        )

        print(
            f"  {feature:35s}"
            f"{count:10,d}"
            f"{percentage:11.2f}%"
        )


def print_split(
    name: str,
    statistics: Dict[str, Any],
) -> None:

    print()
    print("=" * 80)
    print(name.upper())
    print("=" * 80)

    print(
        f"Sentences retained                    : "
        f"{statistics['sentences']:,}"
    )

    print(
        f"Tagged NOUN tokens                   : "
        f"{statistics['tagged_nouns']:,}"
    )

    print(
        f"Tagged PROPN tokens                  : "
        f"{statistics['tagged_proper_nouns']:,}"
    )

    print(
        f"Tagged NOUN + PROPN tokens           : "
        f"{statistics['tagged_total']:,}"
    )

    print(
        f"Empty-feature NOUN records excluded  : "
        f"{statistics['empty_feature_nouns']:,}"
    )

    print(
        f"Empty-feature PROPN records excluded : "
        f"{statistics['empty_feature_proper_nouns']:,}"
    )

    print(
        f"All empty-feature records excluded   : "
        f"{statistics['empty_feature_records']:,}"
    )

    print(
        f"Sentences containing ≥1 empty record : "
        f"{statistics['sentences_containing_empty_records']:,}"
    )

    print()
    print("COMPLETE FEATURE FREQUENCIES")
    print()

    print_feature_breakdown(
        statistics["complete_feature_counts"],
        statistics["tagged_total"],
    )

    print()
    print("INDEPENDENT FEATURE FREQUENCIES")
    print()

    print_feature_breakdown(
        statistics[
            "independent_feature_counts"
        ],
        statistics["tagged_total"],
    )


def print_side_by_side_feature_table(
    train_stats: Dict[str, Any],
    validation_stats: Dict[str, Any],
    test_stats: Dict[str, Any],
) -> None:
    """
    Compare complete feature frequencies directly across splits.
    """

    train_counts = train_stats[
        "complete_feature_counts"
    ]

    validation_counts = validation_stats[
        "complete_feature_counts"
    ]

    test_counts = test_stats[
        "complete_feature_counts"
    ]

    all_features = sorted(
        set(train_counts)
        | set(validation_counts)
        | set(test_counts)
    )

    print()
    print("=" * 110)
    print("COMPLETE FEATURE DISTRIBUTION ACROSS SPLITS")
    print("=" * 110)

    print(
        f"{'Feature':32s}"
        f"{'Train':>10s}"
        f"{'Train %':>10s}"
        f"{'Valid':>10s}"
        f"{'Valid %':>10s}"
        f"{'Test':>10s}"
        f"{'Test %':>10s}"
    )

    print("-" * 92)

    train_total = train_stats["tagged_total"]
    validation_total = validation_stats[
        "tagged_total"
    ]
    test_total = test_stats["tagged_total"]

    for feature in all_features:

        train_count = train_counts[feature]
        validation_count = validation_counts[
            feature
        ]
        test_count = test_counts[feature]

        train_pct = (
            100.0 * train_count / train_total
            if train_total
            else 0.0
        )

        validation_pct = (
            100.0
            * validation_count
            / validation_total
            if validation_total
            else 0.0
        )

        test_pct = (
            100.0 * test_count / test_total
            if test_total
            else 0.0
        )

        print(
            f"{feature:32s}"
            f"{train_count:10,d}"
            f"{train_pct:9.2f}%"
            f"{validation_count:10,d}"
            f"{validation_pct:9.2f}%"
            f"{test_count:10,d}"
            f"{test_pct:9.2f}%"
        )


# =============================================================================
# Main
# =============================================================================

def main() -> None:

    # -------------------------------------------------------------------------
    # Load data
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("LOADING DATA")
    print("=" * 80)

    train_sentences = load_conllu(
        TRAIN_CONLLU_PATH
    )

    test_sentences = load_conllu(
        TEST_CONLLU_PATH
    )

    train_tagged_records = load_tagged_records(
        TRAIN_JSONL_PATH
    )

    test_tagged_records = load_tagged_records(
        TEST_JSONL_PATH
    )

    # -------------------------------------------------------------------------
    # Eligibility
    # -------------------------------------------------------------------------

    eligible_train_indices = (
        get_eligible_sentence_indices(
            train_sentences,
            train_tagged_records,
        )
    )

    eligible_test_indices = (
        get_eligible_sentence_indices(
            test_sentences,
            test_tagged_records,
        )
    )

    train_ineligible_indices = sorted(
        set(range(len(train_sentences)))
        - set(eligible_train_indices)
    )

    test_ineligible_indices = sorted(
        set(range(len(test_sentences)))
        - set(eligible_test_indices)
    )

    # -------------------------------------------------------------------------
    # Same train/validation split as training
    # -------------------------------------------------------------------------

    split = make_train_validation_split(
        eligible_train_indices,
        train_tagged_records,
    )

    train_indices = split["train"]
    validation_indices = split["validation"]

    # Official test set is not subdivided.
    test_indices = list(
        eligible_test_indices
    )

    # -------------------------------------------------------------------------
    # Important sentence-level exclusion summary
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("SENTENCE-LEVEL EXCLUSIONS")
    print("=" * 80)

    print()
    print("TRAINING CORPUS BEFORE TRAIN/VALIDATION SPLIT")
    print(
        f"Total source sentences              : "
        f"{len(train_sentences):,}"
    )
    print(
        f"Eligible sentences                  : "
        f"{len(eligible_train_indices):,}"
    )
    print(
        f"Excluded: no usable features        : "
        f"{len(train_ineligible_indices):,}"
    )
    print(
        f"Exclusion rate                      : "
        f"{100.0 * len(train_ineligible_indices) / len(train_sentences):.2f}%"
    )

    print()
    print("OFFICIAL TEST CORPUS")
    print(
        f"Total source sentences              : "
        f"{len(test_sentences):,}"
    )
    print(
        f"Eligible sentences                  : "
        f"{len(eligible_test_indices):,}"
    )
    print(
        f"Excluded: no usable features        : "
        f"{len(test_ineligible_indices):,}"
    )
    print(
        f"Exclusion rate                      : "
        f"{100.0 * len(test_ineligible_indices) / len(test_sentences):.2f}%"
    )

    print()
    print("TRAIN / VALIDATION SPLIT")
    print(
        f"Training sentences retained         : "
        f"{len(train_indices):,}"
    )
    print(
        f"Validation sentences retained       : "
        f"{len(validation_indices):,}"
    )
    print(
        f"Validation fraction                 : "
        f"{len(validation_indices) / len(eligible_train_indices):.4f}"
    )
    print(
        f"Split seed used                     : "
        f"{split['split_seed_used']}"
    )
    print(
        f"Split attempt                       : "
        f"{split['attempt']}"
    )

    print()
    print(
        "NOTE: the training-corpus sentence exclusions cannot be "
        "separately labelled 'train exclusions' and 'validation exclusions' "
        "because they are removed BEFORE the train/validation split is made."
    )

    # -------------------------------------------------------------------------
    # Detailed retained-split statistics
    # -------------------------------------------------------------------------

    train_stats = get_split_statistics(
        train_tagged_records,
        train_indices,
    )

    validation_stats = get_split_statistics(
        train_tagged_records,
        validation_indices,
    )

    test_stats = get_split_statistics(
        test_tagged_records,
        test_indices,
    )

    print_split(
        "Training",
        train_stats,
    )

    print_split(
        "Validation",
        validation_stats,
    )

    print_split(
        "Test",
        test_stats,
    )

    # -------------------------------------------------------------------------
    # Easy direct comparison
    # -------------------------------------------------------------------------

    print_side_by_side_feature_table(
        train_stats,
        validation_stats,
        test_stats,
    )


if __name__ == "__main__":
    main()

LOADING DATA

SENTENCE-LEVEL EXCLUSIONS

TRAINING CORPUS BEFORE TRAIN/VALIDATION SPLIT
Total source sentences              : 12,544
Eligible sentences                  : 11,198
Excluded: no usable features        : 1,346
Exclusion rate                      : 10.73%

OFFICIAL TEST CORPUS
Total source sentences              : 2,077
Eligible sentences                  : 1,759
Excluded: no usable features        : 318
Exclusion rate                      : 15.31%

TRAIN / VALIDATION SPLIT
Training sentences retained         : 8,885
Validation sentences retained       : 2,313
Validation fraction                 : 0.2066
Split seed used                     : 42
Split attempt                       : 1

NOTE: the training-corpus sentence exclusions cannot be separately labelled 'train exclusions' and 'validation exclusions' because they are removed BEFORE the train/validation split is made.

TRAINING
Sentences retained                    : 8,885
Tagged NOUN tokens                   : 25,277
Tag

<h1>Hyperparameter tuning</h1>

In [26]:
#!/usr/bin/env python3
"""
Fine-tune RoBERTa for hierarchical multi-label semantic feature tagging of
NOUN/PROPN tokens.

Compatible with Python 3.9.21.

Key design changes
------------------
1. The model is trained only on independent semantic features. For each token,
   any feature that can be inherited from a more specific present feature is
   removed from the neural target. This is a per-token reduction rather than a
   global "terminal leaf only" rule, so features such as Human, Time, Shape,
   Communication, etc. can still be predicted directly when they are the most
   specific available feature for that token.

2. All inherited features are reconstructed deterministically from
   FEATURE_PARENTS for validation, testing, reporting, and later inference.

3. A separate sigmoid threshold is optimised for every neural output using the
   validation split only. The selected thresholds are frozen before the test
   split is evaluated and are saved to prediction_thresholds.json.

4. Train/validation splitting is performed at SENTENCE level with iterative
   multilabel stratification over sentence-level independent-feature presence.
   The split is fixed across repeated fine-tuning runs.

5. A single fixed training seed is used for a validation-only grid search over
   learning rate and effective batch size. This selects a candidate setting
   for the later repeated-seed experiment, not a final performance estimate.

6. The official test corpus is not loaded or evaluated during tuning.

Tagged data:
    Training/validation:
        ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl

Sentence/token sources:
    Training/validation:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-train.conllu

The validation split is created only from the training corpus. The official
test corpus remains untouched until final evaluation.

Loss:
    BCEWithLogitsLoss over INDEPENDENT feature labels only.

Headline metrics:
    Computed after deterministic hierarchy expansion, so they describe the
    complete semantic feature sets rather than only the neural outputs.

Requirements:
    python -m pip install torch transformers datasets accelerate conllu scikit-learn iterative-stratification

CUDA is required.
"""

from __future__ import annotations

import csv
import gc
import json
import shutil
from itertools import product
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import torch
import torch.nn as nn

from conllu import parse_incr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import Dataset

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)


# =============================================================================
# Configuration
# =============================================================================

TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_train_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

# MODEL_NAME = "FacebookAI/roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"

OUTPUT_DIR = Path("../models/roberta_semantic_features_grid_search")

# Validation is sampled only from eligible training sentences.
VALIDATION_FRACTION = 0.20

# Fixed seed for the sentence-level train/validation split.
SPLIT_SEED = 42

# All grid candidates use the same random seed and split.
TUNING_SEED = 11

# 3 learning rates x 4 batch configurations = 12 training runs.
LEARNING_RATES = [1.5e-5, 2.0e-5, 2.5e-5, 3.0e-5, 3.5e-5, 4.0e-5, 4.5e-5, 5.0e-5]
#LEARNING_RATES = [4.5e-5, 5.0e-5]
LEARNING_RATE_SCHEDULER = "linear"
PER_DEVICE_TRAIN_BATCH_SIZES = [4, 8]
EFFECTIVE_BATCH_SIZES = [16]
WARMUP_STEPS=278
# Gradient accumulation is effective_batch_size // per_device_train_batch_size.
# For a single GPU, the four configurations are (4, 4), (8, 2),
# (4, 8), and (8, 4), where pairs mean (device batch, accumulation).

# The multilabel stratifier is retried with deterministic derived seeds until
# every independent feature still has at least this many positive token targets
# in the TRAIN subset. This prevents a rare direct target from disappearing.
MIN_TRAIN_POSITIVES_PER_FEATURE = 1
MAX_STRATIFICATION_ATTEMPTS = 200

MAX_LENGTH = 256

WEIGHT_DECAY = 0.01

EVAL_BATCH_SIZE = 16

NUM_EPOCHS = 5

# Threshold used during epoch-by-epoch Trainer evaluation. Final thresholds are
# optimised per independent feature on the validation set after training.
DEFAULT_PREDICTION_THRESHOLD = 0.5

# Validation-only threshold search grid.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.01

# Only these POS categories receive feature supervision.
TARGET_POS = {"NOUN", "PROPN"}

# A NOUN/PROPN with no features is considered a failed source annotation rather
# than a valid all-negative target.
EXCLUDE_EMPTY_FEATURE_RECORDS = True

# Optional per-feature switches. Any feature not listed here is enabled.
# Set a feature to False to remove it from:
#   - sentence-level stratification,
#   - neural training targets,
#   - threshold calibration,
#   - hierarchy-expanded gold/predicted feature sets,
#   - per-feature and aggregate evaluation, and
#   - the sentence-cluster bootstrap.
#
# Disabled intermediate hierarchy nodes are skipped but their enabled ancestors
# are still inherited. For example, if Human were disabled, Feminine would still
# inherit Animate/Natural/Concrete/Internal_Control via the hierarchy.
FEATURE_SWITCHES: Dict[str, bool] = {
    "State": False,
    "State_of_Matter": False,
    "Shape": False,
    "Rigidity": False,
}

EVAL_STRATEGY = "epoch"

SPLIT_PATH = OUTPUT_DIR / "sentence_split.json"
LABELS_PATH = OUTPUT_DIR / "semantic_features.json"
GRID_RESULTS_PATH = OUTPUT_DIR / "hyperparameter_grid_results.json"
GRID_CSV_PATH = OUTPUT_DIR / "hyperparameter_grid_results.csv"


# =============================================================================
# Feature hierarchy
# =============================================================================

def is_feature_enabled(feature: str) -> bool:
    """Return whether *feature* participates in this experiment."""

    return bool(FEATURE_SWITCHES.get(feature, True))


def validate_feature_switches(
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> None:
    """Fail early on misspelled feature-switch names and report disabled labels."""

    observed: Set[str] = set()

    for source in (train_tagged_records,):
        for sentence_records in source.values():
            for record in sentence_records.values():
                raw = record.get("features", [])
                if isinstance(raw, list):
                    observed.update(
                        str(feature).strip()
                        for feature in raw
                        if str(feature).strip()
                    )

    known = set(observed) | set(FEATURE_PARENTS)
    for parents in FEATURE_PARENTS.values():
        known.update(parents)

    unknown = sorted(set(FEATURE_SWITCHES) - known)
    if unknown:
        raise ValueError(
            "FEATURE_SWITCHES contains unknown feature name(s): "
            + ", ".join(unknown)
        )

    disabled = sorted(
        feature for feature, enabled in FEATURE_SWITCHES.items() if not enabled
    )

    print("=" * 80)
    print("FEATURE SWITCHES")
    print("=" * 80)
    if disabled:
        print("Disabled features: " + ", ".join(disabled))
    else:
        print("Disabled features: none")
    print("All unlisted features are enabled.")
    print()


# child -> direct parent(s)
FEATURE_PARENTS: Dict[str, Set[str]] = {
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
        "Natural",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },
    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },
    "Inanimate": {
        "Concrete",
    },
    "Garment": {
        "Artefact",
    },
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },
    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },
    "Time": {
        "Abstract",
    },
    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


def validate_feature_hierarchy() -> None:
    """Fail early if FEATURE_PARENTS contains a cycle or invalid name."""

    all_features: Set[str] = set(FEATURE_PARENTS)

    for parents in FEATURE_PARENTS.values():
        all_features.update(parents)

    for feature in all_features:
        if not isinstance(feature, str) or not feature.strip():
            raise ValueError(
                "FEATURE_PARENTS contains an empty or non-string feature name."
            )

    visiting: Set[str] = set()
    visited: Set[str] = set()

    def visit(feature: str) -> None:
        if feature in visited:
            return

        if feature in visiting:
            raise ValueError(
                f"Cycle detected in FEATURE_PARENTS involving {feature!r}."
            )

        visiting.add(feature)

        for parent in FEATURE_PARENTS.get(feature, set()):
            visit(parent)

        visiting.remove(feature)
        visited.add(feature)

    for feature in sorted(all_features):
        visit(feature)


@lru_cache(maxsize=None)
def get_feature_ancestors(feature: str) -> frozenset:
    """Return every direct and indirect parent of *feature*."""

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(feature, set()):
        if is_feature_enabled(parent):
            ancestors.add(parent)
        # Traverse even through a disabled intermediate node so enabled
        # grandparents remain inherited.
        ancestors.update(get_feature_ancestors(parent))

    return frozenset(ancestors)


def expand_feature_hierarchy(features: Iterable[str]) -> Set[str]:
    """Add every deterministic ancestor required by FEATURE_PARENTS."""

    expanded: Set[str] = {
        feature for feature in features if is_feature_enabled(feature)
    }

    for feature in list(expanded):
        expanded.update(get_feature_ancestors(feature))

    return expanded


def reduce_to_independent_features(features: Iterable[str]) -> Set[str]:
    """
    Reduce a complete feature set to the features that must be predicted.

    A feature is redundant when it is an ancestor of another feature present
    for the same token. Such an ancestor can be reconstructed deterministically.

    Examples:
        {Human, Animate, Natural, Concrete, Internal_Control}
            -> {Human}

        {Feminine, Human, Animate, Natural, Concrete, Internal_Control}
            -> {Feminine}

        {Statement, Communication, Abstract, Time}
            -> {Statement, Time}

    This is deliberately PER TOKEN. It is not the same as training only labels
    that are global terminal leaves in the ontology. A generic Human can still
    be a direct neural target if no more specific Human child is present.
    """

    feature_set = set(features)

    inherited_from_other_features: Set[str] = set()

    for feature in feature_set:
        inherited_from_other_features.update(
            get_feature_ancestors(feature)
        )

    return feature_set - inherited_from_other_features


def missing_hierarchy_features(features: Iterable[str]) -> Set[str]:
    """Return ancestors required by the hierarchy but absent from *features*."""

    feature_set = set(features)
    return expand_feature_hierarchy(feature_set) - feature_set


# =============================================================================
# CUDA
# =============================================================================

def check_cuda() -> None:
    """Require CUDA rather than silently falling back to the CPU."""

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. This script is configured to require "
            "a CUDA-capable GPU."
        )

    print("=" * 80)
    print("CUDA")
    print("=" * 80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM: "
        f"{props.total_memory / (1024 ** 3):.2f} GiB"
    )
    print()


# =============================================================================
# Load CoNLL-U
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load all CoNLL-U sentences."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence in parse_incr(handle):
            sentences.append(sentence)

    return sentences


def get_sentence_words(sentence: Any) -> Tuple[List[str], List[int]]:
    """
    Return ordinary integer-ID tokens and their original CoNLL-U token IDs.

    Multiword-token placeholder rows and empty nodes are excluded.
    """

    words: List[str] = []
    token_ids: List[int] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        form = token.get("form")

        if form is None:
            continue

        words.append(str(form))
        token_ids.append(token_id)

    return words, token_ids


# =============================================================================
# Load tagged JSONL
# =============================================================================

def load_tagged_records(
    path: Path,
) -> Dict[int, Dict[int, Dict[str, Any]]]:
    """
    Return:

        sentence_index
            -> token_id
                -> JSONL record

    Only NOUN and PROPN records are retained.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find tagged JSONL:\n{path.resolve()}"
        )

    records: Dict[int, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    "Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if not isinstance(token_id, int):
                continue

            records[sentence_index][token_id] = record

    return dict(records)


# =============================================================================
# Labels
# =============================================================================

def clean_features(record: Dict[str, Any]) -> List[str]:
    """Return unique, non-empty COMPLETE feature names from a JSONL record."""

    raw = record.get("features", [])

    if not isinstance(raw, list):
        return []

    return sorted(
        {
            str(feature).strip()
            for feature in raw
            if (
                str(feature).strip()
                and is_feature_enabled(str(feature).strip())
            )
        }
    )


def get_independent_record_features(record: Dict[str, Any]) -> List[str]:
    """Return the per-token minimal feature set that the model must predict."""

    return sorted(
        reduce_to_independent_features(
            clean_features(record)
        )
    )


def build_full_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """
    Build the COMPLETE evaluation vocabulary from training data only.

    Deterministic ancestors are included even if a training record omitted one,
    although hierarchy closure is separately verified and should normally make
    this addition unnecessary.
    """

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            features = clean_features(record)
            labels.update(features)
            labels.update(expand_feature_hierarchy(features))

    if not labels:
        raise RuntimeError(
            "No semantic features were found in the training split."
        )

    return sorted(labels)


def build_prediction_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """Build the neural-output vocabulary from independent training targets."""

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            labels.update(
                get_independent_record_features(record)
            )

    if not labels:
        raise RuntimeError(
            "No independent semantic features were found in the training split."
        )

    return sorted(labels)


# =============================================================================
# Sentence selection and splitting
# =============================================================================

def get_eligible_sentence_indices(
    sentences: Sequence[Any],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> List[int]:
    """Eligible sentences contain at least one non-empty NOUN/PROPN target."""

    eligible: List[int] = []

    for sentence_index in range(len(sentences)):
        token_records = tagged_records.get(sentence_index, {})

        valid_records = [
            record
            for record in token_records.values()
            if clean_features(record)
        ]

        if valid_records:
            eligible.append(sentence_index)

    return eligible


def get_sentence_independent_feature_counts(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_index: int,
) -> Counter:
    """Count independent positive token targets in one source sentence."""

    counts: Counter = Counter()

    for record in tagged_records.get(sentence_index, {}).values():
        if not clean_features(record):
            continue

        for feature in get_independent_record_features(record):
            counts[feature] += 1

    return counts


def build_sentence_multilabel_matrix(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Tuple[np.ndarray, List[str], Dict[int, Counter]]:
    """
    Build one multilabel stratification row per SENTENCE.

    y_sentence[sentence, feature] is 1 when at least one eligible NOUN/PROPN
    token in that sentence has the independent feature. Token counts are kept
    separately for the post-split training-coverage constraint.
    """

    sentence_feature_counts: Dict[int, Counter] = {
        sentence_index: get_sentence_independent_feature_counts(
            tagged_records,
            sentence_index,
        )
        for sentence_index in eligible_indices
    }

    stratification_labels = sorted(
        {
            feature
            for counts in sentence_feature_counts.values()
            for feature, count in counts.items()
            if count > 0
        }
    )

    if not stratification_labels:
        raise RuntimeError(
            "No independent semantic features were found in eligible "
            "training sentences."
        )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    matrix = np.zeros(
        (len(eligible_indices), len(stratification_labels)),
        dtype=np.int8,
    )

    for row, sentence_index in enumerate(eligible_indices):
        for feature in sentence_feature_counts[sentence_index]:
            matrix[row, label2column[feature]] = 1

    return matrix, stratification_labels, sentence_feature_counts


def _sentence_prevalence_error(
    y_all: np.ndarray,
    validation_rows: np.ndarray,
    validation_fraction: float,
) -> float:
    """
    Score how closely validation sentence-level label prevalences match target.

    Lower is better. The score is used only to choose among deterministic
    iterative-stratification attempts that satisfy the training-coverage rule.
    """

    total_support = y_all.sum(axis=0).astype(np.float64)
    validation_support = y_all[validation_rows].sum(axis=0).astype(np.float64)

    target_support = total_support * float(validation_fraction)

    # Scale by at least one expected example so rare and common labels both
    # contribute without allowing extremely common labels to dominate.
    scale = np.maximum(target_support, 1.0)

    return float(
        np.mean(
            np.abs(validation_support - target_support) / scale
        )
    )


def make_train_validation_split(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Dict[str, Any]:
    """
    Create a SENTENCE-LEVEL multilabel-stratified split from TRAINING only.

    Each sentence receives a binary vector indicating which independent feature
    labels occur anywhere among its eligible NOUN/PROPN tokens. Iterative
    multilabel stratification then assigns whole sentences to train/validation.

    The split is retried deterministically when necessary so that every
    independent feature retains at least MIN_TRAIN_POSITIVES_PER_FEATURE
    positive TOKEN targets in the training subset.

    The official test corpus is deliberately not accepted by this function, so
    test labels cannot influence the split.
    """

    if not 0.0 < VALIDATION_FRACTION < 1.0:
        raise ValueError(
            "VALIDATION_FRACTION must be strictly between 0 and 1."
        )

    if MIN_TRAIN_POSITIVES_PER_FEATURE < 1:
        raise ValueError(
            "MIN_TRAIN_POSITIVES_PER_FEATURE must be at least 1."
        )

    if MAX_STRATIFICATION_ATTEMPTS < 1:
        raise ValueError(
            "MAX_STRATIFICATION_ATTEMPTS must be at least 1."
        )

    eligible_indices = list(eligible_indices)

    if len(eligible_indices) < 2:
        raise RuntimeError(
            "Need at least two eligible training sentences to create a "
            "train/validation split."
        )

    (
        sentence_matrix,
        stratification_labels,
        sentence_feature_counts,
    ) = build_sentence_multilabel_matrix(
        eligible_indices,
        tagged_records,
    )

    all_token_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_token_counts.update(counts)

    x_dummy = np.zeros((len(eligible_indices), 1), dtype=np.int8)

    best_candidate: Optional[Dict[str, Any]] = None
    rejected_for_coverage = 0

    for attempt in range(MAX_STRATIFICATION_ATTEMPTS):
        attempt_seed = SPLIT_SEED + attempt

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=VALIDATION_FRACTION,
            random_state=attempt_seed,
        )

        train_rows, validation_rows = next(
            splitter.split(x_dummy, sentence_matrix)
        )

        train_indices = [
            eligible_indices[int(row)]
            for row in train_rows
        ]
        validation_indices = [
            eligible_indices[int(row)]
            for row in validation_rows
        ]

        train_token_counts: Counter = Counter()

        for sentence_index in train_indices:
            train_token_counts.update(
                sentence_feature_counts[sentence_index]
            )

        missing_or_undercovered = sorted(
            feature
            for feature, total_count in all_token_counts.items()
            if (
                total_count > 0
                and train_token_counts[feature]
                < MIN_TRAIN_POSITIVES_PER_FEATURE
            )
        )

        if missing_or_undercovered:
            rejected_for_coverage += 1
            continue

        prevalence_error = _sentence_prevalence_error(
            sentence_matrix,
            validation_rows,
            VALIDATION_FRACTION,
        )

        candidate = {
            "train": train_indices,
            "validation": validation_indices,
            "split_seed_used": attempt_seed,
            "attempt": attempt + 1,
            "prevalence_error": prevalence_error,
            "stratification_labels": stratification_labels,
        }

        if (
            best_candidate is None
            or prevalence_error < best_candidate["prevalence_error"]
        ):
            best_candidate = candidate

        # Iterative stratification is already designed to balance multilabel
        # prevalence. Once a valid candidate is found, further attempts are not
        # needed unless coverage forced earlier candidates to be rejected.
        if rejected_for_coverage == 0:
            break

    if best_candidate is None:
        raise RuntimeError(
            "Could not create a sentence-level multilabel-stratified "
            "train/validation split that preserved the required training "
            f"support after {MAX_STRATIFICATION_ATTEMPTS} attempts."
        )

    train_indices = best_candidate["train"]
    validation_indices = best_candidate["validation"]

    train_set = set(train_indices)
    validation_set = set(validation_indices)

    if train_set & validation_set:
        raise RuntimeError(
            "Internal split error: at least one sentence occurs in both "
            "training and validation."
        )

    if train_set | validation_set != set(eligible_indices):
        raise RuntimeError(
            "Internal split error: sentence-level split does not exactly cover "
            "the eligible training sentences."
        )

    print()
    print("Sentence-level multilabel-stratified train/validation split:")
    print(
        f"  Eligible training sentences : {len(eligible_indices):,}"
    )
    print(
        f"  Target validation sentences : "
        f"{round(len(eligible_indices) * VALIDATION_FRACTION):,}"
    )
    print(
        f"  Actual validation sentences : {len(validation_indices):,}"
    )
    print(
        f"  Training sentences retained : {len(train_indices):,}"
    )
    print(
        f"  Split seed used             : "
        f"{best_candidate['split_seed_used']}"
    )
    print(
        f"  Stratification labels       : "
        f"{len(stratification_labels):,}"
    )
    print(
        f"  Coverage-rejected attempts  : {rejected_for_coverage:,}"
    )
    print(
        f"  Mean relative prevalence error: "
        f"{best_candidate['prevalence_error']:.6f}"
    )

    # Show sentence-level prevalence agreement, which is the quantity actually
    # controlled by the stratifier.
    validation_fraction_actual = (
        len(validation_indices) / len(eligible_indices)
    )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    row_by_sentence = {
        sentence_index: row
        for row, sentence_index in enumerate(eligible_indices)
    }

    train_rows_lookup = [
        row_by_sentence[index]
        for index in train_indices
    ]
    validation_rows_lookup = [
        row_by_sentence[index]
        for index in validation_indices
    ]

    print()
    print(
        f"{'Feature':35s} "
        f"{'All sent.':>10s} "
        f"{'Train':>10s} "
        f"{'Valid':>10s} "
        f"{'Valid %':>10s}"
    )
    print("-" * 80)

    for feature in stratification_labels:
        column = label2column[feature]
        all_support = int(sentence_matrix[:, column].sum())
        train_support = int(
            sentence_matrix[train_rows_lookup, column].sum()
        )
        validation_support = int(
            sentence_matrix[validation_rows_lookup, column].sum()
        )

        print(
            f"{feature:35s} "
            f"{all_support:10d} "
            f"{train_support:10d} "
            f"{validation_support:10d} "
            f"{validation_support / max(all_support, 1):10.3f}"
        )

    print(
        f"\nActual validation sentence fraction: "
        f"{validation_fraction_actual:.4f}"
    )
    print()

    return best_candidate


# =============================================================================
# Dataset
# =============================================================================

class SemanticFeatureDataset(Dataset):
    """
    RoBERTa multi-label token-classification dataset.

    IMPORTANT:
        labels contain only INDEPENDENT neural targets.

    Labels have shape:
        [sequence_length, number_of_prediction_features]

    For supervised tokens:
        0 = independent feature absent
        1 = independent feature present

    For ignored tokens:
        -100 across all labels

    Only the first RoBERTa subword corresponding to each source token receives
    supervision.
    """

    def __init__(
        self,
        sentences: Sequence[Any],
        sentence_indices: Sequence[int],
        tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
        tokenizer: Any,
        label2id: Dict[str, int],
        max_length: int,
    ) -> None:
        self.sentences = sentences
        self.sentence_indices = list(sentence_indices)
        self.tagged_records = tagged_records
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.num_labels = len(label2id)
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.sentence_indices)

    def __getitem__(
        self,
        dataset_index: int,
    ) -> Dict[str, Any]:
        sentence_index = self.sentence_indices[dataset_index]
        sentence = self.sentences[sentence_index]

        words, token_ids = get_sentence_words(sentence)

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        labels: List[List[float]] = []
        previous_word_id: Optional[int] = None
        sentence_records = self.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                labels.append([-100.0] * self.num_labels)
                previous_word_id = word_id
                continue

            # Ignore all but the first subword.
            if word_id == previous_word_id:
                labels.append([-100.0] * self.num_labels)
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                labels.append([-100.0] * self.num_labels)
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                labels.append([-100.0] * self.num_labels)
                continue

            full_features = clean_features(record)

            if not full_features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                labels.append([-100.0] * self.num_labels)
                continue

            independent_features = reduce_to_independent_features(
                full_features
            )

            target = [0.0] * self.num_labels

            for feature in independent_features:
                label_id = self.label2id.get(feature)

                if label_id is not None:
                    target[label_id] = 1.0

            labels.append(target)

        encoding["labels"] = labels
        encoding["sentence_index"] = sentence_index

        return encoding


# =============================================================================
# Data collator
# =============================================================================

class MultiLabelTokenCollator:
    """Dynamically pad inputs and 3-D multi-label token targets."""

    def __init__(
        self,
        tokenizer: Any,
        num_labels: int,
    ) -> None:
        self.tokenizer = tokenizer
        self.num_labels = num_labels

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        clean_features_list: List[Dict[str, Any]] = []
        labels_list: List[List[List[float]]] = []

        for item in features:
            item = dict(item)
            item.pop("sentence_index", None)

            labels = item.pop("labels")
            labels_list.append(labels)
            clean_features_list.append(item)

        batch = self.tokenizer.pad(
            clean_features_list,
            padding=True,
            return_tensors="pt",
        )

        batch_size = len(labels_list)
        max_seq_length = batch["input_ids"].shape[1]

        padded_labels = torch.full(
            (
                batch_size,
                max_seq_length,
                self.num_labels,
            ),
            -100.0,
            dtype=torch.float32,
        )

        for batch_index, labels in enumerate(labels_list):
            label_tensor = torch.tensor(
                labels,
                dtype=torch.float32,
            )

            sequence_length = label_tensor.shape[0]

            padded_labels[
                batch_index,
                :sequence_length,
                :,
            ] = label_tensor

        batch["labels"] = padded_labels

        return batch


# =============================================================================
# Model
# =============================================================================

class RobertaMultiLabelModel(nn.Module):
    """RoBERTa token classifier with BCE loss for independent feature labels."""

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        id2label: Dict[int, str],
        label2id: Dict[str, int],
    ) -> None:
        super().__init__()

        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )

        self.config = self.model.config

        self.loss_function = nn.BCEWithLogitsLoss(
            reduction="mean"
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        logits = outputs.logits
        loss = None

        if labels is not None:
            valid_mask = labels[:, :, 0] != -100.0

            valid_logits = logits[valid_mask]
            valid_labels = labels[valid_mask]

            if valid_logits.numel() == 0:
                loss = logits.sum() * 0.0
            else:
                loss = self.loss_function(
                    valid_logits,
                    valid_labels,
                )

        return {
            "loss": loss,
            "logits": logits,
        }

    def save_pretrained(
        self,
        save_directory,
        **kwargs,
    ):
        return self.model.save_pretrained(
            save_directory,
            **kwargs,
        )


# =============================================================================
# Hierarchy matrices and array helpers
# =============================================================================

def build_hierarchy_expansion_matrix(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
) -> np.ndarray:
    """
    Build a boolean matrix mapping neural outputs to complete feature sets.

    Shape:
        [number_of_prediction_features, number_of_full_features]
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    matrix = np.zeros(
        (len(prediction_labels), len(full_labels)),
        dtype=np.int8,
    )

    for prediction_id, feature in enumerate(prediction_labels):
        expanded = expand_feature_hierarchy({feature})

        for expanded_feature in expanded:
            full_id = full_label2id.get(expanded_feature)

            if full_id is None:
                raise RuntimeError(
                    "Hierarchy expansion produced a feature absent from the "
                    f"full vocabulary: {expanded_feature!r}."
                )

            matrix[prediction_id, full_id] = 1

    return matrix


def expand_binary_feature_matrix(
    independent_matrix: np.ndarray,
    hierarchy_matrix: np.ndarray,
) -> np.ndarray:
    """Expand [tokens, independent] binary values to [tokens, complete]."""

    expanded = independent_matrix.astype(np.int32) @ hierarchy_matrix.astype(
        np.int32
    )

    return (expanded > 0).astype(np.int32)


def extract_valid_token_arrays(
    logits: np.ndarray,
    labels: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return probabilities and true independent labels for supervised tokens."""

    valid_token_mask = labels[:, :, 0] != -100.0

    probabilities = sigmoid_numpy(logits)[
        valid_token_mask
    ]

    true_labels = labels[
        valid_token_mask
    ].astype(np.int32)

    return probabilities, true_labels


# =============================================================================
# Metrics and thresholds
# =============================================================================

def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""

    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


def make_threshold_vector(
    prediction_labels: Sequence[str],
    thresholds: Dict[str, float],
) -> np.ndarray:
    """Return thresholds in neural-output column order."""

    return np.asarray(
        [
            float(
                thresholds.get(
                    label,
                    DEFAULT_PREDICTION_THRESHOLD,
                )
            )
            for label in prediction_labels
        ],
        dtype=np.float32,
    )


def threshold_probabilities(
    probabilities: np.ndarray,
    threshold_vector: np.ndarray,
) -> np.ndarray:
    """Apply one threshold per independent feature."""

    return (
        probabilities >= threshold_vector.reshape(1, -1)
    ).astype(np.int32)


def calculate_average_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    prefix: str = "",
) -> Dict[str, float]:
    """Calculate micro and macro precision/recall/F1."""

    if y_true.shape[0] == 0:
        return {
            f"{prefix}precision_micro": 0.0,
            f"{prefix}recall_micro": 0.0,
            f"{prefix}f1_micro": 0.0,
            f"{prefix}precision_macro": 0.0,
            f"{prefix}recall_macro": 0.0,
            f"{prefix}f1_macro": 0.0,
        }

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        f"{prefix}precision_micro": float(precision_micro),
        f"{prefix}recall_micro": float(recall_micro),
        f"{prefix}f1_micro": float(f1_micro),
        f"{prefix}precision_macro": float(precision_macro),
        f"{prefix}recall_macro": float(recall_macro),
        f"{prefix}f1_macro": float(f1_macro),
    }


def calculate_hierarchical_metrics_from_probabilities(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    threshold_vector: np.ndarray,
    hierarchy_matrix: np.ndarray,
    y_true_full: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Report both direct neural-output metrics and complete hierarchy metrics.

    Unprefixed metrics are the COMPLETE hierarchy metrics.  During ordinary
    training/validation, complete truth can be reconstructed from the
    independent targets.  For a genuinely held-out test corpus, ``y_true_full``
    may instead be supplied directly from the source annotations so an unseen
    direct independent label is not silently erased from the gold truth.
    """

    y_pred_independent = threshold_probabilities(
        probabilities,
        threshold_vector,
    )

    if y_true_full is None:
        y_true_full = expand_binary_feature_matrix(
            y_true_independent,
            hierarchy_matrix,
        )

    if y_true_full.shape[0] != y_true_independent.shape[0]:
        raise RuntimeError(
            "Complete and independent truth matrices contain different "
            f"numbers of supervised tokens: {y_true_full.shape[0]} vs "
            f"{y_true_independent.shape[0]}."
        )

    y_pred_full = expand_binary_feature_matrix(
        y_pred_independent,
        hierarchy_matrix,
    )

    metrics = calculate_average_metrics(
        y_true_full,
        y_pred_full,
        prefix="",
    )

    metrics.update(
        calculate_average_metrics(
            y_true_independent,
            y_pred_independent,
            prefix="direct_",
        )
    )

    return metrics


def make_compute_metrics(
    prediction_labels: Sequence[str],
    hierarchy_matrix: np.ndarray,
):
    """Create Trainer metrics using the fixed default 0.5 thresholds."""

    default_thresholds = np.full(
        len(prediction_labels),
        DEFAULT_PREDICTION_THRESHOLD,
        dtype=np.float32,
    )

    def compute_metrics(eval_prediction) -> Dict[str, float]:
        logits, labels = eval_prediction

        probabilities, y_true_independent = extract_valid_token_arrays(
            logits,
            labels,
        )

        return calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=y_true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )

    return compute_metrics


def build_threshold_grid() -> np.ndarray:
    """Build an inclusive, stable validation threshold search grid."""

    if THRESHOLD_STEP <= 0:
        raise ValueError("THRESHOLD_STEP must be > 0.")

    if not 0.0 < THRESHOLD_MIN < 1.0:
        raise ValueError("THRESHOLD_MIN must be between 0 and 1.")

    if not 0.0 < THRESHOLD_MAX < 1.0:
        raise ValueError("THRESHOLD_MAX must be between 0 and 1.")

    if THRESHOLD_MIN > THRESHOLD_MAX:
        raise ValueError("THRESHOLD_MIN cannot exceed THRESHOLD_MAX.")

    count = int(
        round(
            (THRESHOLD_MAX - THRESHOLD_MIN)
            / THRESHOLD_STEP
        )
    ) + 1

    return np.linspace(
        THRESHOLD_MIN,
        THRESHOLD_MAX,
        count,
        dtype=np.float64,
    )


def binary_metrics_for_column(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Tuple[float, float, float, int]:
    """Return binary precision, recall, F1 and positive support."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    support = int(y_true.sum())

    return (
        float(precision),
        float(recall),
        float(f1),
        support,
    )


def optimise_feature_thresholds(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    prediction_labels: Sequence[str],
) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """
    Optimise one sigmoid threshold per independent feature on VALIDATION ONLY.

    Objective:
        maximise that feature's binary F1.

    Stability rules:
    - If validation has no positives or no negatives for a feature, keep 0.5.
    - If several thresholds tie for best F1, choose the one closest to 0.5.
      This avoids unnecessarily extreme thresholds for flat validation curves.
    """

    grid = build_threshold_grid()

    thresholds: Dict[str, float] = {}
    diagnostics: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(prediction_labels):
        y_true = y_true_independent[:, label_id]
        probs = probabilities[:, label_id]

        positive_support = int(y_true.sum())
        negative_support = int((1 - y_true).sum())

        default_pred = (
            probs >= DEFAULT_PREDICTION_THRESHOLD
        ).astype(np.int32)

        (
            default_precision,
            default_recall,
            default_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            default_pred,
        )

        if positive_support == 0 or negative_support == 0:
            best_threshold = DEFAULT_PREDICTION_THRESHOLD
        else:
            best_f1 = -1.0
            best_candidates: List[float] = []

            for threshold in grid:
                y_pred = (probs >= threshold).astype(np.int32)

                _, _, f1, _ = binary_metrics_for_column(
                    y_true,
                    y_pred,
                )

                if f1 > best_f1 + 1e-12:
                    best_f1 = f1
                    best_candidates = [float(threshold)]
                elif abs(f1 - best_f1) <= 1e-12:
                    best_candidates.append(float(threshold))

            best_threshold = min(
                best_candidates,
                key=lambda value: (
                    abs(value - DEFAULT_PREDICTION_THRESHOLD),
                    value,
                ),
            )

        best_pred = (
            probs >= best_threshold
        ).astype(np.int32)

        (
            best_precision,
            best_recall,
            best_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            best_pred,
        )

        thresholds[feature] = float(best_threshold)

        diagnostics[feature] = {
            "threshold": float(best_threshold),
            "support": positive_support,
            "negative_support": negative_support,
            "precision_default": default_precision,
            "recall_default": default_recall,
            "f1_default": default_f1,
            "precision_optimised": best_precision,
            "recall_optimised": best_recall,
            "f1_optimised": best_f1,
            "f1_change": best_f1 - default_f1,
        }

    return thresholds, diagnostics


# =============================================================================
# Dataset diagnostics and validation
# =============================================================================

def count_feature_occurrences(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
    independent_only: bool = False,
) -> Counter:
    """Count complete or independent positive labels in a split."""

    counts: Counter = Counter()

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if independent_only:
                features = get_independent_record_features(record)
            else:
                features = clean_features(record)

            for feature in features:
                counts[feature] += 1

    return counts


def count_empty_records(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
) -> int:
    """Count NOUN/PROPN records having no complete feature tags."""

    count = 0

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if not clean_features(record):
                count += 1

    return count


def verify_full_label_coverage(
    full_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """Ensure validation contains no complete label unseen in training."""

    training_labels = set(full_labels)

    for split_name in ("validation",):
        tagged_records, sentence_indices = split_sources[split_name]
        observed: Set[str] = set()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                observed.update(clean_features(record))

        unknown = observed - training_labels

        if unknown:
            raise RuntimeError(
                f"{split_name} contains full labels unseen in training: "
                f"{sorted(unknown)}"
            )


def verify_prediction_label_coverage(
    prediction_labels: Sequence[str],
    split_sources: Dict[
        str, Tuple[Dict[int, Dict[int, Dict[str, Any]]], Sequence[int]]
    ],
) -> None:
    """Require every validation independent target to occur in training."""

    training_labels = set(prediction_labels)
    records, indices = split_sources["validation"]
    observed = {
        feature
        for sentence_index in indices
        for record in records.get(sentence_index, {}).values()
        for feature in get_independent_record_features(record)
    }
    unknown = observed - training_labels
    if unknown:
        raise RuntimeError(
            "Validation contains independent targets without a training "
            f"output head: {sorted(unknown)}"
        )


def verify_hierarchy_closure(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """
    Ensure source feature sets already contain all ancestors required by the
    hierarchy. This prevents evaluation against inconsistent source targets.
    """

    problems: List[str] = []
    max_examples = 20

    for split_name in ("train", "validation"):
        tagged_records, sentence_indices = split_sources[split_name]
        problem_count = 0

        for sentence_index in sentence_indices:
            for token_id, record in tagged_records.get(
                sentence_index,
                {},
            ).items():
                features = clean_features(record)

                if not features:
                    continue

                missing = missing_hierarchy_features(features)

                if not missing:
                    continue

                problem_count += 1

                if len(problems) < max_examples:
                    problems.append(
                        f"{split_name}: sentence={sentence_index}, "
                        f"token={token_id}, "
                        f"form={record.get('form')!r}, "
                        f"missing={sorted(missing)}"
                    )

        if problem_count:
            print(
                f"Hierarchy closure problems in {split_name}: "
                f"{problem_count:,} token(s)"
            )

    if problems:
        example_text = "\n".join(
            f"  - {problem}"
            for problem in problems
        )

        raise RuntimeError(
            "The input dataset is not hierarchy-closed. Fix the source tags "
            "before training so deterministic reduction/expansion is exact.\n"
            f"Examples:\n{example_text}"
        )


def print_split_statistics(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    print("=" * 80)
    print("DATASET SPLIT")
    print("=" * 80)

    for split_name in ("train", "validation"):
        tagged_records, indices = split_sources[split_name]

        full_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=False,
        )

        independent_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=True,
        )

        empty_count = count_empty_records(
            tagged_records,
            indices,
        )

        noun_count = sum(
            len(tagged_records.get(index, {}))
            for index in indices
        )

        source_name = (
            "training file"
            if split_name in {"train", "validation"}
            else "test file"
        )

        print(
            f"{split_name:12s}: "
            f"{len(indices):,} sentences | "
            f"{noun_count:,} NOUN/PROPN tokens | "
            f"{sum(full_counts.values()):,} complete positives -> "
            f"{sum(independent_counts.values()):,} independent positives | "
            f"{empty_count:,} empty-feature tokens excluded | "
            f"source={source_name}"
        )

    print()


def print_prediction_label_statistics(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    train_sentence_indices: Sequence[int],
) -> None:
    """Show which labels are neural outputs versus deterministic-only parents."""

    train_independent_counts = count_feature_occurrences(
        train_tagged_records,
        train_sentence_indices,
        independent_only=True,
    )

    prediction_set = set(prediction_labels)
    deterministic_only = [
        feature
        for feature in full_labels
        if feature not in prediction_set
    ]

    print("=" * 80)
    print("NEURAL OUTPUT FEATURES")
    print("=" * 80)
    print(
        f"Complete evaluation features: {len(full_labels)}"
    )
    print(
        f"Independent neural outputs: {len(prediction_labels)}"
    )
    print(
        f"Deterministic-only features in this training split: "
        f"{len(deterministic_only)}"
    )
    print()

    print(
        f"{'Feature':35s} "
        f"{'Independent train support':>26s}"
    )
    print("-" * 64)

    for feature in prediction_labels:
        print(
            f"{feature:35s} "
            f"{train_independent_counts[feature]:26d}"
        )

    if deterministic_only:
        print()
        print("Deterministic-only in this split:")

        for feature in deterministic_only:
            print(f"  - {feature}")

    print()


def make_json_serialisable(value: Any) -> Any:
    """Recursively convert NumPy scalars/arrays to ordinary JSON values."""

    if isinstance(value, dict):
        return {
            str(key): make_json_serialisable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_serialisable(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (np.floating, np.integer)):
        return value.item()

    return value


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    validate_feature_hierarchy()
    check_cuda()

    if not LEARNING_RATES or not PER_DEVICE_TRAIN_BATCH_SIZES or not EFFECTIVE_BATCH_SIZES:
        raise ValueError("All hyperparameter grid lists must be non-empty.")
    if any(lr <= 0 for lr in LEARNING_RATES):
        raise ValueError("Learning rates must be positive.")

    n_devices = torch.cuda.device_count()
    if n_devices < 1:
        raise RuntimeError("At least one CUDA device is required.")

    grid = []
    for lr, device_batch, effective_batch in product(
        LEARNING_RATES, PER_DEVICE_TRAIN_BATCH_SIZES, EFFECTIVE_BATCH_SIZES
    ):
        denominator = device_batch * n_devices
        if device_batch <= 0 or effective_batch <= 0 or effective_batch % denominator:
            raise ValueError(
                "Each effective batch size must be a positive multiple of "
                "per-device batch size times the number of CUDA devices: "
                f"device_batch={device_batch}, effective_batch={effective_batch}, "
                f"CUDA devices={n_devices}."
            )
        grid.append((lr, device_batch, effective_batch,
                     effective_batch // denominator))
    if len(grid) != len(set(grid)):
        raise ValueError("The hyperparameter grid contains duplicate configurations.")

    print("=" * 80)
    print("VALIDATION-ONLY HYPERPARAMETER GRID SEARCH")
    print("=" * 80)
    print(f"Configurations: {len(grid)} | fine-tuning seed: {TUNING_SEED}")
    print(f"CUDA devices: {n_devices}")
    print("Selection metric: hierarchy-expanded validation macro F1 at threshold 0.5")
    print("The official test corpus is not read during this search.\n")

    train_sentences = load_conllu(TRAIN_CONLLU_PATH)
    train_tagged_records = load_tagged_records(TRAIN_JSONL_PATH)
    validate_feature_switches(train_tagged_records)

    eligible_indices = get_eligible_sentence_indices(
        train_sentences, train_tagged_records
    )
    split = make_train_validation_split(eligible_indices, train_tagged_records)
    train_indices = split["train"]
    validation_indices = split["validation"]
    sources = {
        "train": (train_tagged_records, train_indices),
        "validation": (train_tagged_records, validation_indices),
    }
    split_document = {
        "split_unit": "sentence",
        "stratification": "iterative_multilabel_sentence_presence",
        "split_seed_requested": SPLIT_SEED,
        "split_seed_used": split["split_seed_used"],
        "split_attempt": split["attempt"],
        "validation_fraction": VALIDATION_FRACTION,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "prevalence_error": split["prevalence_error"],
        "stratification_labels": split["stratification_labels"],
        "train_jsonl": str(TRAIN_JSONL_PATH),
        "train_conllu": str(TRAIN_CONLLU_PATH),
        "index_spaces": {"train": "training corpus", "validation": "training corpus"},
        "train": train_indices,
        "validation": validation_indices,
    }
    with SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(make_json_serialisable(split_document), handle,
                  ensure_ascii=False, indent=2)

    verify_hierarchy_closure(sources)
    full_labels = build_full_label_vocabulary(train_tagged_records, train_indices)
    prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records, train_indices
    )
    verify_full_label_coverage(full_labels, sources)
    verify_prediction_label_coverage(prediction_labels, sources)

    label2id = {label: index for index, label in enumerate(prediction_labels)}
    id2label = {index: label for label, index in label2id.items()}
    hierarchy_matrix = build_hierarchy_expansion_matrix(
        prediction_labels, full_labels
    )
    label_document = {
        "labels": prediction_labels,
        "label2id": label2id,
        "id2label": {str(key): value for key, value in id2label.items()},
        "prediction_labels": prediction_labels,
        "prediction_label2id": label2id,
        "full_labels": full_labels,
        "full_label2id": {label: index for index, label in enumerate(full_labels)},
        "feature_parents": {
            child: sorted(parents) for child, parents in sorted(FEATURE_PARENTS.items())
        },
        "target_representation": "per_token_independent_features",
        "split_unit": "sentence",
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature for feature, enabled in FEATURE_SWITCHES.items() if not enabled
        ),
    }
    with LABELS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(label_document, handle, ensure_ascii=False, indent=2)

    print_split_statistics(sources)
    print_prediction_label_statistics(
        prediction_labels, full_labels, train_tagged_records, train_indices
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, use_fast=True, add_prefix_space=True
    )
    train_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=train_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=label2id,
        max_length=MAX_LENGTH,
    )
    validation_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=validation_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=label2id,
        max_length=MAX_LENGTH,
    )
    collator = MultiLabelTokenCollator(
        tokenizer=tokenizer, num_labels=len(prediction_labels)
    )
    use_bf16 = torch.cuda.is_bf16_supported()
    results = []
    work_root = OUTPUT_DIR / "_grid_trainer_work"
    work_root.mkdir(parents=True, exist_ok=True)

    def save_results() -> None:
        # A partial result survives a later interrupted or out-of-memory trial.
        ranked = sorted(
            results,
            key=lambda row: (-row["validation_f1_macro_at_0_5"],
                             -row["validation_f1_micro_at_0_5"], row["trial"]),
        )
        document = {
            "model": MODEL_NAME,
            "selection_data": "validation_only",
            "official_test_evaluated": False,
            "selection_metric": "hierarchy_expanded_validation_f1_macro_at_0_5",
            "tie_breaker": "validation_f1_micro_at_0_5_then_grid_order",
            "training_seed": TUNING_SEED,
            "split_seed": split["split_seed_used"],
            "epochs": NUM_EPOCHS,
            "cuda_devices": n_devices,
            "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
            "grid_size": len(grid),
            "completed_trials": len(results),
            "best": ranked[0] if ranked else None,
            "results_ranked": ranked,
        }
        with GRID_RESULTS_PATH.open("w", encoding="utf-8") as handle:
            json.dump(make_json_serialisable(document), handle,
                      ensure_ascii=False, indent=2)
        columns = [
            "rank", "trial", "learning_rate", "per_device_train_batch_size",
            "gradient_accumulation_steps", "effective_batch_size",
            "training_seed", "best_epoch",
            "best_epoch_validation_f1_macro_at_0_5",
            "validation_f1_macro_at_0_5", "validation_f1_micro_at_0_5",
            "validation_f1_macro_calibrated_diagnostic",
            "validation_f1_micro_calibrated_diagnostic",
        ]
        with GRID_CSV_PATH.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns)
            writer.writeheader()
            for rank, row in enumerate(ranked, start=1):
                writer.writerow({name: rank if name == "rank" else row.get(name)
                                 for name in columns})

    for trial, (lr, device_batch, effective_batch, accumulation) in enumerate(
        grid, start=1
    ):
        print("\n" + "=" * 80)
        print(f"TRIAL {trial}/{len(grid)}: LR={lr:g}, device batch={device_batch}, "
              f"accumulation={accumulation}, effective batch={effective_batch}")
        print("=" * 80)
        set_seed(TUNING_SEED)
        work_dir = work_root / f"trial_{trial:02d}"
        if work_dir.exists():
            shutil.rmtree(work_dir)

        model = RobertaMultiLabelModel(
            model_name=MODEL_NAME,
            num_labels=len(prediction_labels),
            id2label=id2label,
            label2id=label2id,
        )
        args = TrainingArguments(
            output_dir=str(work_dir),
            learning_rate=lr,
            lr_scheduler_type=LEARNING_RATE_SCHEDULER,
            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=device_batch,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            gradient_accumulation_steps=accumulation,
            num_train_epochs=NUM_EPOCHS,
            eval_strategy=EVAL_STRATEGY,
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            save_total_limit=2,
            bf16=use_bf16,
            fp16=not use_bf16,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            optim="adamw_torch",
            report_to="none",
            seed=TUNING_SEED,
            data_seed=TUNING_SEED,
            remove_unused_columns=False,
            eval_accumulation_steps=16,
            warmup_steps = WARMUP_STEPS,
        )
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            processing_class=tokenizer,
            data_collator=collator,
            compute_metrics=make_compute_metrics(
                prediction_labels, hierarchy_matrix
            ),
        )
        trainer.train()
        validation_output = trainer.predict(
            validation_dataset, metric_key_prefix="validation"
        )
        probabilities, true_independent = extract_valid_token_arrays(
            validation_output.predictions, validation_output.label_ids
        )
        default_thresholds = np.full(
            len(prediction_labels), DEFAULT_PREDICTION_THRESHOLD,
            dtype=np.float32,
        )
        default_metrics = calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )
        calibrated_thresholds, _ = optimise_feature_thresholds(
            probabilities=probabilities,
            y_true_independent=true_independent,
            prediction_labels=prediction_labels,
        )
        calibrated_metrics = calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=true_independent,
            threshold_vector=make_threshold_vector(
                prediction_labels, calibrated_thresholds
            ),
            hierarchy_matrix=hierarchy_matrix,
        )
        best_metric = trainer.state.best_metric
        best_checkpoint = trainer.state.best_model_checkpoint
        best_epoch = None
        if best_checkpoint:
            for entry in trainer.state.log_history:
                if "eval_f1_macro" in entry and "step" in entry:
                    # The best checkpoint's step is encoded in its directory.
                    if best_checkpoint.endswith(f"checkpoint-{entry['step']}"):
                        best_epoch = entry.get("epoch")
                        break

        row = {
            "trial": trial,
            "learning_rate": lr,
            "per_device_train_batch_size": device_batch,
            "gradient_accumulation_steps": accumulation,
            "effective_batch_size": effective_batch,
            "training_seed": TUNING_SEED,
            "best_epoch": best_epoch,
            "best_epoch_validation_f1_macro_at_0_5": best_metric,
            "validation_f1_macro_at_0_5": default_metrics["f1_macro"],
            "validation_f1_micro_at_0_5": default_metrics["f1_micro"],
            "validation_f1_macro_calibrated_diagnostic": calibrated_metrics["f1_macro"],
            "validation_f1_micro_calibrated_diagnostic": calibrated_metrics["f1_micro"],
            "validation_metrics_at_0_5": default_metrics,
            "validation_metrics_calibrated_diagnostic": calibrated_metrics,
            "calibrated_thresholds_diagnostic": calibrated_thresholds,
        }
        results.append(row)
        save_results()
        print(f"Validation macro F1 @ 0.5: {row['validation_f1_macro_at_0_5']:.4f}")
        print(f"Validation micro F1 @ 0.5: {row['validation_f1_micro_at_0_5']:.4f}")
        print(f"Results saved: {GRID_RESULTS_PATH.resolve()}")

        del validation_output, trainer, model
        gc.collect()
        torch.cuda.empty_cache()
        if work_dir.exists():
            shutil.rmtree(work_dir)

    if work_root.exists() and not any(work_root.iterdir()):
        work_root.rmdir()

    print("\n" + "=" * 80)
    print("GRID SEARCH RANKING (VALIDATION ONLY)")
    print("=" * 80)
    ranked = sorted(
        results,
        key=lambda row: (-row["validation_f1_macro_at_0_5"],
                         -row["validation_f1_micro_at_0_5"], row["trial"]),
    )
    print(f"{'Rank':>4} {'LR':>10} {'Device':>7} {'Accum':>7} "
          f"{'Effective':>9} {'Macro F1':>10} {'Micro F1':>10}")
    for rank, row in enumerate(ranked, start=1):
        print(f"{rank:4d} {row['learning_rate']:10.1e} "
              f"{row['per_device_train_batch_size']:7d} "
              f"{row['gradient_accumulation_steps']:7d} "
              f"{row['effective_batch_size']:9d} "
              f"{row['validation_f1_macro_at_0_5']:10.4f} "
              f"{row['validation_f1_micro_at_0_5']:10.4f}")
    print(f"\nBest candidate: LR={ranked[0]['learning_rate']:g}, "
          f"device batch={ranked[0]['per_device_train_batch_size']}, "
          f"accumulation={ranked[0]['gradient_accumulation_steps']}, "
          f"effective batch={ranked[0]['effective_batch_size']}")
    print("Use these settings for the separate 30-seed experiment; "
          "this single-seed ranking is provisional.")
    print(f"JSON: {GRID_RESULTS_PATH.resolve()}")
    print(f"CSV:  {GRID_CSV_PATH.resolve()}")


if __name__ == "__main__":
    main()

CUDA
PyTorch version: 2.13.0+cu130
CUDA available: True
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.92 GiB

VALIDATION-ONLY HYPERPARAMETER GRID SEARCH
Configurations: 16 | fine-tuning seed: 11
CUDA devices: 1
Selection metric: hierarchy-expanded validation macro F1 at threshold 0.5
The official test corpus is not read during this search.

FEATURE SWITCHES
Disabled features: Rigidity, Shape, State, State_of_Matter
All unlisted features are enabled.


Sentence-level multilabel-stratified train/validation split:
  Eligible training sentences : 11,198
  Target validation sentences : 2,240
  Actual validation sentences : 2,313
  Training sentences retained : 8,885
  Split seed used             : 42
  Stratification labels       : 37
  Coverage-rejected attempts  : 0
  Mean relative prevalence error: 0.008441

Feature                              All sent.      Train      Valid    Valid %
--------------------------------------------------------------------------------
Abstrac

Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 975.97it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.259344,0.054808,0.914266,0.834047,0.872316,0.699083,0.611796,0.644420,0.841953,0.745421,0.790752,0.567072,0.487738,0.516667
2,0.166275,0.039637,0.925143,0.898638,0.911698,0.853864,0.766282,0.793328,0.860092,0.839487,0.849664,0.756165,0.664313,0.691845
3,0.124622,0.034279,0.932804,0.909509,0.921010,0.865050,0.817232,0.833398,0.881242,0.854579,0.867706,0.798751,0.734385,0.755643
4,0.094588,0.033520,0.934046,0.918814,0.926368,0.899624,0.842797,0.862017,0.880841,0.865934,0.873324,0.857841,0.761133,0.786391
5,0.084652,0.033058,0.936273,0.919838,0.927983,0.900747,0.849453,0.868285,0.881584,0.869377,0.875438,0.858597,0.771390,0.796079


Validation macro F1 @ 0.5: 0.8683
Validation micro F1 @ 0.5: 0.9280
Results saved: /home/mark/verbnet-roberta/models/roberta_semantic_features_grid_search/hyperparameter_grid_results.json

TRIAL 2/16: LR=1.5e-05, device batch=8, accumulation=2, effective batch=16


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 911.55it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.221948,0.045099,0.913794,0.872098,0.892459,0.829630,0.759623,0.788649,0.847995,0.807179,0.827084,0.732865,0.654398,0.685343
2,0.141138,0.034141,0.935008,0.901164,0.917774,0.888031,0.829732,0.853643,0.874878,0.856484,0.865583,0.844685,0.757361,0.785524
3,0.098445,0.033900,0.935465,0.919582,0.927455,0.883221,0.877152,0.878718,0.884776,0.868571,0.876599,0.854771,0.812133,0.822214
4,0.059268,0.032552,0.937932,0.927735,0.932806,0.896582,0.879314,0.885804,0.886673,0.879267,0.882954,0.865585,0.818140,0.834126
5,0.039268,0.032909,0.941446,0.931573,0.936483,0.906518,0.889080,0.897299,0.892984,0.885788,0.889371,0.876349,0.831867,0.847928


Validation macro F1 @ 0.5: 0.8973
Validation micro F1 @ 0.5: 0.9365
Results saved: /home/mark/verbnet-roberta/models/roberta_semantic_features_grid_search/hyperparameter_grid_results.json

TRIAL 16/16: LR=5e-05, device batch=8, accumulation=2, effective batch=16


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 975.58it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.102139,0.043602,0.917528,0.874752,0.895629,0.838168,0.752817,0.786493,0.855645,0.807253,0.830745,0.742429,0.646002,0.682596
2,0.066364,0.034215,0.939963,0.888598,0.913559,0.914673,0.815997,0.855492,0.886792,0.843590,0.864652,0.876821,0.745645,0.790301
3,0.044391,0.033334,0.939620,0.910597,0.924881,0.896137,0.866078,0.878262,0.893094,0.859267,0.875854,0.875521,0.801737,0.821630
4,0.027031,0.030614,0.940368,0.925785,0.933020,0.908576,0.888660,0.896817,0.887506,0.882564,0.885028,0.874925,0.833991,0.844677
5,0.020720,0.031152,0.942406,0.929750,0.936035,0.907895,0.886720,0.896798,0.893518,0.884615,0.889044,0.872577,0.829953,0.844099


Validation macro F1 @ 0.5: 0.8968
Validation micro F1 @ 0.5: 0.9330
Results saved: /home/mark/verbnet-roberta/models/roberta_semantic_features_grid_search/hyperparameter_grid_results.json

GRID SEARCH RANKING (VALIDATION ONLY)
Rank         LR  Device   Accum Effective   Macro F1   Micro F1
   1    4.0e-05       4       4        16     0.9048     0.9381
   2    4.5e-05       4       4        16     0.9032     0.9378
   3    3.5e-05       4       4        16     0.9020     0.9383
   4    4.0e-05       8       2        16     0.9015     0.9388
   5    4.5e-05       8       2        16     0.9012     0.9352
   6    5.0e-05       4       4        16     0.8973     0.9365
   7    5.0e-05       8       2        16     0.8968     0.9330
   8    3.0e-05       4       4        16     0.8960     0.9355
   9    3.5e-05       8       2        16     0.8954     0.9375
  10    3.0e-05       8       2        16     0.8948     0.9371
  11    2.5e-05       8       2        16     0.8894     0.9357
  12 

<h1>Inference</h1>

In [48]:
#!/usr/bin/env python3
"""
Inference script for the fine-tuned RoBERTa semantic-feature tagger.

Compatible with Python 3.9.21.

The script:

1. Loads the saved RoBERTa model.
2. Loads the independent prediction labels from semantic_features.json.
3. Loads the validation-calibrated per-feature thresholds from
   prediction_thresholds.json.
4. Uses spaCy to identify NOUN and PROPN tokens in arbitrary text.
5. Predicts independent semantic features for those tokens.
6. Reconstructs inherited semantic features deterministically using the
   saved feature hierarchy.
7. Prints the tagged sentences.
8. Writes one JSONL record per tagged NOUN/PROPN token.

No input() call is used. Put the text to tag directly in SENTENCES.

Requirements:

    python -m pip install torch transformers spacy safetensors

Install the spaCy English model if necessary:

    python -m spacy download en_core_web_sm
"""

from __future__ import annotations

import json
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Set

import spacy
import torch

from safetensors.torch import load_file as load_safetensors_file
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
)


# =============================================================================
# Configuration
# =============================================================================

# The training script saves the first predeclared run here after Trainer has
# restored that run's best validation checkpoint.
MODEL_DIR = Path(
    "../models/roberta_semantic_features/reference_model"
)

# Needed to reconstruct the architecture when Trainer saved the custom wrapper's
# state_dict rather than a directly reloadable Hugging Face model.
BASE_MODEL_NAME = "FacebookAI/roberta-large"

LABELS_PATH = MODEL_DIR / "semantic_features.json"
THRESHOLDS_PATH = MODEL_DIR / "prediction_thresholds.json"

SPACY_MODEL = "en_core_web_sm"

MAX_LENGTH = 256

TARGET_POS = {
    "NOUN",
    "PROPN",
    "PRON",
}

OUTPUT_JSONL = Path("tagged_sentences.jsonl")

# Include the neural-output probabilities in the JSONL output.
INCLUDE_PROBABILITIES = True


# =============================================================================
# Text to tag
# =============================================================================

SENTENCES = """
The old dog slept beside the fireplace.
Microsoft opened a new office in London last year.
The committee discussed the proposal for three hours.
A sharp knife was lying on the wooden table.
The water flowed through the valley.
I poured the water down the drain.
John coiled the wire around the pole.
The snake wound itself around the tree branch.
Steam erupted from the volcanic vents.
I need to buy some orange juice at the supermarket tomorrow.
Pour some olive oil into a small pan before frying the fish.
I am looking forward to attending the party tonight.
Our trip to New York was a lot of fun!
London and Paris have signed a new agreement for closer economic relations.
I read the old diary with difficulty. The ink was smudged in places, making the handwriting hard to decypher.
Harriet prodded the bug with the tip of a rolled-up newspaper.
I know that the Earth orbits the Sun.
John ate the pasta.
John ate the milk.
The ATM ate my bank card.
The cheese smells.
The skunk smells.
The skunk smells the cheese.
The book is in the library.
John gave Mary a present.
John gave a chicken to Mary.
Mary unwrapped her present.
"""

# SENTENCES = """
# John ate the apple.
# John ate.
# John ate from the bowl.
# John ate off the plate.
# John ate out of the box.
# John ate at the cake.
# They ate of the fruit.
# John ate the meat off the bone.
# She ate the berries from the bush.
# He ate the cereal out of the box.
# The goats ate the field bare.
# The children ate the cupboard bare.
# They ate the plate clean.
# John ate himself sick.
# John ate himself full.
# John ate himself into a food coma.
# John ate himself to death.
# The children ate us out of food.
# The guests ate us out of house and home.
# The goats ate the hedge down to stubs.
# He ate the apple down to the core.
# She ate her way through the buffet.
# The caterpillar ate its way through the leaf.
# The caterpillar ate a hole through the leaf.
# The moths ate holes in the jumper.
# This bread eats easily.
# """

# SENTENCES = """
# Mary ate the sandwich.
# The child ate the banana.
# David ate the bread.
# The dog ate the biscuit.
# Sarah ate the carrot.
# The horse ate the apple.
# The students ate the pizza.
# The rabbit ate the lettuce.
# The farmer ate the potato.
# The bear ate the fish.
# The cat ate the chicken.
# The elephant ate the leaves.
# The mouse ate the cheese.
# The woman ate the rice.
# The boy ate the orange.
# The cow ate the grass.
# The monkey ate the mango.
# The wolf ate the meat.
# The girl ate the chocolate.
# The deer ate the berries.
# The chef ate the pasta.
# The goat ate the cabbage.
# The fox ate the egg.
# The duck ate the corn.
# The tourist ate the noodles.
# The squirrel ate the seeds.
# The pig ate the food.
# The lion ate the steak.
# The worker ate the cake.
# The chimpanzee ate the fruit.
# """

# SENTENCES = """
# Susan talked with Rachel.
# David talked with Michael.
# Sarah talked with Emma.
# James talked with Robert.
# Maria talked with Anna.
# Peter talked with Thomas.
# Linda talked with Helen.
# Daniel talked with Sophie.
# Alice talked with George.
# Matthew talked with Laura.
# The teacher talked with the student.
# The doctor talked with the patient.
# The mother talked with the child.
# The father talked with the daughter.
# The boy talked with the girl.
# The woman talked with the man.
# The nurse talked with the surgeon.
# The farmer talked with the neighbour.
# The lawyer talked with the client.
# The manager talked with the employee.
# The professor talked with the researcher.
# The journalist talked with the politician.
# The customer talked with the cashier.
# The passenger talked with the driver.
# The officer talked with the witness.
# The chef talked with the waiter.
# The tourist talked with the guide.
# The athlete talked with the coach.
# The singer talked with the musician.
# The actor talked with the director.
# The scientist talked with the engineer.
# The librarian talked with the reader.
# The student talked with the lecturer.
# The worker talked with the supervisor.
# The shopper talked with the assistant.
# The mechanic talked with the owner.
# The neighbour talked with the visitor.
# The parent talked with the teenager.
# The grandmother talked with the grandson.
# The sister talked with the brother.
# The bride talked with the groom.
# The captain talked with the sailor.
# The judge talked with the lawyer.
# The dentist talked with the receptionist.
# The baker talked with the customer.
# The pilot talked with the passenger.
# The doctor talked with the nurse.
# The child talked with the teacher.
# The employee talked with the manager.
# The researcher talked with the professor.
# """

# SENTENCES = """
# Allison poked the cloth with the needle.
# David poked the balloon with the pin.
# Sarah poked the soil with the stick.
# James poked the cake with the fork.
# Maria poked the cushion with the pencil.
# Peter poked the sand with the rod.
# Linda poked the fabric with the skewer.
# Daniel poked the dough with the spoon.
# Alice poked the sponge with the probe.
# Matthew poked the paper with the pen.
# The child poked the puddle with the stick.
# The doctor poked the skin with the needle.
# The baker poked the pastry with the fork.
# The gardener poked the earth with the trowel.
# The student poked the clay with the pencil.
# The farmer poked the hay with the pole.
# The chef poked the potato with the skewer.
# The tailor poked the cloth with the pin.
# The scientist poked the sample with the probe.
# The mechanic poked the rubber with the screwdriver.
# The boy poked the mud with the stick.
# The girl poked the balloon with the pencil.
# The woman poked the cushion with the needle.
# The man poked the mattress with the rod.
# The nurse poked the arm with the needle.
# The artist poked the canvas with the stylus.
# The potter poked the clay with the tool.
# The cook poked the meat with the fork.
# The camper poked the fire with the stick.
# The fisherman poked the bait with the hook.
# The worker poked the foam with the rod.
# The researcher poked the tissue with the probe.
# The tailor poked the leather with the awl.
# The child poked the jelly with the spoon.
# The baker poked the bread with the skewer.
# The gardener poked the compost with the fork.
# The teacher poked the model with the pointer.
# The sculptor poked the wax with the stylus.
# The technician poked the membrane with the probe.
# The farmer poked the ground with the pole.
# The chef poked the sausage with the fork.
# The camper poked the embers with the stick.
# The student poked the cork with the pencil.
# The mechanic poked the gasket with the screwdriver.
# The scientist poked the gel with the probe.
# The tailor poked the felt with the needle.
# The cook poked the dumpling with the skewer.
# The gardener poked the moss with the stick.
# The nurse poked the finger with the lancet.
# The artist poked the plaster with the stylus.
# """

# =============================================================================
# General helpers
# =============================================================================

def load_json(path: Path) -> Dict[str, Any]:
    """Load a JSON document."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find required file:\n{path.resolve()}"
        )

    with path.open("r", encoding="utf-8") as handle:
        document = json.load(handle)

    if not isinstance(document, dict):
        raise ValueError(
            f"Expected a JSON object in:\n{path.resolve()}"
        )

    return document


# =============================================================================
# Load saved label information
# =============================================================================

def load_label_information() -> Dict[str, Any]:
    """
    Load the neural-output vocabulary and hierarchy metadata.

    semantic_features.json contains separate vocabularies for:

        prediction_labels
            Independent features directly predicted by the neural model.

        full_labels
            Complete features used after deterministic hierarchy expansion.
    """

    document = load_json(LABELS_PATH)

    prediction_labels = document.get("prediction_labels")

    # Backwards-compatible fallback.
    if prediction_labels is None:
        prediction_labels = document.get("labels")

    if not isinstance(prediction_labels, list):
        raise ValueError(
            "semantic_features.json does not contain prediction_labels."
        )

    prediction_labels = [
        str(feature)
        for feature in prediction_labels
    ]

    full_labels = document.get("full_labels")

    if not isinstance(full_labels, list):
        raise ValueError(
            "semantic_features.json does not contain full_labels."
        )

    full_labels = [
        str(feature)
        for feature in full_labels
    ]

    raw_feature_parents = document.get("feature_parents")

    if not isinstance(raw_feature_parents, dict):
        raise ValueError(
            "semantic_features.json does not contain feature_parents."
        )

    feature_parents: Dict[str, Set[str]] = {}

    for child, parents in raw_feature_parents.items():

        if not isinstance(parents, list):
            raise ValueError(
                "Invalid feature hierarchy entry for "
                f"{child!r}: expected a list."
            )

        feature_parents[str(child)] = {
            str(parent)
            for parent in parents
        }

    raw_switches = document.get(
        "feature_switches",
        {},
    )

    if not isinstance(raw_switches, dict):
        raise ValueError(
            "feature_switches must be a JSON object."
        )

    feature_switches = {
        str(feature): bool(enabled)
        for feature, enabled in raw_switches.items()
    }

    return {
        "document": document,
        "prediction_labels": prediction_labels,
        "full_labels": full_labels,
        "feature_parents": feature_parents,
        "feature_switches": feature_switches,
    }


# =============================================================================
# Thresholds
# =============================================================================

def load_prediction_thresholds(
    prediction_labels: List[str],
) -> Dict[str, float]:
    """
    Load validation-calibrated thresholds.

    Every neural output should have its own threshold.
    """

    document = load_json(THRESHOLDS_PATH)

    raw_thresholds = document.get("thresholds")

    if not isinstance(raw_thresholds, dict):
        raise ValueError(
            "prediction_thresholds.json does not contain "
            "a 'thresholds' object."
        )

    thresholds: Dict[str, float] = {}

    missing: List[str] = []

    for feature in prediction_labels:

        if feature not in raw_thresholds:
            missing.append(feature)
            continue

        threshold = float(raw_thresholds[feature])

        if not 0.0 <= threshold <= 1.0:
            raise ValueError(
                f"Invalid threshold for {feature!r}: "
                f"{threshold}"
            )

        thresholds[feature] = threshold

    if missing:
        raise ValueError(
            "No calibrated threshold was found for the following "
            "prediction labels:\n"
            + ", ".join(missing)
        )

    return thresholds


# =============================================================================
# Feature hierarchy
# =============================================================================

def make_hierarchy_expander(
    feature_parents: Dict[str, Set[str]],
    feature_switches: Dict[str, bool],
    full_labels: List[str],
):
    """
    Construct hierarchy-expansion functions matching the training script.

    Disabled intermediate nodes are traversed even though they are not returned.

    For example, if an intermediate hierarchy feature were disabled:

        Child -> DisabledParent -> Grandparent

    Child can still inherit Grandparent.
    """

    full_label_set = set(full_labels)

    def is_feature_enabled(feature: str) -> bool:
        return bool(
            feature_switches.get(
                feature,
                True,
            )
        )

    @lru_cache(maxsize=None)
    def get_feature_ancestors(
        feature: str,
    ) -> frozenset:

        ancestors: Set[str] = set()

        for parent in feature_parents.get(
            feature,
            set(),
        ):

            if is_feature_enabled(parent):
                ancestors.add(parent)

            # Always traverse through the node, even when the node itself
            # has been disabled.
            ancestors.update(
                get_feature_ancestors(parent)
            )

        return frozenset(ancestors)

    def expand_feature_hierarchy(
        features: Iterable[str],
    ) -> Set[str]:

        expanded = {
            feature
            for feature in features
            if is_feature_enabled(feature)
        }

        for feature in list(expanded):
            expanded.update(
                get_feature_ancestors(feature)
            )

        # The training-derived full vocabulary determines which features are
        # legitimate final output labels.
        expanded.intersection_update(
            full_label_set
        )

        return expanded

    return expand_feature_hierarchy


# =============================================================================
# Model loading
# =============================================================================

def load_saved_state_dict(
    model_dir: Path,
) -> Dict[str, torch.Tensor]:
    """
    Load weights saved by Hugging Face Trainer.

    The custom training model wraps AutoModelForTokenClassification inside
    self.model. Depending on the Transformers version, Trainer.save_model()
    may therefore save keys such as:

        model.roberta....
        model.classifier....

    rather than:

        roberta....
        classifier....

    This loader handles either representation.
    """

    safetensors_path = (
        model_dir / "model.safetensors"
    )

    pytorch_path = (
        model_dir / "pytorch_model.bin"
    )

    if safetensors_path.exists():

        state_dict = load_safetensors_file(
            str(safetensors_path),
            device="cpu",
        )

    elif pytorch_path.exists():

        loaded = torch.load(
            str(pytorch_path),
            map_location="cpu",
        )

        if (
            isinstance(loaded, dict)
            and "state_dict" in loaded
            and isinstance(
                loaded["state_dict"],
                dict,
            )
        ):
            state_dict = loaded["state_dict"]
        else:
            state_dict = loaded

    else:
        raise FileNotFoundError(
            "Could not find model weights in:\n"
            f"{model_dir.resolve()}\n\n"
            "Expected either:\n"
            "  model.safetensors\n"
            "or:\n"
            "  pytorch_model.bin"
        )

    if not isinstance(state_dict, dict):
        raise RuntimeError(
            "Saved model weights are not a state_dict."
        )

    return state_dict


def normalise_state_dict_keys(
    state_dict: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    """
    Remove wrapper prefixes introduced by the custom training model.
    """

    normalised: Dict[str, torch.Tensor] = {}

    for key, value in state_dict.items():

        new_key = key

        # Common distributed/compiled-model prefixes.
        while new_key.startswith("module."):
            new_key = new_key[len("module."):]

        while new_key.startswith("_orig_mod."):
            new_key = new_key[len("_orig_mod."):]

        # RobertaMultiLabelModel stores the Hugging Face model in self.model.
        if new_key.startswith("model."):
            new_key = new_key[len("model."):]

        normalised[new_key] = value

    return normalised


def load_model(
    prediction_labels: List[str],
    device: torch.device,
):
    """
    Recreate AutoModelForTokenClassification and load the fine-tuned weights.
    """

    label2id = {
        label: index
        for index, label in enumerate(
            prediction_labels
        )
    }

    id2label = {
        index: label
        for label, index in label2id.items()
    }

    print("Loading RoBERTa architecture...")

    model = (
        AutoModelForTokenClassification
        .from_pretrained(
            BASE_MODEL_NAME,
            num_labels=len(prediction_labels),
            id2label=id2label,
            label2id=label2id,

            # The original pretrained model does not have our custom
            # multi-label classifier dimensions.
            ignore_mismatched_sizes=True,
        )
    )

    print("Loading fine-tuned weights...")

    state_dict = load_saved_state_dict(
        MODEL_DIR
    )

    state_dict = normalise_state_dict_keys(
        state_dict
    )

    incompatible = model.load_state_dict(
        state_dict,
        strict=False,
    )

    missing_keys = list(
        incompatible.missing_keys
    )

    unexpected_keys = list(
        incompatible.unexpected_keys
    )

    if missing_keys or unexpected_keys:

        message = []

        if missing_keys:
            message.append(
                "Missing keys:\n  "
                + "\n  ".join(missing_keys)
            )

        if unexpected_keys:
            message.append(
                "Unexpected keys:\n  "
                + "\n  ".join(unexpected_keys)
            )

        raise RuntimeError(
            "The saved weights do not exactly match the "
            "reconstructed RoBERTa model.\n\n"
            + "\n\n".join(message)
        )

    # Ensure the label mapping reflects semantic_features.json rather than any
    # mapping inherited from the original Facebook checkpoint.
    model.config.label2id = label2id
    model.config.id2label = id2label

    model.to(device)
    model.eval()

    return model


# =============================================================================
# Inference
# =============================================================================

def predict_sentence(
    sentence: Any,
    sentence_index: int,
    model: Any,
    tokenizer: Any,
    prediction_labels: List[str],
    thresholds: Dict[str, float],
    expand_feature_hierarchy: Any,
    device: torch.device,
) -> List[Dict[str, Any]]:
    """
    Predict semantic features for NOUN and PROPN tokens in one spaCy sentence.
    """

    tokens = list(sentence)

    if not tokens:
        return []

    words = [
        token.text
        for token in tokens
    ]

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
        return_tensors="pt",
    )

    # word_ids maps RoBERTa subwords back to the original spaCy tokens.
    word_ids = encoding.word_ids(
        batch_index=0
    )

    first_subword_position: Dict[int, int] = {}

    for sequence_position, word_id in enumerate(
        word_ids
    ):

        if word_id is None:
            continue

        if word_id not in first_subword_position:
            first_subword_position[
                word_id
            ] = sequence_position

    encoded_word_count = (
        max(first_subword_position) + 1
        if first_subword_position
        else 0
    )

    if encoded_word_count < len(words):

        print(
            f"Warning: sentence {sentence_index + 1} "
            f"was truncated at MAX_LENGTH={MAX_LENGTH}. "
            f"Only {encoded_word_count}/{len(words)} source "
            "tokens were encoded."
        )

    model_inputs = {
        key: value.to(device)
        for key, value in encoding.items()
    }

    with torch.inference_mode():

        outputs = model(
            **model_inputs
        )

        probabilities = torch.sigmoid(
            outputs.logits
        )[0].float().cpu()

    records: List[Dict[str, Any]] = []

    for word_index, token in enumerate(tokens):

        if token.pos_ not in TARGET_POS:
            continue

        sequence_position = (
            first_subword_position.get(
                word_index
            )
        )

        # The token was probably removed by truncation.
        if sequence_position is None:
            continue

        token_probabilities = (
            probabilities[
                sequence_position
            ]
        )

        independent_features: List[str] = []

        probability_document: Dict[
            str,
            float,
        ] = {}

        for label_id, feature in enumerate(
            prediction_labels
        ):

            probability = float(
                token_probabilities[
                    label_id
                ].item()
            )

            threshold = thresholds[
                feature
            ]

            if probability >= threshold:

                independent_features.append(
                    feature
                )

                probability_document[
                    feature
                ] = probability

        independent_features.sort()

        complete_features = sorted(
            expand_feature_hierarchy(
                independent_features
            )
        )

        record: Dict[str, Any] = {
            "sentence_index": sentence_index,
            "token_id": word_index + 1,
            "form": token.text,
            "lemma": token.lemma_,
            "upos": token.pos_,
            "features": complete_features,
            "independent_features": (
                independent_features
            ),
        }

        if INCLUDE_PROBABILITIES:

            record[
                "independent_probabilities"
            ] = {
                feature: round(
                    probability_document[
                        feature
                    ],
                    6,
                )
                for feature in (
                    independent_features
                )
            }

            record[
                "thresholds"
            ] = {
                feature: thresholds[
                    feature
                ]
                for feature in (
                    independent_features
                )
            }

        records.append(record)

    return records


# =============================================================================
# Display
# =============================================================================

def print_sentence_results(
    sentence_index: int,
    sentence_text: str,
    records: List[Dict[str, Any]],
) -> None:
    """Pretty-print predictions for one sentence."""

    print()
    print("=" * 100)
    print(
        f"SENTENCE {sentence_index + 1}"
    )
    print("=" * 100)
    print(sentence_text.strip())
    print()

    if not records:
        print(
            "No NOUN/PROPN tokens were found."
        )
        return

    for record in records:

        form = record["form"]
        upos = record["upos"]

        independent = record[
            "independent_features"
        ]

        complete = record[
            "features"
        ]

        print(
            f"{form!r} [{upos}]"
        )

        print(
            "  Independent : "
            + (
                ", ".join(independent)
                if independent
                else "(none)"
            )
        )

        print(
            "  Complete    : "
            + (
                ", ".join(complete)
                if complete
                else "(none)"
            )
        )

        if (
            INCLUDE_PROBABILITIES
            and independent
        ):

            probability_strings = []

            for feature in independent:

                probability = record[
                    "independent_probabilities"
                ][feature]

                threshold = record[
                    "thresholds"
                ][feature]

                probability_strings.append(
                    f"{feature}="
                    f"{probability:.3f}"
                    f" (threshold {threshold:.2f})"
                )

            print(
                "  Scores      : "
                + "; ".join(
                    probability_strings
                )
            )

        print()


# =============================================================================
# Main
# =============================================================================

def main() -> None:

    if not SENTENCES.strip():
        raise ValueError(
            "SENTENCES is empty. Add the text you want to tag "
            "to the triple-quoted SENTENCES variable."
        )

    if not MODEL_DIR.exists():
        raise FileNotFoundError(
            "Could not find model directory:\n"
            f"{MODEL_DIR.resolve()}"
        )

    # -------------------------------------------------------------------------
    # Device
    # -------------------------------------------------------------------------

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("=" * 100)
    print("DEVICE")
    print("=" * 100)
    print(f"Device: {device}")

    if device.type == "cuda":
        print(
            "GPU: "
            + torch.cuda.get_device_name(0)
        )

    print()

    # -------------------------------------------------------------------------
    # Metadata
    # -------------------------------------------------------------------------

    label_information = (
        load_label_information()
    )

    prediction_labels = (
        label_information[
            "prediction_labels"
        ]
    )

    full_labels = (
        label_information[
            "full_labels"
        ]
    )

    feature_parents = (
        label_information[
            "feature_parents"
        ]
    )

    feature_switches = (
        label_information[
            "feature_switches"
        ]
    )

    thresholds = (
        load_prediction_thresholds(
            prediction_labels
        )
    )

    expand_feature_hierarchy = (
        make_hierarchy_expander(
            feature_parents=feature_parents,
            feature_switches=feature_switches,
            full_labels=full_labels,
        )
    )

    print("=" * 100)
    print("SEMANTIC FEATURE MODEL")
    print("=" * 100)

    print(
        f"Model directory             : "
        f"{MODEL_DIR.resolve()}"
    )

    print(
        f"Independent neural outputs : "
        f"{len(prediction_labels):,}"
    )

    print(
        f"Complete hierarchy features: "
        f"{len(full_labels):,}"
    )

    disabled_features = sorted(
        feature
        for feature, enabled
        in feature_switches.items()
        if not enabled
    )

    print(
        "Disabled hierarchy features: "
        + (
            ", ".join(disabled_features)
            if disabled_features
            else "none"
        )
    )

    print()

    # -------------------------------------------------------------------------
    # Tokeniser
    # -------------------------------------------------------------------------

    print(
        "Loading RoBERTa tokeniser..."
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            str(MODEL_DIR),
            use_fast=True,
        )
    )

    if not tokenizer.is_fast:
        raise RuntimeError(
            "A fast Hugging Face tokeniser is required because "
            "inference relies on word_ids() for subword alignment."
        )

    # -------------------------------------------------------------------------
    # Model
    # -------------------------------------------------------------------------

    model = load_model(
        prediction_labels=prediction_labels,
        device=device,
    )

    # -------------------------------------------------------------------------
    # spaCy
    # -------------------------------------------------------------------------

    print(
        f"Loading spaCy model "
        f"{SPACY_MODEL!r}..."
    )

    nlp = spacy.load(
        SPACY_MODEL,
        disable=[
            "ner",
        ],
    )

    print()

    # -------------------------------------------------------------------------
    # Parse supplied text
    # -------------------------------------------------------------------------

    doc = nlp(
        SENTENCES.strip()
    )

    sentences = list(
        doc.sents
    )

    print(
        f"Sentences to tag: "
        f"{len(sentences):,}"
    )

    print()

    # -------------------------------------------------------------------------
    # Inference
    # -------------------------------------------------------------------------

    all_records: List[
        Dict[str, Any]
    ] = []

    for sentence_index, sentence in enumerate(
        sentences
    ):

        records = predict_sentence(
            sentence=sentence,
            sentence_index=sentence_index,
            model=model,
            tokenizer=tokenizer,
            prediction_labels=prediction_labels,
            thresholds=thresholds,
            expand_feature_hierarchy=(
                expand_feature_hierarchy
            ),
            device=device,
        )

        all_records.extend(records)

        print_sentence_results(
            sentence_index=sentence_index,
            sentence_text=sentence.text,
            records=records,
        )

    # -------------------------------------------------------------------------
    # JSONL output
    # -------------------------------------------------------------------------

    with OUTPUT_JSONL.open(
        "w",
        encoding="utf-8",
    ) as handle:

        for record in all_records:

            handle.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    print("=" * 100)
    print("COMPLETE")
    print("=" * 100)

    print(
        f"Tagged NOUN/PROPN tokens: "
        f"{len(all_records):,}"
    )

    print(
        "JSONL output:\n"
        f"  {OUTPUT_JSONL.resolve()}"
    )


if __name__ == "__main__":
    main()

DEVICE
Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti

SEMANTIC FEATURE MODEL
Model directory             : /home/mark/verbnet-roberta/models/roberta_semantic_features/reference_model
Independent neural outputs : 37
Complete hierarchy features: 39
Disabled hierarchy features: Rigidity, Shape, State, State_of_Matter

Loading RoBERTa tokeniser...
Loading RoBERTa architecture...


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:01<00:00, 256.20it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading fine-tuned weights...
Loading spaCy model 'en_core_web_sm'...

Sentences to tag: 28


SENTENCE 1
The old dog slept beside the fireplace.

'dog' [NOUN]
  Independent : Animal, Solid
  Complete    : Animal, Animate, Concrete, Internal_Control, Natural, Solid
  Scores      : Animal=0.992 (threshold 0.83); Solid=0.995 (threshold 0.49)

'fireplace' [NOUN]
  Independent : Artefact, Inanimate, Location, Rigid
  Complete    : Artefact, Concrete, Inanimate, Location, Rigid, Solid
  Scores      : Artefact=0.992 (threshold 0.64); Inanimate=0.996 (threshold 0.46); Location=0.805 (threshold 0.41); Rigid=0.993 (threshold 0.16)


SENTENCE 2
Microsoft opened a new office in London last year.

'Microsoft' [PROPN]
  Independent : Organisation
  Complete    : Abstract, Internal_Control, Organisation
  Scores      : Organisation=0.995 (threshold 0.54)

'office' [NOUN]
  Independent : Artefact, Inanimate, Location, Rigid, Solid
  Complete    : Artefact, Concrete, Inanimate, Location, Rigid, Solid
 

<h1>Feature distillation</h1>

In [38]:
#!/usr/bin/env python3
"""
Distil semantic restrictions on the arguments of repeated VerbNet-style
frame examples.

Compatible with Python 3.9.21.

The script supports frames of ANY number of arguments.

Examples
--------

Two-argument frame:

    Susan talked with Rachel.

    FRAME_ARGUMENTS = [
        "Speaker",
        "Co_interlocutor",
    ]


Three-argument frame:

    Susan talked about the problem to Rachel.

    FRAME_ARGUMENTS = [
        "Speaker",
        "Topic",
        "Addressee",
    ]


Automatic positional mode:

    FRAME_ARGUMENTS = None

The script will inspect the JSONL data and create:

    Argument_1
    Argument_2
    Argument_3
    ...

Mutually exclusive features are resolved BEFORE threshold filtering.

For example:

    Rigid       9/30
    Non_Rigid  27/30

Both are removed before the 0.70 threshold is applied.

A feature such as Comestible is NOT removed merely because it inherits
from a more frequent feature such as Concrete.
"""

from __future__ import annotations

import json
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple


# =============================================================================
# Configuration
# =============================================================================

INPUT_JSONL = Path(
    "tagged_sentences.jsonl"
)

OUTPUT_JSON = Path(
    "distilled_frame_features.json"
)


# =============================================================================
# Frame definition
# =============================================================================

FRAME_NAME = (
     "poke: Agent V Patient with Instrument"
)


# Specify as many arguments as the frame contains.
#
# The order MUST correspond to the order of the tagged arguments in
# each sentence.
#
# Example:
#
#     Susan talked about the problem to Rachel.
#
# Tagged arguments:
#
#     Susan    -> Speaker
#     problem  -> Topic
#     Rachel   -> Addressee
#
FRAME_ARGUMENTS = [
    "Agent",
    "Patient",
    "Instrument",
]


# Alternatively:
#
# FRAME_ARGUMENTS = None
#
# This causes the script to infer the number of arguments from the data
# and generate:
#
#     Argument_1
#     Argument_2
#     ...
#


# =============================================================================
# Feature settings
# =============================================================================

FEATURE_FIELD = "features"


# A feature must occur in MORE THAN 70% of argument fillers.
#
# With 30 examples:
#
#     22/30 = 73.3% -> retained
#     21/30 = 70.0% -> excluded
#
# With 50 examples:
#
#     36/50 = 72.0% -> retained
#     35/50 = 70.0% -> excluded
#
MIN_FEATURE_PROPORTION = 0.70


EPSILON = 1e-9


# =============================================================================
# Feature hierarchy
# =============================================================================

FEATURE_PARENTS: Dict[str, Set[str]] = {

    "Organisation": {
        "Internal_Control",
        "Abstract",
    },

    "Natural_Force": {
        "Internal_Control",
    },

    "Animate": {
        "Internal_Control",
        "Natural",
    },

    "Human": {
        "Animate",
    },

    "Feminine": {
        "Human",
    },

    "Masculine": {
        "Human",
    },

    "Unspecified_Gender_or_Neuter": {
        "Human",
    },

    "Animal": {
        "Animate",
    },

    "Plant": {
        "Animate",
    },

    "Body_Part": {
        "Animate",
    },

    "Machine": {
        "Internal_Control",
        "Artefact",
    },

    "Vehicle": {
        "Machine",
    },

    "Natural": {
        "Concrete",
    },

    "Artefact": {
        "Concrete",
    },

    "Comestible": {
        "Concrete",
    },

    "Substance": {
        "Concrete",
    },

    "State_of_Matter": {
        "Concrete",
    },

    "Inanimate": {
        "Concrete",
    },

    "Garment": {
        "Artefact",
    },

    "Solid": {
        "State_of_Matter",
    },

    "Fluid": {
        "State_of_Matter",
    },

    "Rigidity": {
        "Solid",
    },

    "Shape": {
        "Solid",
    },

    "Rigid": {
        "Rigidity",
    },

    "Non_Rigid": {
        "Rigidity",
    },

    "Elongated": {
        "Shape",
    },

    "Pointy": {
        "Shape",
    },

    "Point_in_Time": {
        "Time",
    },

    "Duration": {
        "Time",
    },

    "Time": {
        "Abstract",
    },

    "Language": {
        "Abstract",
    },

    "Sound": {
        "Abstract",
    },

    "Communication": {
        "Abstract",
    },

    "Eventive": {
        "Abstract",
    },

    "Statement": {
        "Communication",
    },

    "Question": {
        "Communication",
    },
}


# =============================================================================
# Mutually exclusive feature groups
# =============================================================================

MUTUALLY_EXCLUSIVE_GROUPS: List[Set[str]] = [

    {
        "Rigid",
        "Non_Rigid",
    },

    {
        "Solid",
        "Fluid",
    },

    {
        "Feminine",
        "Masculine",
        "Unspecified_Gender_or_Neuter",
    },

    {
        "Point_in_Time",
        "Duration",
    },
]


# =============================================================================
# Hierarchy helpers
# =============================================================================

@lru_cache(maxsize=None)
def get_ancestors(
    feature: str,
) -> frozenset:
    """Return all ancestors of a feature."""

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(
        feature,
        set(),
    ):

        ancestors.add(
            parent
        )

        ancestors.update(
            get_ancestors(
                parent
            )
        )

    return frozenset(
        ancestors
    )


def get_descendants(
    ancestor: str,
    available_features: Set[str],
) -> Set[str]:
    """Return observed descendants of an ancestor."""

    return {
        feature
        for feature in available_features
        if (
            feature != ancestor
            and ancestor in get_ancestors(feature)
        )
    }


# =============================================================================
# Loading
# =============================================================================

def load_jsonl(
    path: Path,
) -> List[Dict[str, Any]]:
    """Load inference JSONL."""

    if not path.exists():

        raise FileNotFoundError(
            f"Could not find:\n"
            f"{path.resolve()}"
        )

    records: List[
        Dict[str, Any]
    ] = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for line_number, line in enumerate(
            handle,
            start=1,
        ):

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(
                    line
                )

            except json.JSONDecodeError as exc:

                raise ValueError(
                    f"Invalid JSON on line "
                    f"{line_number}"
                ) from exc

            if not isinstance(
                record,
                dict,
            ):

                raise ValueError(
                    f"Line {line_number} is "
                    f"not a JSON object."
                )

            records.append(
                record
            )

    if not records:

        raise ValueError(
            "The input file contains no records."
        )

    return records


# =============================================================================
# Group records by sentence
# =============================================================================

def group_by_sentence(
    records: List[Dict[str, Any]],
) -> Dict[
    int,
    List[Dict[str, Any]],
]:
    """
    Group argument records by sentence.

    Records within each sentence are sorted by token position.
    """

    grouped: Dict[
        int,
        List[Dict[str, Any]],
    ] = defaultdict(list)

    for record in records:

        if "sentence_index" not in record:

            raise ValueError(
                "Record is missing sentence_index."
            )

        if "token_id" not in record:

            raise ValueError(
                "Record is missing token_id."
            )

        sentence_index = int(
            record[
                "sentence_index"
            ]
        )

        grouped[
            sentence_index
        ].append(
            record
        )

    for sentence_index in grouped:

        grouped[
            sentence_index
        ].sort(
            key=lambda record: int(
                record[
                    "token_id"
                ]
            )
        )

    return dict(
        sorted(
            grouped.items()
        )
    )


# =============================================================================
# Determine frame length and role names
# =============================================================================

def resolve_frame_arguments(
    sentences: Dict[
        int,
        List[Dict[str, Any]],
    ],
) -> List[str]:
    """
    Determine the semantic role inventory for this frame.

    The number of tagged arguments is inferred from the data.

    All sentences must contain the same number of tagged arguments.

    If FRAME_ARGUMENTS is None, generic role labels are generated:

        Argument_1
        Argument_2
        ...

    If FRAME_ARGUMENTS is supplied, its length must match the detected
    argument count.
    """

    if not sentences:

        raise ValueError(
            "No frame examples were found."
        )

    # -------------------------------------------------------------------------
    # Count arguments in each sentence
    # -------------------------------------------------------------------------

    argument_counts: Dict[
        int,
        int,
    ] = {
        sentence_index: len(
            records
        )
        for sentence_index, records
        in sentences.items()
    }

    unique_counts = set(
        argument_counts.values()
    )

    # -------------------------------------------------------------------------
    # Every example must instantiate the same frame length
    # -------------------------------------------------------------------------

    if len(unique_counts) != 1:

        details: List[str] = []

        for sentence_index, records in sentences.items():

            forms = [
                str(
                    record.get(
                        "form",
                        "?",
                    )
                )
                for record in records
            ]

            details.append(
                f"Sentence "
                f"{sentence_index + 1}: "
                f"{len(records)} arguments "
                f"{forms}"
            )

        raise ValueError(
            "Frame examples do not all contain "
            "the same number of tagged arguments.\n\n"
            + "\n".join(details)
        )

    detected_count = next(
        iter(
            unique_counts
        )
    )

    if detected_count == 0:

        raise ValueError(
            "No tagged arguments were found."
        )

    # -------------------------------------------------------------------------
    # Automatic role naming
    # -------------------------------------------------------------------------

    if FRAME_ARGUMENTS is None:

        roles = [
            f"Argument_{index + 1}"
            for index in range(
                detected_count
            )
        ]

        print(
            "FRAME_ARGUMENTS is None."
        )

        print(
            f"Automatically detected "
            f"{detected_count} arguments."
        )

        print(
            "Generated roles: "
            + ", ".join(
                roles
            )
        )

        return roles

    # -------------------------------------------------------------------------
    # Explicit role naming
    # -------------------------------------------------------------------------

    roles = [
        str(role)
        for role
        in FRAME_ARGUMENTS
    ]

    if len(roles) != detected_count:

        first_sentence_index = min(
            sentences
        )

        first_forms = [
            str(
                record.get(
                    "form",
                    "?",
                )
            )
            for record
            in sentences[
                first_sentence_index
            ]
        ]

        raise ValueError(
            "FRAME_ARGUMENTS does not match "
            "the frame found in the data.\n\n"
            f"Configured roles : "
            f"{len(roles)}\n"
            f"Detected arguments: "
            f"{detected_count}\n\n"
            f"Configured:\n"
            f"  {roles}\n\n"
            f"First frame instance:\n"
            f"  {first_forms}\n\n"
            "Either update FRAME_ARGUMENTS "
            "to contain one role for every argument, "
            "or set:\n\n"
            "    FRAME_ARGUMENTS = None\n\n"
            "to use automatic positional labels."
        )

    # -------------------------------------------------------------------------
    # Duplicate role names would make the output ambiguous
    # -------------------------------------------------------------------------

    if len(
        set(
            roles
        )
    ) != len(
        roles
    ):

        raise ValueError(
            "FRAME_ARGUMENTS contains duplicate "
            "role names:\n"
            f"{roles}"
        )

    return roles


# =============================================================================
# Assign arguments
# =============================================================================

def assign_arguments(
    sentences: Dict[
        int,
        List[Dict[str, Any]],
    ],
    frame_arguments: List[str],
) -> Dict[
    str,
    List[Dict[str, Any]],
]:
    """
    Assign records to argument positions.

    Works for frames of arbitrary length.
    """

    arguments: Dict[
        str,
        List[Dict[str, Any]],
    ] = {
        role: []
        for role
        in frame_arguments
    }

    expected = len(
        frame_arguments
    )

    for sentence_index, records in sentences.items():

        if len(records) != expected:

            raise ValueError(
                f"Sentence "
                f"{sentence_index + 1} "
                f"contains {len(records)} "
                f"arguments; expected {expected}."
            )

        for role, record in zip(
            frame_arguments,
            records,
        ):

            arguments[
                role
            ].append(
                record
            )

    return arguments


# =============================================================================
# Feature helpers
# =============================================================================

def get_feature_set(
    record: Dict[str, Any],
) -> Set[str]:
    """Return features for one argument filler."""

    raw_features = record.get(
        FEATURE_FIELD,
        [],
    )

    if raw_features is None:

        return set()

    if not isinstance(
        raw_features,
        list,
    ):

        raise ValueError(
            f"{FEATURE_FIELD!r} must "
            f"contain a list."
        )

    return {
        str(feature)
        for feature
        in raw_features
    }


# =============================================================================
# Resolve mutually-exclusive features
# =============================================================================

def resolve_mutual_exclusions(
    observed_features: Set[str],
) -> Tuple[
    Set[str],
    List[Dict[str, Any]],
]:
    """
    Resolve semantic clashes BEFORE frequency thresholding.

    If two or more mutually-exclusive features occur anywhere in the
    argument position, all observed members of that exclusion group
    are removed.

    Example:

        Rigid       9/30
        Non_Rigid  27/30

    Both Rigid and Non_Rigid are removed before the 0.70 threshold
    is applied.
    """

    resolved = set(
        observed_features
    )

    conflicts: List[
        Dict[str, Any]
    ] = []

    for group in MUTUALLY_EXCLUSIVE_GROUPS:

        present = (
            observed_features
            & group
        )

        if len(present) < 2:

            continue

        resolved.difference_update(
            present
        )

        conflicts.append(
            {
                "group": sorted(
                    group
                ),
                "removed": sorted(
                    present
                ),
            }
        )

    return (
        resolved,
        conflicts,
    )


# =============================================================================
# Below-threshold hierarchy diagnostics
# =============================================================================

def analyse_below_threshold_hierarchy(
    counts: Counter,
    total: int,
) -> List[Dict[str, Any]]:
    """
    Examine specific features that fall below the main threshold.

    This is diagnostic only.

    It does NOT remove above-threshold features such as Comestible.

    For example:

        Human   25/50
        Animal  25/50
        Animate 50/50

    can help explain why Animate is strongly supported even though its
    more specific descendants are not.
    """

    available_features = set(
        counts.keys()
    )

    reports: List[
        Dict[str, Any]
    ] = []

    for ancestor in sorted(
        available_features
    ):

        ancestor_count = int(
            counts.get(
                ancestor,
                0,
            )
        )

        ancestor_proportion = (
            ancestor_count
            / total
        )

        descendants = get_descendants(
            ancestor=ancestor,
            available_features=(
                available_features
            ),
        )

        weak_descendants: List[
            Dict[str, Any]
        ] = []

        for descendant in descendants:

            count = int(
                counts[
                    descendant
                ]
            )

            proportion = (
                count
                / total
            )

            if (
                proportion
                <=
                MIN_FEATURE_PROPORTION
                + EPSILON
            ):

                weak_descendants.append(
                    {
                        "feature": (
                            descendant
                        ),
                        "count": (
                            count
                        ),
                        "proportion": (
                            proportion
                        ),
                    }
                )

        if len(
            weak_descendants
        ) < 2:

            continue

        weak_descendants.sort(
            key=lambda item: (
                -item[
                    "proportion"
                ],
                item[
                    "feature"
                ],
            )
        )

        reports.append(
            {
                "ancestor": (
                    ancestor
                ),
                "ancestor_count": (
                    ancestor_count
                ),
                "ancestor_proportion": (
                    ancestor_proportion
                ),
                "below_threshold_descendants": (
                    weak_descendants
                ),
            }
        )

    return reports


# =============================================================================
# Analyse one argument position
# =============================================================================

def analyse_argument(
    records: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """Analyse one argument position across all frame examples."""

    if not records:

        raise ValueError(
            "Cannot analyse an empty "
            "argument set."
        )

    total = len(
        records
    )

    feature_sets = [
        get_feature_set(
            record
        )
        for record
        in records
    ]

    # -------------------------------------------------------------------------
    # Count feature occurrence
    # -------------------------------------------------------------------------

    counts: Counter = Counter()

    for features in feature_sets:

        counts.update(
            features
        )

    proportions: Dict[
        str,
        float,
    ] = {
        feature: (
            count
            / total
        )
        for feature, count
        in counts.items()
    }

    # -------------------------------------------------------------------------
    # All observed features
    # -------------------------------------------------------------------------

    observed_features: Set[str] = set(
        counts.keys()
    )

    # -------------------------------------------------------------------------
    # FIRST: remove mutually-exclusive clashes
    # -------------------------------------------------------------------------

    (
        non_conflicting_features,
        conflicts,
    ) = resolve_mutual_exclusions(
        observed_features=(
            observed_features
        ),
    )

    # -------------------------------------------------------------------------
    # SECOND: apply threshold
    # -------------------------------------------------------------------------

    final_features: Set[str] = {
        feature
        for feature
        in non_conflicting_features
        if (
            proportions[
                feature
            ]
            >
            MIN_FEATURE_PROPORTION
        )
    }

    # -------------------------------------------------------------------------
    # Below-threshold hierarchy diagnostics
    # -------------------------------------------------------------------------

    below_threshold_hierarchy = (
        analyse_below_threshold_hierarchy(
            counts=counts,
            total=total,
        )
    )

    # -------------------------------------------------------------------------
    # Example information
    # -------------------------------------------------------------------------

    examples: List[
        Dict[str, Any]
    ] = []

    for record, features in zip(
        records,
        feature_sets,
    ):

        examples.append(
            {
                "sentence_index": int(
                    record[
                        "sentence_index"
                    ]
                ),
                "token_id": int(
                    record[
                        "token_id"
                    ]
                ),
                "form": str(
                    record.get(
                        "form",
                        "",
                    )
                ),
                "lemma": str(
                    record.get(
                        "lemma",
                        "",
                    )
                ),
                "upos": str(
                    record.get(
                        "upos",
                        "",
                    )
                ),
                "features": sorted(
                    features
                ),
            }
        )

    return {

        "n_examples": (
            total
        ),

        "minimum_feature_proportion": (
            MIN_FEATURE_PROPORTION
        ),

        "observed_features": sorted(
            observed_features
        ),

        "non_conflicting_features": sorted(
            non_conflicting_features
        ),

        "final_features": sorted(
            final_features
        ),

        "mutual_exclusion_conflicts": (
            conflicts
        ),

        "below_threshold_hierarchy": (
            below_threshold_hierarchy
        ),

        "feature_frequencies": {
            feature: {
                "count": int(
                    count
                ),
                "total": (
                    total
                ),
                "proportion": (
                    count
                    / total
                ),
            }
            for feature, count
            in sorted(
                counts.items(),
                key=lambda item: (
                    -item[1],
                    item[0],
                ),
            )
        },

        "examples": (
            examples
        ),
    }


# =============================================================================
# Analyse complete frame
# =============================================================================

def analyse_frame(
    arguments: Dict[
        str,
        List[Dict[str, Any]],
    ],
    frame_arguments: List[str],
) -> Dict[str, Any]:
    """
    Analyse every argument position.

    No assumption is made about frame length.
    """

    return {
        role: analyse_argument(
            arguments[
                role
            ]
        )
        for role
        in frame_arguments
    }


# =============================================================================
# Display one role
# =============================================================================

def print_role_analysis(
    role: str,
    analysis: Dict[str, Any],
) -> None:
    """Display results for one argument position."""

    print()
    print("=" * 100)
    print(
        role.upper()
    )
    print("=" * 100)

    total = analysis[
        "n_examples"
    ]

    print(
        f"Examples          : "
        f"{total}"
    )

    print(
        f"Feature threshold : "
        f"> "
        f"{MIN_FEATURE_PROPORTION:.0%}"
    )

    # -------------------------------------------------------------------------
    # Fillers
    # -------------------------------------------------------------------------

    print()
    print("ARGUMENT FILLERS")
    print("-" * 100)

    print(
        ", ".join(
            example[
                "form"
            ]
            for example
            in analysis[
                "examples"
            ]
        )
    )

    # -------------------------------------------------------------------------
    # Prepare markers
    # -------------------------------------------------------------------------

    final_features = set(
        analysis[
            "final_features"
        ]
    )

    conflicted_features: Set[str] = set()

    for conflict in analysis[
        "mutual_exclusion_conflicts"
    ]:

        conflicted_features.update(
            conflict[
                "removed"
            ]
        )

    # -------------------------------------------------------------------------
    # Frequencies
    # -------------------------------------------------------------------------

    print()
    print("ALL FEATURE FREQUENCIES")
    print("-" * 100)

    frequencies = analysis[
        "feature_frequencies"
    ]

    for feature, information in frequencies.items():

        if feature in conflicted_features:

            marker = "!"

        elif feature in final_features:

            marker = "*"

        else:

            marker = " "

        print(
            f"{marker} "
            f"{feature:<32} "
            f"{information['count']:>3}/"
            f"{information['total']:<3} "
            f"("
            f"{information['proportion']:>6.1%}"
            f")"
        )

    print()

    print(
        "* = retained in final restriction"
    )

    print(
        "! = removed as a mutually-exclusive "
        "clash before thresholding"
    )

    # -------------------------------------------------------------------------
    # Mutual exclusions
    # -------------------------------------------------------------------------

    print()
    print("MUTUAL-EXCLUSION CLASHES")
    print("-" * 100)

    conflicts = analysis[
        "mutual_exclusion_conflicts"
    ]

    if not conflicts:

        print(
            "(none)"
        )

    else:

        for conflict in conflicts:

            print(
                "Removed: "
                + ", ".join(
                    conflict[
                        "removed"
                    ]
                )
            )

            print(
                "Exclusive group: "
                + ", ".join(
                    conflict[
                        "group"
                    ]
                )
            )

            print()

    # -------------------------------------------------------------------------
    # Final restrictions
    # -------------------------------------------------------------------------

    print()
    print("FINAL ARGUMENT RESTRICTION")
    print("-" * 100)

    final_features_list = analysis[
        "final_features"
    ]

    if not final_features_list:

        print(
            "(none)"
        )

    else:

        for feature in final_features_list:

            information = frequencies[
                feature
            ]

            print(
                f"  {feature:<32} "
                f"{information['count']:>3}/"
                f"{information['total']:<3} "
                f"("
                f"{information['proportion']:>6.1%}"
                f")"
            )


# =============================================================================
# Print distilled frame
# =============================================================================

def print_distilled_frame(
    analysis: Dict[str, Any],
    frame_arguments: List[str],
) -> None:
    """Print concise restrictions for every frame argument."""

    print()
    print("=" * 100)
    print("DISTILLED FRAME")
    print("=" * 100)

    print()
    print(
        FRAME_NAME
    )

    print()

    for index, role in enumerate(
        frame_arguments,
        start=1,
    ):

        features = analysis[
            role
        ][
            "final_features"
        ]

        if features:

            feature_string = (
                ", ".join(
                    features
                )
            )

        else:

            feature_string = (
                "(no restriction found)"
            )

        print(
            f"{index:>2}. "
            f"{role:<20} "
            f"{feature_string}"
        )


# =============================================================================
# Save results
# =============================================================================

def save_results(
    analysis: Dict[str, Any],
    frame_arguments: List[str],
) -> None:
    """Write results to JSON."""

    output = {

        "frame_name": (
            FRAME_NAME
        ),

        "argument_count": len(
            frame_arguments
        ),

        "argument_order": (
            frame_arguments
        ),

        "feature_field": (
            FEATURE_FIELD
        ),

        "minimum_feature_proportion": (
            MIN_FEATURE_PROPORTION
        ),

        "processing_order": [
            "count_features",
            "resolve_mutual_exclusions",
            "apply_frequency_threshold",
        ],

        "arguments": (
            analysis
        ),
    }

    with OUTPUT_JSON.open(
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            output,
            handle,
            ensure_ascii=False,
            indent=2,
        )

        handle.write(
            "\n"
        )


# =============================================================================
# Main
# =============================================================================

def main() -> None:

    print("=" * 100)
    print(
        "FRAME FEATURE DISTILLATION"
    )
    print("=" * 100)

    print(
        f"Frame             : "
        f"{FRAME_NAME}"
    )

    print(
        f"Input             : "
        f"{INPUT_JSONL.resolve()}"
    )

    print(
        f"Feature threshold : "
        f"> "
        f"{MIN_FEATURE_PROPORTION:.0%}"
    )

    # -------------------------------------------------------------------------
    # Load inference records
    # -------------------------------------------------------------------------

    records = load_jsonl(
        INPUT_JSONL
    )

    # -------------------------------------------------------------------------
    # Reconstruct sentences
    # -------------------------------------------------------------------------

    sentences = group_by_sentence(
        records
    )

    print(
        f"Frame examples    : "
        f"{len(sentences):,}"
    )

    # -------------------------------------------------------------------------
    # Determine frame length
    # -------------------------------------------------------------------------

    frame_arguments = (
        resolve_frame_arguments(
            sentences
        )
    )

    print(
        f"Arguments detected: "
        f"{len(frame_arguments)}"
    )

    print(
        f"Argument order    : "
        + " -> ".join(
            frame_arguments
        )
    )

    # -------------------------------------------------------------------------
    # Assign positional records to roles
    # -------------------------------------------------------------------------

    arguments = assign_arguments(
        sentences=sentences,
        frame_arguments=(
            frame_arguments
        ),
    )

    # -------------------------------------------------------------------------
    # Analyse
    # -------------------------------------------------------------------------

    analysis = analyse_frame(
        arguments=arguments,
        frame_arguments=(
            frame_arguments
        ),
    )

    # -------------------------------------------------------------------------
    # Detailed results
    # -------------------------------------------------------------------------

    for role in frame_arguments:

        print_role_analysis(
            role=role,
            analysis=analysis[
                role
            ],
        )

    # -------------------------------------------------------------------------
    # Concise frame
    # -------------------------------------------------------------------------

    print_distilled_frame(
        analysis=analysis,
        frame_arguments=(
            frame_arguments
        ),
    )

    # -------------------------------------------------------------------------
    # Save
    # -------------------------------------------------------------------------

    save_results(
        analysis=analysis,
        frame_arguments=(
            frame_arguments
        ),
    )

    print()
    print("=" * 100)
    print("COMPLETE")
    print("=" * 100)

    print(
        "Output:\n"
        f"  {OUTPUT_JSON.resolve()}"
    )


if __name__ == "__main__":
    main()

FRAME FEATURE DISTILLATION
Frame             : poke: Agent V Patient with Instrument
Input             : /home/mark/verbnet-roberta/notebooks/tagged_sentences.jsonl
Feature threshold : > 70%
Frame examples    : 50
Arguments detected: 3
Argument order    : Agent -> Patient -> Instrument

AGENT
Examples          : 50
Feature threshold : > 70%

ARGUMENT FILLERS
----------------------------------------------------------------------------------------------------
Allison, David, Sarah, James, Maria, Peter, Linda, Daniel, Alice, Matthew, child, doctor, baker, gardener, student, farmer, chef, tailor, scientist, mechanic, boy, girl, woman, man, nurse, artist, potter, cook, camper, fisherman, worker, researcher, tailor, child, baker, gardener, teacher, sculptor, technician, farmer, chef, camper, student, mechanic, scientist, tailor, cook, gardener, nurse, artist

ALL FEATURE FREQUENCIES
----------------------------------------------------------------------------------------------------
* Animate